# RSNA Knee — RGB 2.5D voisins DICOM, 320

**Shard 6/12** (CPU, internet off). **1778 series**, ~10.7 Go.

UID **figés** depuis `data/crop130_rgb25d/manifest_rgb25d_proto_part*.csv` (6 slots v20b, contraste exact). Même recette que le proto 224, seul le resize passe à **320** :

1. crop physique **130 mm** (`PixelSpacing`) ou crop centre 70 %
2. bande profondeur **20–80 %**
3. linspace **24** centres dans la bande
4. pour chaque centre `k`, **`k-1, k, k+1` dans le volume DICOM**
5. resize **320×320**, uint8, percentiles 1–99

Shape `(24, 3, 320, 320)`, canaux R=k-1, G=k, B=k+1.

Fichiers dans `/kaggle/working/` :
- `volumes_crop130_rgb25d_320_part6.zip`
- `manifest_rgb25d_320_part6.csv`
- `RECIPE.json`

Download : `python -X utf8 kaggle_DICOM_exctract/download_rgb25d_320.py` → `data/crop130_rgb25d_320/`.

In [ ]:
SHARD_ID = 6
N_SHARDS = 12

import os, io, gc, time, json, traceback
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import cv2
import numpy as np
import pandas as pd
import pydicom

# --- Recette (doit etre reproduite a l'identique au submit) ----------------
CROP_MM = 130.0
CROP_REL = 0.70          # fallback si PixelSpacing absent
BAND = (0.20, 0.80)      # profondeur : on ignore les 20 % de chaque bord
N_SLICES = 24
IMG_SIZE = 320
PLANES = ["Sagittal", "Coronal", "Axial"]
N_WORKERS = 4
# --------------------------------------------------------------------------

KAGGLE = os.path.exists("/kaggle")
OUT_DIR = Path("/kaggle/working") if KAGGLE else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE", KAGGLE, "| SHARD", SHARD_ID, "/", N_SHARDS)
print("out", OUT_DIR)


In [ ]:

def find_comp_dir() -> Path:
    if not KAGGLE:
        p = Path("../data/csv")
        if (p / "train_series.csv").exists():
            return p
        raise FileNotFoundError("train_series.csv introuvable en local")
    data = Path("/kaggle/input")
    print("contenu /kaggle/input :", sorted(x.name for x in data.iterdir()))
    for base in [data / "competitions", data]:
        if not base.exists():
            continue
        for p in sorted(base.iterdir()):
            if (p / "train.csv").exists() and (p / "train_series.csv").exists():
                return p
    raise FileNotFoundError("train.csv + train_series.csv introuvables sous /kaggle/input")

COMP_DIR = find_comp_dir()
TRAIN_SERIES_CSV = COMP_DIR / "train_series.csv"
TRAIN_SERIES_DIR = COMP_DIR / "train_series"
if not TRAIN_SERIES_DIR.exists():
    for cand in COMP_DIR.iterdir():
        if cand.is_dir() and (cand / "train_series").exists():
            TRAIN_SERIES_DIR = cand / "train_series"
            break
print("COMP_DIR         ", COMP_DIR)
print("TRAIN_SERIES_CSV ", TRAIN_SERIES_CSV, TRAIN_SERIES_CSV.exists())
print("TRAIN_SERIES_DIR ", TRAIN_SERIES_DIR, TRAIN_SERIES_DIR.exists())


In [ ]:
import json
jobs = json.loads('[{"study":"1.2.826.0.1.3680043.8.498.10004945927472656027199792075652399585","series":"1.2.826.0.1.3680043.8.498.99023014295026495734071089488213553280","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10009639203170750274174707434356622764","series":"1.2.826.0.1.3680043.8.498.76947689399889739752390906772011627242","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10018552945042470316597008531304016339","series":"1.2.826.0.1.3680043.8.498.83614010998801212494242431788612349272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10021288557171866794624317186288618513","series":"1.2.826.0.1.3680043.8.498.71462627071432782612166533162526483355","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10029168398037940681915299760479434938","series":"1.2.826.0.1.3680043.8.498.10398122283136151623747293132260100759","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10039686090738454718354181337513293981","series":"1.2.826.0.1.3680043.8.498.23046051137177793352388369210745150056","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10059200627175320481920009076697103801","series":"1.2.826.0.1.3680043.8.498.10705298691863112379533269403436857260","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10064642360075735959136394136507408269","series":"1.2.826.0.1.3680043.8.498.60540822688937024151515211249136527086","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10077803906105306230229342168184970097","series":"1.2.826.0.1.3680043.8.498.74334922299557471072402543805514695663","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10085187975213640798228717119613397941","series":"1.2.826.0.1.3680043.8.498.56139651590917417326146433456594976281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10088707526476055079083082931629995449","series":"1.2.826.0.1.3680043.8.498.20043323759072541095414617174838138298","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10090631811016939819371208464230332678","series":"1.2.826.0.1.3680043.8.498.11063356380296977745889863467626087110","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10096216489317602387766193675918767557","series":"1.2.826.0.1.3680043.8.498.81693348047871174074420628411856256035","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10100718559177860518999368876392117659","series":"1.2.826.0.1.3680043.8.498.77895759160477356898695433254206894683","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10118623027360228027776975311765133100","series":"1.2.826.0.1.3680043.8.498.17355150797297345667861720960301187303","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10123751739767586025744123799948754343","series":"1.2.826.0.1.3680043.8.498.11416464853182626707737982439110698105","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10124701778993602117271554854098856833","series":"1.2.826.0.1.3680043.8.498.63588274439837263681578203156909569430","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10132169498032330070492330913783717864","series":"1.2.826.0.1.3680043.8.498.10547858428902627227141655046812274423","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10135587809070718435691939321937314104","series":"1.2.826.0.1.3680043.8.498.10431242443728663712272201424026785351","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10143536655047216390592811329604738314","series":"1.2.826.0.1.3680043.8.498.29750195135795939693896442301999293994","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10147717839726734017869499799005084725","series":"1.2.826.0.1.3680043.8.498.30408508872190527653389633705344809821","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10149967031679082348901055550415519790","series":"1.2.826.0.1.3680043.8.498.37077644804808535042670994700386715899","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10155611590208164695256621422536643320","series":"1.2.826.0.1.3680043.8.498.13224452811235418040565202817533130058","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10159551371305936155888341171022480536","series":"1.2.826.0.1.3680043.8.498.11689504985769779618588241781208618567","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10162168547770229150261479353227398478","series":"1.2.826.0.1.3680043.8.498.61436613152337143341651094242302156767","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269","series":"1.2.826.0.1.3680043.8.498.45571369239262736508574040931936336167","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10181291283712008544475048457589664787","series":"1.2.826.0.1.3680043.8.498.36901938847843894559041565588551914645","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10183939841890895037058579455663880292","series":"1.2.826.0.1.3680043.8.498.38265444076831201006431018541125176189","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10197515493414564227930855210542221039","series":"1.2.826.0.1.3680043.8.498.93874544465317613358066683026399395267","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10201197020583611347810070328854760801","series":"1.2.826.0.1.3680043.8.498.17119669813127611916401473113653677494","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10205592316907266136938526795826751183","series":"1.2.826.0.1.3680043.8.498.62716337151876230734804799336736466852","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10211034309703220222647821582770542129","series":"1.2.826.0.1.3680043.8.498.41275905300976523890458326089749702439","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10218285921271312931891912812808052375","series":"1.2.826.0.1.3680043.8.498.12691312591633346949329305916775259321","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10229223546412441924801428332647762115","series":"1.2.826.0.1.3680043.8.498.55304327072157257009276640992522901216","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10240237086492917349933491458808329326","series":"1.2.826.0.1.3680043.8.498.12217217044073800739312789348216521025","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10247930792182001003497189021027466833","series":"1.2.826.0.1.3680043.8.498.42655534637312585571893824568232095907","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10252163031705285141538200795896549203","series":"1.2.826.0.1.3680043.8.498.20013967067892078473845526499599848515","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10255067212695306715722782826234429766","series":"1.2.826.0.1.3680043.8.498.66598498551591365503860420402652490336","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10262024906324456863812446480254600405","series":"1.2.826.0.1.3680043.8.498.10605967991068652864400796144979813176","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10271567891603078238024189515805908561","series":"1.2.826.0.1.3680043.8.498.15940085543420426570069535792310785642","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10274956886032850418742861294277331382","series":"1.2.826.0.1.3680043.8.498.17051735353762250068580937597343013095","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10284263494252700630857173843779724077","series":"1.2.826.0.1.3680043.8.498.18248817344830129435677242346152910065","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10299396087685174806129850512856451005","series":"1.2.826.0.1.3680043.8.498.87214451593001630031568136096024432187","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10301089872627496769249366231947887097","series":"1.2.826.0.1.3680043.8.498.93495652587822368992224521369968101882","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10309477218703786877612946261946268132","series":"1.2.826.0.1.3680043.8.498.10218757288375429931218419922496684748","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10312350714164739081126996907250465738","series":"1.2.826.0.1.3680043.8.498.10045399788540407838166435840006493073","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10321461243762453573005633063753741725","series":"1.2.826.0.1.3680043.8.498.13272554668810713061355545724464328238","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10321702681363532511953946967334431952","series":"1.2.826.0.1.3680043.8.498.40605939406486221410792371410610189705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10326352669696598934883805147731461341","series":"1.2.826.0.1.3680043.8.498.11145186758637043518446857366399416749","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10336221220793312503236159680373829726","series":"1.2.826.0.1.3680043.8.498.12167449940389703086621390892658137623","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10339654093023509742663983374586998753","series":"1.2.826.0.1.3680043.8.498.12880956709329274627094397114192376185","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10349368332722015923160611226877741306","series":"1.2.826.0.1.3680043.8.498.13323518992038409474498100312023197493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10352011938574988727803498915608467179","series":"1.2.826.0.1.3680043.8.498.24673250192942605758782883087023023100","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10354576637002647863048798607011636355","series":"1.2.826.0.1.3680043.8.498.41905079025977384392297566099626161206","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10371162371589200772388569623043753617","series":"1.2.826.0.1.3680043.8.498.24818396430228426354226600995292734402","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10378535286770237477615181844634903700","series":"1.2.826.0.1.3680043.8.498.69029661178389771772074972317839685178","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387227472110643414333498953513810066","series":"1.2.826.0.1.3680043.8.498.11365950413806191354382685836693130801","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10387876708310019345913209598612198432","series":"1.2.826.0.1.3680043.8.498.12512504807746259515139594057306618701","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10401813253995823397883215400000450724","series":"1.2.826.0.1.3680043.8.498.43614035606094585496373250632655348836","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10405028141768633100332386157519916334","series":"1.2.826.0.1.3680043.8.498.11220000050750131038029343935480763721","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10415616591430643294142401482460086713","series":"1.2.826.0.1.3680043.8.498.11005619584562790889907645845831317795","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10418888172081069709796646795446885715","series":"1.2.826.0.1.3680043.8.498.96557260978387941500935579002254150061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10422130685453254363700365908336246863","series":"1.2.826.0.1.3680043.8.498.42606993567248633550079604293085206460","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10425131723456547056365404016184663035","series":"1.2.826.0.1.3680043.8.498.66735727914846518402904991892228580424","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10429032846672074760218668873927822056","series":"1.2.826.0.1.3680043.8.498.48867926226552814715014504242639517711","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10440858363080910322920496514558347561","series":"1.2.826.0.1.3680043.8.498.46585137425050406240391797131783123157","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10445893053733106322562086357169418409","series":"1.2.826.0.1.3680043.8.498.13368568234445359003303843715171660447","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10455589411958266725060942213473374667","series":"1.2.826.0.1.3680043.8.498.69196110753701096243935529855861279516","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10459901081962705567398560859229437818","series":"1.2.826.0.1.3680043.8.498.16404458212489235759519520359726665777","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10472519452146922631626638652503679695","series":"1.2.826.0.1.3680043.8.498.55209899621551879954731886130222982852","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10491012711700179118010977263805828326","series":"1.2.826.0.1.3680043.8.498.87422659519667366982278218932095607870","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10495855084258410825408867079175646590","series":"1.2.826.0.1.3680043.8.498.18330874241754295767557491645332724371","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.10501068619333941198079851978532746117","series":"1.2.826.0.1.3680043.8.498.64646964510933624546107603900465243160","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10505282086293876725410939800054491454","series":"1.2.826.0.1.3680043.8.498.67890987531409138009444151065151011784","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10519480654942067141173172019311858213","series":"1.2.826.0.1.3680043.8.498.11734552831258476703633615044376550599","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10522585237725235866013737575867341946","series":"1.2.826.0.1.3680043.8.498.12344337104785422820115671682753011777","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10530226787566708645477257565528473018","series":"1.2.826.0.1.3680043.8.498.63056817456696347356021690845483275877","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10534260101860670685224894870019533364","series":"1.2.826.0.1.3680043.8.498.13084170268009812391701899080099976299","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10537503354035115750949427337500909327","series":"1.2.826.0.1.3680043.8.498.50405759671195518472512888033447959345","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10540980111087702906123811593906850156","series":"1.2.826.0.1.3680043.8.498.10452917563177180345266581460732294086","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10541215772904358938061268226270989801","series":"1.2.826.0.1.3680043.8.498.10135821375172863699392732809609582781","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10546313646515875953943156661172963953","series":"1.2.826.0.1.3680043.8.498.10418561691648873812099710891667260964","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10562412345413239960204081248510238736","series":"1.2.826.0.1.3680043.8.498.48040653850603861422474154902380990422","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10578013533179563448221888658506023084","series":"1.2.826.0.1.3680043.8.498.69813374485984353241674423312834344670","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10581718331223731732815674582085184169","series":"1.2.826.0.1.3680043.8.498.10634161618229808802230706892657078935","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10590148272001480024086117269799629816","series":"1.2.826.0.1.3680043.8.498.58048289417151759953728675164027586450","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10591507262178959553431147886647585916","series":"1.2.826.0.1.3680043.8.498.11898480838154899814371688171626543958","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10600654987896381983520458496207741798","series":"1.2.826.0.1.3680043.8.498.10401640265345549371695495496642546650","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10601711635138681269270505152909539290","series":"1.2.826.0.1.3680043.8.498.12069473940220088270591721393180140353","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10603952065748622499432940419853732241","series":"1.2.826.0.1.3680043.8.498.82692024738076562366999275778398800483","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10619486897279334206912764495067756449","series":"1.2.826.0.1.3680043.8.498.41975218590491556903019789623742612443","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10624417936475721846835761375334601839","series":"1.2.826.0.1.3680043.8.498.65217240116590229540549875582251240435","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10626411974489938664325062396760010912","series":"1.2.826.0.1.3680043.8.498.66386136914980875728127427361173345893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10631238417904652761254062374051500506","series":"1.2.826.0.1.3680043.8.498.13256047460739124706479506738823753831","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10642286343421893916348249679230632066","series":"1.2.826.0.1.3680043.8.498.13373520658463548329880708384827413001","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10652685418462611085839874321267337079","series":"1.2.826.0.1.3680043.8.498.44960911526511640638895347662189809480","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10653432159060774337155662807644609967","series":"1.2.826.0.1.3680043.8.498.62908964540746430673916978721350011038","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10657506540109137829814058249895525831","series":"1.2.826.0.1.3680043.8.498.10319775322816110943063130906326157364","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10659781998801845781246073928620795082","series":"1.2.826.0.1.3680043.8.498.18030423512659336927032241144123924558","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10674673641872871321890995010616205087","series":"1.2.826.0.1.3680043.8.498.68430162860603939298286709715065647061","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10676253869668012791900504544106495589","series":"1.2.826.0.1.3680043.8.498.65009092331383208161472396164162032580","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10686407053180376367066321489809395592","series":"1.2.826.0.1.3680043.8.498.87027045537634189229233868802694678419","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10702284568670902901846767166679293654","series":"1.2.826.0.1.3680043.8.498.38369819601741858502545937506533665506","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10713285016170190922777298047146228811","series":"1.2.826.0.1.3680043.8.498.49229153025185096442660102273362407365","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10726624821670417064567481545750543282","series":"1.2.826.0.1.3680043.8.498.30407977545623805440644314252683080056","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10729725525284633011179038668107963557","series":"1.2.826.0.1.3680043.8.498.48377584831792562024468929675955745572","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10738640862162795323010965765443420384","series":"1.2.826.0.1.3680043.8.498.56958459450073381620418225215100981954","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10740857795874468664161171813136103519","series":"1.2.826.0.1.3680043.8.498.46306843332895192702235452959256241569","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10751346650129497348457481341700957445","series":"1.2.826.0.1.3680043.8.498.34909842560134249802734216301101999272","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10756245293186761042431460535210737327","series":"1.2.826.0.1.3680043.8.498.11472076706249107993786646114442808583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10761658878615069270304502752374858610","series":"1.2.826.0.1.3680043.8.498.11630631447413724157271553406248363991","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10765617676329232046176300833850901740","series":"1.2.826.0.1.3680043.8.498.10156099427443605074093367030133502101","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10769126730281890056368854871900324448","series":"1.2.826.0.1.3680043.8.498.12681100451381077335207793159909441108","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10777331080659372520952742261140923555","series":"1.2.826.0.1.3680043.8.498.70714881625944686453974789098637689717","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10790689419760453870458170966064168002","series":"1.2.826.0.1.3680043.8.498.12985273724913429834126423786893878368","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10803283287658556463524649793955055077","series":"1.2.826.0.1.3680043.8.498.10180735577701876375549500652833237620","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10818980124981334596808438908811623528","series":"1.2.826.0.1.3680043.8.498.43437254428033785269210356783968702525","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10829579914809383212986717269028170263","series":"1.2.826.0.1.3680043.8.498.36867574525341385553974127784927174613","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10835889400414225749339608096119636738","series":"1.2.826.0.1.3680043.8.498.91812583922265019574285944405820522364","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10843463954622076609489312662891432839","series":"1.2.826.0.1.3680043.8.498.77574746339062590058612008891367008722","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10848350333764008140459854470293737650","series":"1.2.826.0.1.3680043.8.498.19804498360205813332100165708113470938","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10852517092524464338210061305792376636","series":"1.2.826.0.1.3680043.8.498.13817714506919300385620649512647014710","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10855290714402117092990651362697310270","series":"1.2.826.0.1.3680043.8.498.45992278556164570774312806646699696807","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10857771747813858957718058322384884551","series":"1.2.826.0.1.3680043.8.498.12154070531544273291700769165160477511","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10876177342063734507664188801203192617","series":"1.2.826.0.1.3680043.8.498.45663724849560379915723355107858343697","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10884987036948550523442514147204321748","series":"1.2.826.0.1.3680043.8.498.44875716030273428532384614327427462272","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10895521387894821238259709507311341758","series":"1.2.826.0.1.3680043.8.498.11156421699323964312458550338982861796","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10903003118372337016190215211789983264","series":"1.2.826.0.1.3680043.8.498.11954952069041799404856759609688757645","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10917606677895195837968927716728385881","series":"1.2.826.0.1.3680043.8.498.13045332286239299804742518239931048251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10920269679387171659375091645187772303","series":"1.2.826.0.1.3680043.8.498.96929585573897721903551266665759863627","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10924995713871537837233985073507509566","series":"1.2.826.0.1.3680043.8.498.12155107517777542142396259787060028435","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10931111730564885182398163970692599398","series":"1.2.826.0.1.3680043.8.498.10206453660242899208848019617478353188","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.10933003998197427979325590791540457658","series":"1.2.826.0.1.3680043.8.498.89353978859200722680282155538207557264","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10937585701850306334458420860276069606","series":"1.2.826.0.1.3680043.8.498.34972711138984645960124728527135052828","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10939964887585312209861601414669494409","series":"1.2.826.0.1.3680043.8.498.27073608252720951517697384231378335427","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10959153852077371752490787490374555514","series":"1.2.826.0.1.3680043.8.498.13303376136598938661136320384488159058","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10965573028231658307306828073340279415","series":"1.2.826.0.1.3680043.8.498.84193852744656921962254881584899359131","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10979105661314213697886164132444977106","series":"1.2.826.0.1.3680043.8.498.36342497499103736156162214747993071015","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.10992875360571368325079907908221899699","series":"1.2.826.0.1.3680043.8.498.73044252381624609351882739767751786695","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.10994848378662155711176898659205016445","series":"1.2.826.0.1.3680043.8.498.49489903135287148817355858095549376778","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.10994899269366718796438389394954844979","series":"1.2.826.0.1.3680043.8.498.19948375643720027039216035207659162686","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.10998405409474759941744305665540376339","series":"1.2.826.0.1.3680043.8.498.71744422906339532123807826101637800016","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11007527194601989039620198927241696256","series":"1.2.826.0.1.3680043.8.498.36026175253308207971026568249768281456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11014091506864318694851337553197477906","series":"1.2.826.0.1.3680043.8.498.48682169143178223534447595605970329746","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11041694833778404465582610810664391088","series":"1.2.826.0.1.3680043.8.498.53249229063772796403120029744734414981","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11043926695471016727738735328317975282","series":"1.2.826.0.1.3680043.8.498.12827050470113716868799858134974610649","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11050604958275820465127935558416408008","series":"1.2.826.0.1.3680043.8.498.48685447008300647869778575617459372707","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11052720536194493057619720620119095739","series":"1.2.826.0.1.3680043.8.498.70011032679948221644257986251535029294","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11064977170395595339882840534354229547","series":"1.2.826.0.1.3680043.8.498.12765967538821552634509769714053649533","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11077406555624926104628200252710914930","series":"1.2.826.0.1.3680043.8.498.94534289558864392106469813408261614982","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11080651962904815250736183346281427445","series":"1.2.826.0.1.3680043.8.498.10741632889324923303259748193086923516","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11091958567356981151034727948765010881","series":"1.2.826.0.1.3680043.8.498.61591243620624879472946313945259575278","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11103894265181149878864749992746236768","series":"1.2.826.0.1.3680043.8.498.99833984019170770899810570361289185437","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11113442398569287228475456000446876768","series":"1.2.826.0.1.3680043.8.498.66289228171579366415689932100403402073","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11122485005317809892694992204053124213","series":"1.2.826.0.1.3680043.8.498.21321284355739340698528017578429012688","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11125718814639515292482773418372267821","series":"1.2.826.0.1.3680043.8.498.11780545167941355935676311441545615361","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11129751904857491327872353316480050290","series":"1.2.826.0.1.3680043.8.498.12702622689463382211641854804396731712","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11135115047200279264914390528987153419","series":"1.2.826.0.1.3680043.8.498.37603028353469974956032505063115299661","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11145632111083286399683467139301564425","series":"1.2.826.0.1.3680043.8.498.37329107267355150556246751223540344735","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11150193626240150189232364321568347937","series":"1.2.826.0.1.3680043.8.498.12807887150379099844529794439858496640","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11150751959089942633754454715290410758","series":"1.2.826.0.1.3680043.8.498.83051027641531012413478023650743273529","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11154238555579687338157479416363307605","series":"1.2.826.0.1.3680043.8.498.40513931798213795972848116507640201531","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11173951743276817877673759975853741079","series":"1.2.826.0.1.3680043.8.498.18197452534184519780245131207091930342","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11183915112115196028799391678356464392","series":"1.2.826.0.1.3680043.8.498.44681788168050119818348104943064542440","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11191015483427903713800997920940844550","series":"1.2.826.0.1.3680043.8.498.93432516469827055984220448709093774219","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11201589885283493748455257854104771618","series":"1.2.826.0.1.3680043.8.498.83917796115523109278228736426804166404","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11207013225631251718312189933025529824","series":"1.2.826.0.1.3680043.8.498.12972299354368510426914890245705104304","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11210361780779474361817208167009080083","series":"1.2.826.0.1.3680043.8.498.50725246447014545329859808622328307293","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11214158717183172156320144105486247892","series":"1.2.826.0.1.3680043.8.498.11795173675605423190945360433961943506","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11218782547268547937641080283066300678","series":"1.2.826.0.1.3680043.8.498.81939873212979361904679613959649696892","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11223691171233118900160265287691267884","series":"1.2.826.0.1.3680043.8.498.26620953331975312711842952394853691151","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11229566047228169850473554486278603754","series":"1.2.826.0.1.3680043.8.498.82229545496605394414015171304256912219","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11232674915186872899189440191805868376","series":"1.2.826.0.1.3680043.8.498.12682567067855622828387506871274033447","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11254890841796835340329521299102281678","series":"1.2.826.0.1.3680043.8.498.42497718657780937237950447420581552548","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11257869433929994221899994675448655047","series":"1.2.826.0.1.3680043.8.498.53959362750834334682517691200216672176","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11264140845869814478794346490273061499","series":"1.2.826.0.1.3680043.8.498.40231152506973930380602765264439561983","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11270845320057097450047912107461684885","series":"1.2.826.0.1.3680043.8.498.51127898654226583264344460503556456682","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11276732020240247755618413418003401291","series":"1.2.826.0.1.3680043.8.498.14015016460217103543625771329856409272","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11282162920338602001456704018617788412","series":"1.2.826.0.1.3680043.8.498.12238374023946101585751373019545827175","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11287173314232027941677698384842764593","series":"1.2.826.0.1.3680043.8.498.98553211965994194777791744586892633291","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11291875173369366854889507729635968509","series":"1.2.826.0.1.3680043.8.498.80282726740800135557585073407167462736","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11296867854260286881418279496022384461","series":"1.2.826.0.1.3680043.8.498.10643041425601403055236532580166601013","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11302750434639976191333636496428935632","series":"1.2.826.0.1.3680043.8.498.24636372729805075163396302804426729561","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11312798102268182343541500463824266929","series":"1.2.826.0.1.3680043.8.498.48643952686501039038316084532549916497","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11315871586936178043350206344448851055","series":"1.2.826.0.1.3680043.8.498.54646240049519448387798356560719469069","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11334486185870324597335039163116965763","series":"1.2.826.0.1.3680043.8.498.10004845943302932961578284887251371028","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11338962847958062668991874160027887372","series":"1.2.826.0.1.3680043.8.498.29409917433980367359676038326809603275","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11345369648571407541082013397591193473","series":"1.2.826.0.1.3680043.8.498.10294048115798777353764436235282788705","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11353976221682532422630231842008146722","series":"1.2.826.0.1.3680043.8.498.66603810467246182298128269687447626598","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11369844885381156110080055326139875897","series":"1.2.826.0.1.3680043.8.498.26704922539513080082675845588574338137","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11383056981606331444758741778014516431","series":"1.2.826.0.1.3680043.8.498.12024910481191541372450070080102672779","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11386074316342852649653530230563922899","series":"1.2.826.0.1.3680043.8.498.47396621144976202956667392069364847338","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11395349736725035422736613041202251830","series":"1.2.826.0.1.3680043.8.498.21982980692561337562396223342757165250","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11406304618685790128709325242250648417","series":"1.2.826.0.1.3680043.8.498.29308880891099017934285663562836606543","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11414215447313658600502835138277811260","series":"1.2.826.0.1.3680043.8.498.94974014038294156161545805280295396600","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11425019165071014217290822852865654127","series":"1.2.826.0.1.3680043.8.498.58027857486509260945965118837529483877","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11428482061412578159115168314865188597","series":"1.2.826.0.1.3680043.8.498.15193333061892769887718612159505533288","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11440217845998767853986099058965644463","series":"1.2.826.0.1.3680043.8.498.28368027057975073982358453767084919927","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11446264194433135282793848490754001623","series":"1.2.826.0.1.3680043.8.498.18486014817387861248589956302243736311","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11451448731901996559834481853733404099","series":"1.2.826.0.1.3680043.8.498.20255172285986754995628462592761601729","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11455388290952699607973723923118848769","series":"1.2.826.0.1.3680043.8.498.13314422181886688623801814762150939118","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11458715255296685383451160973781612197","series":"1.2.826.0.1.3680043.8.498.12914861826968650398682345293674689557","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11468812519969607938527116557104410121","series":"1.2.826.0.1.3680043.8.498.13215071405293800009398422780660245599","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11472441320199233244785757534656853558","series":"1.2.826.0.1.3680043.8.498.20167281363967058284632483134882706434","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11482325994550095339373157256174349858","series":"1.2.826.0.1.3680043.8.498.11741428336848269563223885777047437535","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11495133447993934376048707763538781434","series":"1.2.826.0.1.3680043.8.498.84613136805735338238181168381713737774","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11498239309891927151456629915927889165","series":"1.2.826.0.1.3680043.8.498.89497909005532413408718568237510304154","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11498845504486569595587386245872511981","series":"1.2.826.0.1.3680043.8.498.38332240451597927440131173138215449314","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11506642732332295874076832453198901115","series":"1.2.826.0.1.3680043.8.498.64541314136350055714376254947004659004","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11508340006652282767645009271305921426","series":"1.2.826.0.1.3680043.8.498.88656406097274000393951494853291689730","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11516498203727058146656616413699979433","series":"1.2.826.0.1.3680043.8.498.67303631873542046346144000492441800531","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11521971289581057766555206729274639736","series":"1.2.826.0.1.3680043.8.498.19943511145976215889317793554440021609","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11530446425918247469882351709851883983","series":"1.2.826.0.1.3680043.8.498.30834998619975301005268800293577539235","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11542298537902846156898976374522470463","series":"1.2.826.0.1.3680043.8.498.94782647529732311416867641012294986501","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11546127923715190914128277599849591183","series":"1.2.826.0.1.3680043.8.498.44263240555865592653417251813746459729","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935","series":"1.2.826.0.1.3680043.8.498.38603242122624552076191735510850848597","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11551770225828320378339882849245348148","series":"1.2.826.0.1.3680043.8.498.12638273757648806562692506155774760712","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11561519302314146020052258325459371962","series":"1.2.826.0.1.3680043.8.498.10640286487152524062294877059352166139","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11571057273590444914079281238002921868","series":"1.2.826.0.1.3680043.8.498.24081649765614891317357810289459645002","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11574667600340420151274680172753922967","series":"1.2.826.0.1.3680043.8.498.12317497439028818950021792321485866878","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11579571696748051185342754881791715610","series":"1.2.826.0.1.3680043.8.498.78334325675611422740930744592516526023","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11587989960888698769912899464470116193","series":"1.2.826.0.1.3680043.8.498.45808935898690413867896274674087463262","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11592056376890319271545710970155310824","series":"1.2.826.0.1.3680043.8.498.81154520340729836602336014403805461037","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11595338216297251382188195835520492789","series":"1.2.826.0.1.3680043.8.498.17505962601960015648185436568973236643","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11598606533748611505510113033734692457","series":"1.2.826.0.1.3680043.8.498.23797436341955510434263164059885845028","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11604138460812393624691789275263090511","series":"1.2.826.0.1.3680043.8.498.12977043085895407399669296659191222802","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11610176134207131408691324676304640501","series":"1.2.826.0.1.3680043.8.498.11076965464675392441205848750435159583","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11619883329923252065839671636992770849","series":"1.2.826.0.1.3680043.8.498.10711778652343936898817968259934230349","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11629659616217352895144469980856259097","series":"1.2.826.0.1.3680043.8.498.13234857884747203207671880676206825852","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11641866550826314411250696478180861612","series":"1.2.826.0.1.3680043.8.498.47200298337002943087005842312118552120","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11646465814751330102194657154003685957","series":"1.2.826.0.1.3680043.8.498.73088753078609923730245447157351802940","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11651755040149549966484720727140190853","series":"1.2.826.0.1.3680043.8.498.92635077996293735124876819360660219982","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11655692245049710970881619360721342623","series":"1.2.826.0.1.3680043.8.498.58578007264632359441302129579231819719","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11661333712307504012375232163708358493","series":"1.2.826.0.1.3680043.8.498.23504403474054134514787919548193116501","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11679044101697033975478370934058085847","series":"1.2.826.0.1.3680043.8.498.12622307701760037610179730571716201765","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.11682243736539231948790371952356165171","series":"1.2.826.0.1.3680043.8.498.40308096676713629458504126523577820868","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11691245281415120685024733512803575288","series":"1.2.826.0.1.3680043.8.498.31471555538370296961776163182393207405","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11703195077261884009442719578918319962","series":"1.2.826.0.1.3680043.8.498.11792552682827599188884756566219638253","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11709909941833245598246010905322129256","series":"1.2.826.0.1.3680043.8.498.82632948695916629786257411046038972274","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11719502398697390809777045891822044637","series":"1.2.826.0.1.3680043.8.498.50788887103671364226199852932331775127","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11720664166420613534218395854543939928","series":"1.2.826.0.1.3680043.8.498.75614659816322192393677752654313364147","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11723945299462627737346907048285143063","series":"1.2.826.0.1.3680043.8.498.69036723389084954461458060957817500088","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11738872955471637985947490962985266753","series":"1.2.826.0.1.3680043.8.498.86081715224425882952148342040748199024","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11752091432368750506312921028559788225","series":"1.2.826.0.1.3680043.8.498.35333870836159296429634806417957802551","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11762851510866726051342964961256680025","series":"1.2.826.0.1.3680043.8.498.91603083526048744728588853038189255972","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11775352515236862473778507708524782015","series":"1.2.826.0.1.3680043.8.498.25049781253213576590800269566953468838","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11779387602988880691008088661075117406","series":"1.2.826.0.1.3680043.8.498.80226968363695214745776470997017181649","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11786296135006283515679011075884245663","series":"1.2.826.0.1.3680043.8.498.33812072061157209660778319631072657861","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11791028039098776993164396657149877483","series":"1.2.826.0.1.3680043.8.498.73942837544727705561373165608566783524","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11796547950032543541071896156852778697","series":"1.2.826.0.1.3680043.8.498.10451230569720702824303456456406121246","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11799272823743145498601173910069737855","series":"1.2.826.0.1.3680043.8.498.80195989331890473290751334689397478171","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11805131119810457174456565128522037991","series":"1.2.826.0.1.3680043.8.498.10398066629174996152955530252091061274","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11812607848520072826145426247813670653","series":"1.2.826.0.1.3680043.8.498.11616709651635069140837487106396618866","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11823120042104793954689778719229868108","series":"1.2.826.0.1.3680043.8.498.11158801361669284746726028757578830762","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11848828525319749834876185415845381479","series":"1.2.826.0.1.3680043.8.498.61201102809448301177438240454253849338","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604","series":"1.2.826.0.1.3680043.8.498.96460311887938476084320882783854369448","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11867198902144602395671904591496233129","series":"1.2.826.0.1.3680043.8.498.39047098027379245838149899743156800220","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11871142388984150456249558619289234011","series":"1.2.826.0.1.3680043.8.498.10057757335570259591178337456213772366","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11880385031047315524154761508473409762","series":"1.2.826.0.1.3680043.8.498.13148808767081867216386325402779913104","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11887143891500181344364122992066593229","series":"1.2.826.0.1.3680043.8.498.36236879841460975226271585200937983981","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11900355716044995266104285198121445744","series":"1.2.826.0.1.3680043.8.498.24302637484729845561274393665169226316","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11902464730418099436837163594625541861","series":"1.2.826.0.1.3680043.8.498.41046925041025367598425448480044401996","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11906266535122698149967624908585010734","series":"1.2.826.0.1.3680043.8.498.13734748701834340615761294744428904953","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11912276256810863127132294167870461707","series":"1.2.826.0.1.3680043.8.498.12649182938291734589841827373834156545","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581","series":"1.2.826.0.1.3680043.8.498.64673316729830153768997702312444992482","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11920026019322352546372153111621676066","series":"1.2.826.0.1.3680043.8.498.11039730084033679603271156549362533330","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11924999889245769866426422681253524920","series":"1.2.826.0.1.3680043.8.498.81634461208339171808958130816896175957","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11934483804567594363179973225449686890","series":"1.2.826.0.1.3680043.8.498.66465571253371823642110837773853980041","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.11935893582662912138522962760478513770","series":"1.2.826.0.1.3680043.8.498.75422311372051861583612357438023563098","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11944762815616453054528091721553484857","series":"1.2.826.0.1.3680043.8.498.11339513251046625391804349365434733377","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.11956771495610267604936834092317960600","series":"1.2.826.0.1.3680043.8.498.18429758990058807579207652351620353713","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11964826300307458382604479841253457304","series":"1.2.826.0.1.3680043.8.498.96840370346485950544813410240155463502","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11977465062890611952998635094033472451","series":"1.2.826.0.1.3680043.8.498.45870336278727100513028970659206802131","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.11980187124986749407584543247586384443","series":"1.2.826.0.1.3680043.8.498.12111954425606970772275976696587540070","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.11988551795896617815207749665102049499","series":"1.2.826.0.1.3680043.8.498.40474937549159218614525338583527956221","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.11992172807590757302730513420827032389","series":"1.2.826.0.1.3680043.8.498.31445224797992729756505560973797274677","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12006027138204983547873134458107233878","series":"1.2.826.0.1.3680043.8.498.11110432084916348799888209931840733462","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12009816058271421855131593296751372723","series":"1.2.826.0.1.3680043.8.498.11958921224434524887118800521857365886","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12015406188790105426257083591807220760","series":"1.2.826.0.1.3680043.8.498.66170346602565959357047922555459042050","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12031315219947933845048665397149229969","series":"1.2.826.0.1.3680043.8.498.31424284570339197940610467267801133181","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12036404333388388978317335770611404933","series":"1.2.826.0.1.3680043.8.498.27534058922725893623273561492253123848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12045300645580015230470991493790324857","series":"1.2.826.0.1.3680043.8.498.46533216494452349018875001427119448390","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12051169109587120248372991913002670813","series":"1.2.826.0.1.3680043.8.498.28635309647289187724760413207014964296","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12064822036518837577612019712902340953","series":"1.2.826.0.1.3680043.8.498.18170956660730717908370305329353348695","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12068674252637046261029360607215822017","series":"1.2.826.0.1.3680043.8.498.46172550716306571201941636126379196022","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12092744899746753313905432642371563786","series":"1.2.826.0.1.3680043.8.498.18073890757845468380664337390217909480","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12098718763390783455090805975786780205","series":"1.2.826.0.1.3680043.8.498.61158539488521846136697060297019310269","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12106775587428595108771304868101362618","series":"1.2.826.0.1.3680043.8.498.42913763641661014373745147946359596889","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12117377043894182951345966247899775612","series":"1.2.826.0.1.3680043.8.498.31865745507046166492274468137731567199","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12130868132537772155475981935868588761","series":"1.2.826.0.1.3680043.8.498.83817125439473147280383559232008298358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12135456587022560662140325717612726990","series":"1.2.826.0.1.3680043.8.498.55930869276105875857476089640592405354","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12148236440353424874546037143134438088","series":"1.2.826.0.1.3680043.8.498.95181419883257447244059708837422761930","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12153755823296803843819361055386265595","series":"1.2.826.0.1.3680043.8.498.38905141572602378162503594122893334614","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12160065317167766993207036411434801297","series":"1.2.826.0.1.3680043.8.498.96042754572468293593006040122218089259","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12166081590548185636776183599317715045","series":"1.2.826.0.1.3680043.8.498.75142999599986520948782029166885363253","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12170753305862973517190756418865414984","series":"1.2.826.0.1.3680043.8.498.20046450793837678728583655833082579924","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12171790864123519428649395041090102217","series":"1.2.826.0.1.3680043.8.498.20580632768083954994857307457380694197","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12176469158911896593955662590695392044","series":"1.2.826.0.1.3680043.8.498.33836612732028809598006095320453843161","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12182309401920763498667366352681074302","series":"1.2.826.0.1.3680043.8.498.13405876553136147461982222484731784457","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12186494601308291174548814564387764515","series":"1.2.826.0.1.3680043.8.498.11280063003956449674385599980582240272","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12195007606427082866449558290311905939","series":"1.2.826.0.1.3680043.8.498.80132502159191555095453395042066360691","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12201598444731259697380878112783653796","series":"1.2.826.0.1.3680043.8.498.85406638303322326638751092511005200266","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12207279366369605175255203969558548753","series":"1.2.826.0.1.3680043.8.498.82240758773355695461425525486004944169","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12219658385745696203053079500246399481","series":"1.2.826.0.1.3680043.8.498.10785490151346848997432767064917348210","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12225975251304101975065271521710735735","series":"1.2.826.0.1.3680043.8.498.11571719720681787806592394550773609989","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12232127448545053821926845887026738988","series":"1.2.826.0.1.3680043.8.498.12851049362466615188390693961781425112","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12241413500128553133597113709525575494","series":"1.2.826.0.1.3680043.8.498.92658717652212066844692366289075377493","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12243764506921404196245986681613075136","series":"1.2.826.0.1.3680043.8.498.23223388508898135198071962832096509166","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12246086238999179953298261549112671014","series":"1.2.826.0.1.3680043.8.498.48166118273537380990308720990636641339","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12256365473385257798561212457071712327","series":"1.2.826.0.1.3680043.8.498.11622244427668017926253508131840064660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12259896940707176389739553110492954399","series":"1.2.826.0.1.3680043.8.498.10603817152232917993904287377967073692","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12261175520056712414700866621816882336","series":"1.2.826.0.1.3680043.8.498.52523327222602745749596859523976496310","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12266133889557145601371064833389800065","series":"1.2.826.0.1.3680043.8.498.11840863379283649751884156405229799315","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12271616365050313084031931009756086606","series":"1.2.826.0.1.3680043.8.498.62499420094935238619045595091641148842","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12273490180961458167930039242826008249","series":"1.2.826.0.1.3680043.8.498.13112324679441568592670482510763915712","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12275265458583331749472987758046974406","series":"1.2.826.0.1.3680043.8.498.10361163772957053587148227437761903657","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12283662652762881783845275662580546217","series":"1.2.826.0.1.3680043.8.498.11841237029485294991632949700873592119","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12285469859900617016859868410323043734","series":"1.2.826.0.1.3680043.8.498.10577123123268523071425054660363343055","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12295284437770380354725533247473017729","series":"1.2.826.0.1.3680043.8.498.41686850985792334451814112057566190670","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12296624595750602443033689366743784387","series":"1.2.826.0.1.3680043.8.498.32845047960008317816572501179807242779","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12303161232396569915032951761728884199","series":"1.2.826.0.1.3680043.8.498.38602704360960995449006736305001764631","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12305733049746414359748178404911549573","series":"1.2.826.0.1.3680043.8.498.90028489328868076838128883203849815581","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12314729913655626716039376606973474150","series":"1.2.826.0.1.3680043.8.498.17932276748972129024189819309459308724","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12326930847104022196329390608780066947","series":"1.2.826.0.1.3680043.8.498.12193567937367537454939621656609592416","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12332727488899501354972751168284397151","series":"1.2.826.0.1.3680043.8.498.61105542277803416653474457601259851944","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12345064031074704673835101507426902246","series":"1.2.826.0.1.3680043.8.498.30730893215933587125724336384810056258","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12348352904452887226227118450588511578","series":"1.2.826.0.1.3680043.8.498.12508878965325568514753792031117325360","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12349656752404066719917550972333542508","series":"1.2.826.0.1.3680043.8.498.22006914887430713458803962034126918493","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12354685390005531691919754048141542648","series":"1.2.826.0.1.3680043.8.498.98042832657197173260675705753484134689","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12361144153852256276011948786791830435","series":"1.2.826.0.1.3680043.8.498.33198839297313879763745575893137054084","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12366238437963147962850341556508931973","series":"1.2.826.0.1.3680043.8.498.44142214287079494960741947214503494009","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12377852033407536077897684568488365785","series":"1.2.826.0.1.3680043.8.498.37965099687994002186385127651059264534","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12381325071746339154385415220972069782","series":"1.2.826.0.1.3680043.8.498.20583513414042591390037801913536192331","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12386255059403233724886571725482313876","series":"1.2.826.0.1.3680043.8.498.13025319146247515933048116262029420402","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12387241932914995417493041212958089910","series":"1.2.826.0.1.3680043.8.498.98472606956210113820090710568259786167","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12401147012438969595202076588471168894","series":"1.2.826.0.1.3680043.8.498.11949553542797307897688029002198238652","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408186961945791320086301369036829266","series":"1.2.826.0.1.3680043.8.498.17647763468639343359992152166887413117","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12408933790879905193765468926022393696","series":"1.2.826.0.1.3680043.8.498.28819100881071765109474574119548166956","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12411396124304166984011280826756258628","series":"1.2.826.0.1.3680043.8.498.26999053951114313703706393666150531470","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12431272351851438889289190308550697909","series":"1.2.826.0.1.3680043.8.498.10810513489517445373849716621344711909","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12435506274406010562396858512839065232","series":"1.2.826.0.1.3680043.8.498.13200788756898278404298226002219251156","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12438775479300781065075835270622538230","series":"1.2.826.0.1.3680043.8.498.22188541879379387356577552068102528511","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12462498156569316578044317782899399656","series":"1.2.826.0.1.3680043.8.498.84465920520247786145713363393087371372","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12471165461877621368840037945454256612","series":"1.2.826.0.1.3680043.8.498.45427100906831725757930199198299861219","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12473755232610784318190599222252893949","series":"1.2.826.0.1.3680043.8.498.59651370198825515685097710979565648848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12484700311993345120015691857089576898","series":"1.2.826.0.1.3680043.8.498.83168937132877718338408399578589811627","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12489752598467506094826904655029553699","series":"1.2.826.0.1.3680043.8.498.12707440311028479852734571117562305287","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12491395125580387847235487841392723348","series":"1.2.826.0.1.3680043.8.498.34300045281836897272759695888355186075","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12491796764641306046176352066215240074","series":"1.2.826.0.1.3680043.8.498.10347133353228423037706957824557835646","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12499208731138180053966892599139021038","series":"1.2.826.0.1.3680043.8.498.13265280838612421268780384513188459454","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819","series":"1.2.826.0.1.3680043.8.498.64033002899810126597833586036345623131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12509944331769453135724043962410129357","series":"1.2.826.0.1.3680043.8.498.22627974353995732614324412335631003369","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12514357232100339485873453417784586592","series":"1.2.826.0.1.3680043.8.498.11050298643629265286087857184983253027","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12520203592477518476890010185037272575","series":"1.2.826.0.1.3680043.8.498.22865703433413544067083326262396152344","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12525752631743097782762806859112218620","series":"1.2.826.0.1.3680043.8.498.10236474382053599715493576637386686074","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12526876471494728147635430816990398219","series":"1.2.826.0.1.3680043.8.498.11020283634771050365517944354105849855","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12533105585687352326867082376911240942","series":"1.2.826.0.1.3680043.8.498.87770198415788255003475312986690241003","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12548176380550230498070609559486512004","series":"1.2.826.0.1.3680043.8.498.10398612131266382653381374134850477481","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12553010776519233077752150342664160826","series":"1.2.826.0.1.3680043.8.498.60368645347605271956787164595143310149","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12559687473583953266143897736224215680","series":"1.2.826.0.1.3680043.8.498.10577529466230477561526072675705919143","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12568337973304376222940287737691898395","series":"1.2.826.0.1.3680043.8.498.43410753142192666470772871419969660991","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12573518342767030025851169814440863785","series":"1.2.826.0.1.3680043.8.498.88468237367706374407049611439634767202","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12576055921678075813718486704978381993","series":"1.2.826.0.1.3680043.8.498.76154803455618780654305632158339330939","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12579738538063674380235672702328412431","series":"1.2.826.0.1.3680043.8.498.80583742170789066543079553992055808228","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12598627296066406747921546933391681918","series":"1.2.826.0.1.3680043.8.498.14370125155144367669397352426783244294","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973","series":"1.2.826.0.1.3680043.8.498.12346063444250771269882141920568607762","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12607959449008110729566106776800628990","series":"1.2.826.0.1.3680043.8.498.54985002772077929835485622965613314531","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12616376042564380071463081913949255574","series":"1.2.826.0.1.3680043.8.498.53172918602591092663776828392059502090","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12625250919597111075072308824921101918","series":"1.2.826.0.1.3680043.8.498.12570788216194721359655398201312714654","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12629372332421954368597330267888483345","series":"1.2.826.0.1.3680043.8.498.15350996715244077434881612364565108405","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12632129640405661298623288977632682289","series":"1.2.826.0.1.3680043.8.498.11129839823637006492275400609448455482","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12638205949145637289277744267473965719","series":"1.2.826.0.1.3680043.8.498.37969838432712262237293461960151429496","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12644400073937544865441360809610886635","series":"1.2.826.0.1.3680043.8.498.93079173536862715735530461756561265711","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12653936535855858693516515692319733265","series":"1.2.826.0.1.3680043.8.498.61555237284369263110039698566846262118","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12660901572462562819575002681831361427","series":"1.2.826.0.1.3680043.8.498.10117696783557200606612802774791542882","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12673561338227595483297605409292531492","series":"1.2.826.0.1.3680043.8.498.10079696567208827375624803469579708071","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12676359576477452816896379524598090410","series":"1.2.826.0.1.3680043.8.498.16689639848740411661503192665210297946","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12689874339887047433232628649886917274","series":"1.2.826.0.1.3680043.8.498.72169450051466523499777912687750298325","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12692579161952587215276483587686562922","series":"1.2.826.0.1.3680043.8.498.32283456732796696712701756186543031101","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12699751030439008569144814819343161608","series":"1.2.826.0.1.3680043.8.498.31042332238898833866195195713786951050","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12701097867143790763644826659974337046","series":"1.2.826.0.1.3680043.8.498.31582904797407502277163889265440463928","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715067947657150279232238280712066972","series":"1.2.826.0.1.3680043.8.498.90153159764875070779190590607657267254","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12715402081094788880283826571710131940","series":"1.2.826.0.1.3680043.8.498.10101071807418048679151560051780509429","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12719397512102459853075227679498459688","series":"1.2.826.0.1.3680043.8.498.63830319032732665024711132602603776467","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12724250000565012779853884787450109956","series":"1.2.826.0.1.3680043.8.498.48343167332098408635254196198402167770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12727488524631885827798129418558343386","series":"1.2.826.0.1.3680043.8.498.10936627572040502009538635675572049963","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12748840878706905309050091576449488666","series":"1.2.826.0.1.3680043.8.498.89783516856900095974454258996753585863","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12761906443846433581503942530016459978","series":"1.2.826.0.1.3680043.8.498.83184587896160852070321883675372461872","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12762900657007480138202359874575403361","series":"1.2.826.0.1.3680043.8.498.41199897675090436072101942501555563986","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12769887673556938425710414718385118283","series":"1.2.826.0.1.3680043.8.498.27420718658199971456126307310178511819","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12775050278338123442507219047972133270","series":"1.2.826.0.1.3680043.8.498.68922966204841356674942365290104518493","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12778920060586364664953350981285170949","series":"1.2.826.0.1.3680043.8.498.10388292986269407338930117007157095361","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12785620442256422697216192462703273802","series":"1.2.826.0.1.3680043.8.498.74017352951013577673716137517663100325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12792431109288033212787919118858331455","series":"1.2.826.0.1.3680043.8.498.93977599708397082533246128144584019117","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12799849540231315910448269889175806934","series":"1.2.826.0.1.3680043.8.498.45888427689753496781764745850677340653","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12801643268939133682891906705186571010","series":"1.2.826.0.1.3680043.8.498.38818476452362437687332168365747860011","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12810074901642984174782853829149619043","series":"1.2.826.0.1.3680043.8.498.12997559110608825151892045851540801836","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12817753008281742621758605755308036035","series":"1.2.826.0.1.3680043.8.498.69365625149001027282224217812962474280","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12819437506700585448407291050085063174","series":"1.2.826.0.1.3680043.8.498.64282336084917173256555398124774600749","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12826828527176716109949704557150950675","series":"1.2.826.0.1.3680043.8.498.28525244953571217351795451911789567720","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12831769902826114242555323443809541052","series":"1.2.826.0.1.3680043.8.498.99713925352460368985647536465365184520","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12839498544183083639548298117995964432","series":"1.2.826.0.1.3680043.8.498.72000996394125268724493995602278076919","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12866937766198610064474201436305532400","series":"1.2.826.0.1.3680043.8.498.30601897555414017808738152177525325694","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12896740277483734090633132494285615502","series":"1.2.826.0.1.3680043.8.498.12230897518262674859174705692547158175","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12907457183137517669505067507754394039","series":"1.2.826.0.1.3680043.8.498.43289062375655177291807352287318872390","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12914677763148740041755977075799271057","series":"1.2.826.0.1.3680043.8.498.21716944522635811871109004072230902468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12919303621342250989179219420795277559","series":"1.2.826.0.1.3680043.8.498.42471652494499307651020394388141343569","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12936296973456045538701035923586653502","series":"1.2.826.0.1.3680043.8.498.12703208737622771547955069285708912449","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12946163460972553911280889205685745877","series":"1.2.826.0.1.3680043.8.498.49164488811345643382944926380179700670","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.12949322114105642048256996295428966924","series":"1.2.826.0.1.3680043.8.498.12603542347674812451301900557498572669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.12957252711278416684461267069498115130","series":"1.2.826.0.1.3680043.8.498.66376027941131610043783746783474749844","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.12961116776122838195555132309258963046","series":"1.2.826.0.1.3680043.8.498.12007850881223619783754105635995863143","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.12968535455086298222684461157825070395","series":"1.2.826.0.1.3680043.8.498.25391487215524942120059159099592744674","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12978510157202852202776899910529174803","series":"1.2.826.0.1.3680043.8.498.50206199772224497999907595310351485428","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12981413356524881858859912294078498486","series":"1.2.826.0.1.3680043.8.498.22318034120848891602182050229813502870","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.12985683732818911488433522743768046254","series":"1.2.826.0.1.3680043.8.498.10470114619868431273182715938839180068","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.12987428318535758055829100395092244687","series":"1.2.826.0.1.3680043.8.498.12711714041999848431166764428581972613","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13003520962744192762179786723724282013","series":"1.2.826.0.1.3680043.8.498.72347493306797938728236440444714462700","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13015502057686077045104167339480215528","series":"1.2.826.0.1.3680043.8.498.12430569127275728732014140127204798805","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13017199074506191657941905498813292773","series":"1.2.826.0.1.3680043.8.498.13216126506853748542297932132905565857","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13024250150960117282590397343184827466","series":"1.2.826.0.1.3680043.8.498.57644492648553085386251008075453217414","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13026024768908635083636086393662414870","series":"1.2.826.0.1.3680043.8.498.12871036502342638811688417538955451310","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13027024577729659065187419038453181852","series":"1.2.826.0.1.3680043.8.498.75415304318294255985406728338868874000","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363","series":"1.2.826.0.1.3680043.8.498.11876786729672065930957409970502140077","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13035632998746261521316395720102873141","series":"1.2.826.0.1.3680043.8.498.57690554036415847168399266572017846579","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13062438493893996542058869514390908492","series":"1.2.826.0.1.3680043.8.498.11261653077426031332625731979391724111","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13068696303639759696458130704554709416","series":"1.2.826.0.1.3680043.8.498.26487645637995498649072155343811541492","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13072908567275370866813748886351904593","series":"1.2.826.0.1.3680043.8.498.82440198300326652209569712434524754111","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13078009378379325641560321462188164769","series":"1.2.826.0.1.3680043.8.498.13076984349872897554519859353778082936","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13078834627392793401892111941627273421","series":"1.2.826.0.1.3680043.8.498.26151122523823505803814533541792365345","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13086260989634665810181105278930088749","series":"1.2.826.0.1.3680043.8.498.81321625024074362713260922734897134028","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13092094949394905924922534008771583745","series":"1.2.826.0.1.3680043.8.498.12769415042046191991454778888423020250","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13093751810869039645959119469231437894","series":"1.2.826.0.1.3680043.8.498.33714874472818368352229594557446553040","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13102058401501684064017656674827527261","series":"1.2.826.0.1.3680043.8.498.49190557910200362366557557688310048528","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13108275880471592702987713766776683590","series":"1.2.826.0.1.3680043.8.498.80483816063106470006568229294732562806","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13114693814781737495494700902699619138","series":"1.2.826.0.1.3680043.8.498.45426417046227664959475445349136324726","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13118325043138522704531128725514354285","series":"1.2.826.0.1.3680043.8.498.11315068662087746642011982355295035395","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13126070481622692281046898445918832841","series":"1.2.826.0.1.3680043.8.498.10446332118650617353720605002660282199","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13128352910661672966143173193241171100","series":"1.2.826.0.1.3680043.8.498.73371287416998815332597947415665180977","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13129258242739880102472095242703169773","series":"1.2.826.0.1.3680043.8.498.11383228498664321888074044748553558319","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13133755791254469039849289186598412660","series":"1.2.826.0.1.3680043.8.498.72520439398463655839241722637873177229","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13136367866260644484250529322937554927","series":"1.2.826.0.1.3680043.8.498.33656977322700069819916693377860635812","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13137510961466386836387239510287919268","series":"1.2.826.0.1.3680043.8.498.11403005586159518266737794005488822583","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13143446880514535170343243531129864116","series":"1.2.826.0.1.3680043.8.498.12930083115941481243809882526364266661","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13147249459968468500804358644901642270","series":"1.2.826.0.1.3680043.8.498.92511547416092417603133096979993214408","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13150794288945922409613136250303663502","series":"1.2.826.0.1.3680043.8.498.61040367579840097573263166113717702030","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13152690996267297597178587721720788387","series":"1.2.826.0.1.3680043.8.498.10166309717667796024783446307100927593","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13157776420553094741176082047621957775","series":"1.2.826.0.1.3680043.8.498.13155927792880093973460855227342868450","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13162284470388627411596358366248494046","series":"1.2.826.0.1.3680043.8.498.11258312941169368280500019210181090188","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13163727085668231782866823012535268245","series":"1.2.826.0.1.3680043.8.498.70918350809345643890054850143790718038","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13167727300538353052482338786183678684","series":"1.2.826.0.1.3680043.8.498.34271655265798169250066220343283406660","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13173781757294564816839795307893228067","series":"1.2.826.0.1.3680043.8.498.56257541184000733984863485620830871492","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13176146790594027813190776550419533937","series":"1.2.826.0.1.3680043.8.498.11992041556412338412217691019113032237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13197409499722328227876152940338160369","series":"1.2.826.0.1.3680043.8.498.24904224864321258842610845591392208184","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13203246712933982189680087013566455075","series":"1.2.826.0.1.3680043.8.498.51336276193509664685622532354085932717","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13210084125872700841685814144002495145","series":"1.2.826.0.1.3680043.8.498.96341698057613649514276891423679995276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13214267416673058581197714011913363750","series":"1.2.826.0.1.3680043.8.498.60467218753048134151865244795388961472","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13224933005220329091792052916905776921","series":"1.2.826.0.1.3680043.8.498.59480750415179500184169313802076584420","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13229563888378977350299319159748823477","series":"1.2.826.0.1.3680043.8.498.11682716608289085723738469147651351041","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13240749152416621711688053465035013133","series":"1.2.826.0.1.3680043.8.498.10685244774945705014783446619477228060","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13245409855393847268063261203253424169","series":"1.2.826.0.1.3680043.8.498.26762368894363633612747485584691583563","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13261595972961358158990410740461308054","series":"1.2.826.0.1.3680043.8.498.55878900350178600868889123979213620444","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13262473718294425031790270527413368245","series":"1.2.826.0.1.3680043.8.498.39782508957378881979972581093349987497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13267708980958529833316446819156908381","series":"1.2.826.0.1.3680043.8.498.22233408771360290614297260456034121157","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13273890409523605456488117581706115075","series":"1.2.826.0.1.3680043.8.498.70458801903453503855587116235138246802","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13283411144589724334502427926341646116","series":"1.2.826.0.1.3680043.8.498.12074756018258262201697558023606551643","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13286048014183811835847662577023535662","series":"1.2.826.0.1.3680043.8.498.82017207776080441435115348665764657413","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13291943370279119939499960846843879156","series":"1.2.826.0.1.3680043.8.498.12938648157859626963442327413579732127","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13295633458001436592005068637719195786","series":"1.2.826.0.1.3680043.8.498.99305956556113827429147507931253038707","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13312139694018789912028129989483016875","series":"1.2.826.0.1.3680043.8.498.41118461811104505121348891551661118041","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13319886927127591971322856062062654146","series":"1.2.826.0.1.3680043.8.498.60616598981412838032497260129578325155","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13328029743559327467301971917273397645","series":"1.2.826.0.1.3680043.8.498.63056906851455726206557914782436389611","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13332521365069604440230185026865466121","series":"1.2.826.0.1.3680043.8.498.85567070261412192312797359829304380758","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13341507176140340924941654608476073030","series":"1.2.826.0.1.3680043.8.498.21506789274380950263331276695048135055","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13353409340862821090624127486679097099","series":"1.2.826.0.1.3680043.8.498.85610759384836070208803930679827278017","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13358702230966891064267033765095124604","series":"1.2.826.0.1.3680043.8.498.24618077878651200118006993246980940606","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13363562142644508091380459785810556567","series":"1.2.826.0.1.3680043.8.498.78347974062067730359073852928732885389","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13367740087756379568826046059252678700","series":"1.2.826.0.1.3680043.8.498.19250813797275052096685451150664746714","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13369443704765101449274062863864122737","series":"1.2.826.0.1.3680043.8.498.34135020123377088433756212491044296399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13380735447520796902327272696579348141","series":"1.2.826.0.1.3680043.8.498.80435267251733058028182432912017654642","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13390551515761269433524925969561757962","series":"1.2.826.0.1.3680043.8.498.45577004058078194978884266589385913688","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13393268272072264469105890769640363680","series":"1.2.826.0.1.3680043.8.498.48349189156207473747487763070922619591","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.13405120554981098042415743645998258385","series":"1.2.826.0.1.3680043.8.498.17276488279720011745912583346912917358","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.13440909104569690641314836192184434461","series":"1.2.826.0.1.3680043.8.498.10197408536472958680584857503808570567","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13460078631594032326006554321483799928","series":"1.2.826.0.1.3680043.8.498.65769608325440510524895330892942689612","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13461073016327661159616957998589287085","series":"1.2.826.0.1.3680043.8.498.11537883234061070401165650555903248428","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.13575953449958751965232580597585303226","series":"1.2.826.0.1.3680043.8.498.13305408284306240547510108898994380198","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13616982533395291590655841043722813973","series":"1.2.826.0.1.3680043.8.498.71110794349651173186627170248617547989","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.13695989890112408005495264068064110179","series":"1.2.826.0.1.3680043.8.498.56839024366594270963222853649899944190","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.13829111847660458141056077151781118782","series":"1.2.826.0.1.3680043.8.498.12965501479603422905129870063503537796","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13915367834258386094001181500560177470","series":"1.2.826.0.1.3680043.8.498.14459123198346226009498054459693169011","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.13972325309264527611989585341636140156","series":"1.2.826.0.1.3680043.8.498.10885838708233481956985079426822149078","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14032903253348079919799627243690484791","series":"1.2.826.0.1.3680043.8.498.24447380001783732610619228669516198738","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14066537811671722419109228137841995847","series":"1.2.826.0.1.3680043.8.498.12494277882677982117179690056821947144","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.14188540532926067083914263496492255704","series":"1.2.826.0.1.3680043.8.498.11082101628939862120373446941059145951","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14229598279617168024212382336641104878","series":"1.2.826.0.1.3680043.8.498.35850260097964453137320183425471746639","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14300107110014472682373589824643789093","series":"1.2.826.0.1.3680043.8.498.10338385710790860548561110088160690528","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14377917653321813299237274871837695586","series":"1.2.826.0.1.3680043.8.498.11836116279135582536502745405631968062","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.14453847272947043746856632410254555895","series":"1.2.826.0.1.3680043.8.498.55179204923820778954254400849062729949","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14477978091360159774052709992034078166","series":"1.2.826.0.1.3680043.8.498.80081590466073981437934769266961308293","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14495028403937132710814780922642512791","series":"1.2.826.0.1.3680043.8.498.78862485815010288283529715012863698940","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14558486491688263090686005096911018438","series":"1.2.826.0.1.3680043.8.498.11526830812053522286770824095987282332","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.14744714658078107007743063095375646143","series":"1.2.826.0.1.3680043.8.498.83631968658028250900863155367673683482","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.14806659119591146961784818305031900863","series":"1.2.826.0.1.3680043.8.498.12021831996788539695385700645348287855","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.14850534390650184292244380816212794996","series":"1.2.826.0.1.3680043.8.498.11648567984890204372100494414033841840","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.14959732862523218997149041503831001633","series":"1.2.826.0.1.3680043.8.498.31613561414444483494983829841191685872","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15024602758408626305833210011882949904","series":"1.2.826.0.1.3680043.8.498.68374100393425590119767361079763027836","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15065556607080211420503043544482923211","series":"1.2.826.0.1.3680043.8.498.59118158434456430404358980287790490243","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15133818248365103660456294775831248835","series":"1.2.826.0.1.3680043.8.498.12495701555036223911988627707373123596","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15219824433491621460467738389345697247","series":"1.2.826.0.1.3680043.8.498.10129859915363358662742584467713887935","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15337067581747365729183643142512558411","series":"1.2.826.0.1.3680043.8.498.12510639347539854516980928381951224446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15376535778246978844757121776560020722","series":"1.2.826.0.1.3680043.8.498.99828733803849138453939669988780121445","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.15428051518761311558410606976607852351","series":"1.2.826.0.1.3680043.8.498.78275189251283420275694319852040815054","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15486463042236628073590077209793828847","series":"1.2.826.0.1.3680043.8.498.92030109539871470580497188123313237762","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.15539098084847502447238233198010208924","series":"1.2.826.0.1.3680043.8.498.20314738764751634145330365014872345179","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15599161312748548709240395406977042093","series":"1.2.826.0.1.3680043.8.498.12897389797232033766573204524033060085","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.15709232000390534671340322807442769184","series":"1.2.826.0.1.3680043.8.498.99983505777663770150961778920287449750","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.15863325224755490561125835019647303556","series":"1.2.826.0.1.3680043.8.498.50514196817816487484404908007305477421","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16051798246004436152597344963060310360","series":"1.2.826.0.1.3680043.8.498.77347732260670245266147977334901147482","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16101435445990180482665354366724730305","series":"1.2.826.0.1.3680043.8.498.80844551498742632653190582040082514542","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16164261456077269604929662134371603081","series":"1.2.826.0.1.3680043.8.498.22072278473564513750067749852745225215","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16225091905018667993201414191711776943","series":"1.2.826.0.1.3680043.8.498.75937767717577301334391001171655537719","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16318078762749420171833135145622117452","series":"1.2.826.0.1.3680043.8.498.15463925852646849368283621634317691104","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16370303687278282143772106415664171358","series":"1.2.826.0.1.3680043.8.498.93000536059902033019934073406615850506","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16469256144078887607437578587242258846","series":"1.2.826.0.1.3680043.8.498.36272623006124416735109362687518773692","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16534078612126477228267765508485297367","series":"1.2.826.0.1.3680043.8.498.36992640194531095139827034743817098896","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.16550793987525699514242241133391355835","series":"1.2.826.0.1.3680043.8.498.13346108579354439619027160644500005040","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16657553974040122719239506008948241297","series":"1.2.826.0.1.3680043.8.498.11094434102911077710899281291895083302","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16674237773422474998796191258459643061","series":"1.2.826.0.1.3680043.8.498.30465011790426290777205563914395279826","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.16709696133689803501344038926512501779","series":"1.2.826.0.1.3680043.8.498.10981150937599063048100613544895331991","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.16774450022563283092653506225603012645","series":"1.2.826.0.1.3680043.8.498.18866315960380092352344078295665072388","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.16845451648760595922758166878121325861","series":"1.2.826.0.1.3680043.8.498.41503289108239073968438775413780026596","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.16864649493021092944977766326505053153","series":"1.2.826.0.1.3680043.8.498.55260488237414809001536308036811068419","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17056997047488992455301411194938263817","series":"1.2.826.0.1.3680043.8.498.12863888109696927780340000481312604452","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17074746747536999918446136673966046653","series":"1.2.826.0.1.3680043.8.498.21391366533659449219705201843914186442","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17093025979426350838724553020541643719","series":"1.2.826.0.1.3680043.8.498.10120788168700979328849050053097795082","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17123244662958741977525806700986256210","series":"1.2.826.0.1.3680043.8.498.38761531310852667777172658059990743758","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17241550277273207658303011467222594165","series":"1.2.826.0.1.3680043.8.498.10862322256303236461435975213568110255","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17274490471295558668613733667390457509","series":"1.2.826.0.1.3680043.8.498.12609078075269011824818248791559464493","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17415936504684075438800895036740822145","series":"1.2.826.0.1.3680043.8.498.70891806429768932817684321568273035942","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.17427902052389033416295416299697276519","series":"1.2.826.0.1.3680043.8.498.98985497207454879255104706797323713893","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17596657287050550465463272509976106854","series":"1.2.826.0.1.3680043.8.498.50595036998349381288845152045974640874","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.17662880041629379035049367252086125592","series":"1.2.826.0.1.3680043.8.498.11225532576317555678326430491222394299","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.17741907690913585163443565102293192244","series":"1.2.826.0.1.3680043.8.498.10372005409589310670542007822090048567","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17769942698581399165275319502954100325","series":"1.2.826.0.1.3680043.8.498.86524909468217504800966262505430683747","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17844546765907321649997094867791102711","series":"1.2.826.0.1.3680043.8.498.18449110770459991745936951397216109055","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.17865613300681542407687000381594067366","series":"1.2.826.0.1.3680043.8.498.12030011115950915848840541515467610842","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.17967978373588651520812399064902037467","series":"1.2.826.0.1.3680043.8.498.83292896934928471902545604537088262881","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18080511167840608448833484000937072258","series":"1.2.826.0.1.3680043.8.498.37484348350854880574758272365057891482","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18113123512696698441392990570166201548","series":"1.2.826.0.1.3680043.8.498.74599614733226826608889416082514620468","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18175450083338316799857698898702333009","series":"1.2.826.0.1.3680043.8.498.13566161825998721776038404346438260355","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18230847745299902998425838090653825745","series":"1.2.826.0.1.3680043.8.498.10867238032823922439114175601497720647","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18315708641533862849721595991941979080","series":"1.2.826.0.1.3680043.8.498.11955157714217822375089857712542496758","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18389676218363070462526838642303976332","series":"1.2.826.0.1.3680043.8.498.17968834945182670122059503151681756028","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18403627643718880914843209837672242908","series":"1.2.826.0.1.3680043.8.498.10362923031810527018083407075434575411","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18529656610752887218162968403101035390","series":"1.2.826.0.1.3680043.8.498.37723934877769701049493400415661836407","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18614704377104425868545800677445972261","series":"1.2.826.0.1.3680043.8.498.13901079544488451676230732459760944425","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18671006890719350655167441375718348829","series":"1.2.826.0.1.3680043.8.498.31389532580696818767413647028310707811","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.18695727379446620258431748063749176639","series":"1.2.826.0.1.3680043.8.498.75144393291445383484891391817231709242","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.18758199166360428760124471581842191566","series":"1.2.826.0.1.3680043.8.498.16604793905442497849177596057030196338","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.18807952338149288346040972005636698334","series":"1.2.826.0.1.3680043.8.498.58303087844085789339254873594303304975","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.18879049235810379709186936621085221483","series":"1.2.826.0.1.3680043.8.498.13380033733394810246809931227438452140","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.18968967345606818331567480514437075646","series":"1.2.826.0.1.3680043.8.498.22879938123558227273674030499217185603","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19034825848229751701363461935217290295","series":"1.2.826.0.1.3680043.8.498.83099100581317914207590941885683834094","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19124879088354547836118593585646705198","series":"1.2.826.0.1.3680043.8.498.10929190237748516864924559465797531629","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19194090255958016414082788444327981898","series":"1.2.826.0.1.3680043.8.498.24555501325722382184149924819135514821","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.19198152070740205434848422424805073748","series":"1.2.826.0.1.3680043.8.498.30798504638467265143850048721886968522","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19374331384306309057311171252852774841","series":"1.2.826.0.1.3680043.8.498.90938475136185219500173550959901743399","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19439950168933299231685815631786334293","series":"1.2.826.0.1.3680043.8.498.95734032623208705512255917022892820712","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19512754570664577075249836633718382506","series":"1.2.826.0.1.3680043.8.498.17206643651328409788050472287350983868","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19573195389434178432956976770560221462","series":"1.2.826.0.1.3680043.8.498.84777295564595939149219048771120995161","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19624674714110931897302481276137067638","series":"1.2.826.0.1.3680043.8.498.10634106926603826527096532987600256308","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19722687746450343001601428638894755031","series":"1.2.826.0.1.3680043.8.498.40343874840444687228101563005549690874","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.19821498229702528191367700655079805824","series":"1.2.826.0.1.3680043.8.498.66264560776956844696116573754666892213","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19845711038895023426012115550237130530","series":"1.2.826.0.1.3680043.8.498.81546450609611914829873789799736455784","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.19887850741193535685529812263489892001","series":"1.2.826.0.1.3680043.8.498.78366110185732494999906925021098706057","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.19925337578871911327225897926650188274","series":"1.2.826.0.1.3680043.8.498.30476174448276845160808424010689496237","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20023559426196341254845485444587607391","series":"1.2.826.0.1.3680043.8.498.44580128648548841243425231528227059446","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20134168618929769962391688049013971830","series":"1.2.826.0.1.3680043.8.498.45159244927502720417238460552600386903","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20184541870535223885310764061146422870","series":"1.2.826.0.1.3680043.8.498.70521700752910770270351352831225078813","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20204907530808218314127264658715054041","series":"1.2.826.0.1.3680043.8.498.13370785903492304251833888338073507367","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20266523169651039493799975212435349766","series":"1.2.826.0.1.3680043.8.498.50647908153013537700151417449987949262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20375739629244657719385962474248679850","series":"1.2.826.0.1.3680043.8.498.22013217704457487557735926603260083187","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20472525526345683475357475395677168372","series":"1.2.826.0.1.3680043.8.498.49588273318489841193796334367476586358","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20509336519532058090925280050885010886","series":"1.2.826.0.1.3680043.8.498.81380150249570769227179940336120999132","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20522330861615570894210641293344468905","series":"1.2.826.0.1.3680043.8.498.36868720086593019908813924494589394221","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.20600318722531772866761132262772105640","series":"1.2.826.0.1.3680043.8.498.13328611406490515860479942493739379055","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.20733151481921261415295204209910573639","series":"1.2.826.0.1.3680043.8.498.81289767074312154162623534590058930715","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.20801986011906346655458394198951999413","series":"1.2.826.0.1.3680043.8.498.10885616184232770500255428299550507394","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20852351697358589606507109300359572624","series":"1.2.826.0.1.3680043.8.498.55297135162261229046028360762788497600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.20947841686457891570351579406919873254","series":"1.2.826.0.1.3680043.8.498.86234323357918283113102331452645874097","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.20969353615978236040448556356026349615","series":"1.2.826.0.1.3680043.8.498.51351735311004166283507441542721694730","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21060274255072711241653174133146945101","series":"1.2.826.0.1.3680043.8.498.39700814287826387820591764398515957415","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21142867813782396036287673290033987859","series":"1.2.826.0.1.3680043.8.498.85819893003754886749082417717736238098","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21294732642571417618432023007693943849","series":"1.2.826.0.1.3680043.8.498.12827494806031252073550523337032874033","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21322638458116180116030396109651929596","series":"1.2.826.0.1.3680043.8.498.31052903228558120438870663627173693471","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21413285047211074156709106352750752275","series":"1.2.826.0.1.3680043.8.498.12303270633074752055630498437932259449","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.21478660341431677113136859217606773871","series":"1.2.826.0.1.3680043.8.498.13445834469893531210728016842258975634","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.21558570608416990974711670673431746511","series":"1.2.826.0.1.3680043.8.498.74346074521741411461347164371622284559","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21606196343708804451390109412118036330","series":"1.2.826.0.1.3680043.8.498.96329972246084875093609878618295787507","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21636337451527049130190928852193267826","series":"1.2.826.0.1.3680043.8.498.47317638526018857985269804251260538965","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.21798342483930597563659425189400765869","series":"1.2.826.0.1.3680043.8.498.36670138749329922188617124588419319932","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.21868734450232158209414868901539475548","series":"1.2.826.0.1.3680043.8.498.53719300505946698075151357454327376426","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.21937000442536121262190458446148124910","series":"1.2.826.0.1.3680043.8.498.20541770403691678577267364860621717118","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.21991252201684723900045301917670383228","series":"1.2.826.0.1.3680043.8.498.72547433735522012013339746856698368878","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22025729757015485203756049440674914537","series":"1.2.826.0.1.3680043.8.498.71004682170600229100910805159512490956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22057596860528658818833903685827585805","series":"1.2.826.0.1.3680043.8.498.42106304499569603147468504892356213304","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22109739962224309418874538994436903404","series":"1.2.826.0.1.3680043.8.498.13067309220254856807615606366007869221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22126824950877890715014783370071292997","series":"1.2.826.0.1.3680043.8.498.11708393900969262433918234369635905893","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22249763791584161324425922920269052844","series":"1.2.826.0.1.3680043.8.498.11358071312845175980582430904719432038","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22338400095583099753286935111647807038","series":"1.2.826.0.1.3680043.8.498.12520639456358427210347123408940140563","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.22398041676780235097751294563297953841","series":"1.2.826.0.1.3680043.8.498.12182972646304550509070946257637553087","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.22407175411826914385667795929161924645","series":"1.2.826.0.1.3680043.8.498.74368606250009360116849917558220814456","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22480218016816209183384996714060953859","series":"1.2.826.0.1.3680043.8.498.10858940367705224577686445916721343490","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.22583985355672251619550173090948161714","series":"1.2.826.0.1.3680043.8.498.38805023809665479891643838969491489527","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22596004572382994883130705870731193046","series":"1.2.826.0.1.3680043.8.498.14990219036388817974432812768201351451","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22632919018724241015271844108954804020","series":"1.2.826.0.1.3680043.8.498.96616229739541250813793219899224903046","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.22684505799921376181913295621462268931","series":"1.2.826.0.1.3680043.8.498.10633521596412006807753833456931412906","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.22867776500261605752195206933939172136","series":"1.2.826.0.1.3680043.8.498.83970457906240706183142121956305461640","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.22985467719058223198642920273233975053","series":"1.2.826.0.1.3680043.8.498.96960887113358919529065626533742506883","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23043120269252616884307662769766701449","series":"1.2.826.0.1.3680043.8.498.12108183385863448431870286695679152134","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23065868481290440008286663193810913414","series":"1.2.826.0.1.3680043.8.498.78928611441786014193756848656431837751","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23081294963277525274551516232134221416","series":"1.2.826.0.1.3680043.8.498.95721951658360140578677457978901346662","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23166300530486600338569917595744444657","series":"1.2.826.0.1.3680043.8.498.25780033149976406243381754337682422031","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23394601328148358087542766577727876635","series":"1.2.826.0.1.3680043.8.498.93907780885839481665793072175656106037","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23422918444346962998731376196027420200","series":"1.2.826.0.1.3680043.8.498.10598348235067725994506140798807934807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23523176367321534630723586470190430254","series":"1.2.826.0.1.3680043.8.498.47346738682912959275839868823084469561","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.23686732555936573712275826587678502899","series":"1.2.826.0.1.3680043.8.498.38309977987610271114638343915985175494","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23719982492392272262684855675016475265","series":"1.2.826.0.1.3680043.8.498.64601794975335613180813524146877269006","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.23822313132245448173452915228147497501","series":"1.2.826.0.1.3680043.8.498.88910103466947298530714953911111920374","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.23849672490426589132734390202638283865","series":"1.2.826.0.1.3680043.8.498.77874791346378819343113775945914406827","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.23956169649865635269920565218680494366","series":"1.2.826.0.1.3680043.8.498.98109019056441502541420084792910705229","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24068239310155798723127213793623172530","series":"1.2.826.0.1.3680043.8.498.47723493356642983821571183809622981960","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24126422042626612740219325272793737656","series":"1.2.826.0.1.3680043.8.498.19160237476840923062962792900430241091","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24220732394554303343522053195582322788","series":"1.2.826.0.1.3680043.8.498.12710626739743068719749944883498853304","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24283484945988825507450239904606754405","series":"1.2.826.0.1.3680043.8.498.11265764468156960819685465711150461016","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24313647347273496316728514161733307557","series":"1.2.826.0.1.3680043.8.498.11695169296590771260021656632397642606","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24432243084670627958434695475653885175","series":"1.2.826.0.1.3680043.8.498.14487114335891648434539378322233699747","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.24529300482982370922479679056174585978","series":"1.2.826.0.1.3680043.8.498.88891132139321289273350793177360154346","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.24588524807799377432185260702152371503","series":"1.2.826.0.1.3680043.8.498.73504419929841608966519082433621200816","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24661137747502154216858169258339731263","series":"1.2.826.0.1.3680043.8.498.95075207181639818778426395476757700754","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.24714029982431301486050388345836960910","series":"1.2.826.0.1.3680043.8.498.46163036295152320285092476749535072078","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24808246532191535554375304081795021306","series":"1.2.826.0.1.3680043.8.498.12420583606473846676808224597548290232","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.24872746733270791851241663716306993824","series":"1.2.826.0.1.3680043.8.498.29585813457727992358848359847244692426","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.24940437326555494150792221007907527983","series":"1.2.826.0.1.3680043.8.498.70421260894040790671949910132471710773","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25088333117273619121936521384053709674","series":"1.2.826.0.1.3680043.8.498.56209194397143135186909148189233807645","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25107837907285885353845034555062540189","series":"1.2.826.0.1.3680043.8.498.11135551232280655630283914204833540524","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25156207169523051276979595262558652072","series":"1.2.826.0.1.3680043.8.498.11703901486351431573638385778664719744","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25196393144875332073285250936506555534","series":"1.2.826.0.1.3680043.8.498.44676149709194790995141411638418336107","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25259588756496584739814764896903786709","series":"1.2.826.0.1.3680043.8.498.10133898271966557360580225428554849371","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25292715244575997322139502072242615319","series":"1.2.826.0.1.3680043.8.498.10275241355317491859790242151815729501","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25431207706481141354761246438973384670","series":"1.2.826.0.1.3680043.8.498.86351831797114707208263788040521114867","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25480883220477368504575353972328124891","series":"1.2.826.0.1.3680043.8.498.20326306144251834051574362839496883854","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25512240159357553134500245590935010104","series":"1.2.826.0.1.3680043.8.498.25331823657548182591333082075781038189","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25550871519416839807700061317524004550","series":"1.2.826.0.1.3680043.8.498.30019299491787236201995632205319629551","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25616615589791001678067099907823838797","series":"1.2.826.0.1.3680043.8.498.15755633841664393015558520467781747203","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25643897330283020467388814112594686690","series":"1.2.826.0.1.3680043.8.498.16706284490885452348854061880220974496","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25663827732039068221184137540713755716","series":"1.2.826.0.1.3680043.8.498.91766886639559734183540213303933244308","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25695966186129395049687747156570466645","series":"1.2.826.0.1.3680043.8.498.16014775672714479586841217424844144524","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.25765711614507755527825654507920391096","series":"1.2.826.0.1.3680043.8.498.72578100286867773365980001252665223497","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.25766673238950703352855821505938941878","series":"1.2.826.0.1.3680043.8.498.97285082592822843951511058743726606325","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25815857653380672532983223088817283976","series":"1.2.826.0.1.3680043.8.498.23964233906613968970767373741405791297","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.25899681172115698926926115441929702502","series":"1.2.826.0.1.3680043.8.498.61600727380714416906011466113231847164","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.25928779639851562853500186715690156268","series":"1.2.826.0.1.3680043.8.498.33538658679879079607248232528143346353","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.25977532612504997807641730439567378227","series":"1.2.826.0.1.3680043.8.498.21299172006448034014133803292070356919","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.26006372873438224908989402529530833528","series":"1.2.826.0.1.3680043.8.498.57831131583737466356869960890296585706","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26066524302907849854339129271283426208","series":"1.2.826.0.1.3680043.8.498.92765594638588556968424369815661239560","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26201626427712146573573952548179938304","series":"1.2.826.0.1.3680043.8.498.85978827534777287764049173132048893274","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26266119962108543565575361589492142091","series":"1.2.826.0.1.3680043.8.498.69469841769883262260276339601367576945","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26351577824204863242390892891061594387","series":"1.2.826.0.1.3680043.8.498.12503955340230339341450331417166625782","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26431974442245797095978351976773354669","series":"1.2.826.0.1.3680043.8.498.12710974273568860049035247136190119823","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26485916396240515006042297316506505324","series":"1.2.826.0.1.3680043.8.498.67692066318721542441672895353298671051","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.26536992666456596813876340849399448897","series":"1.2.826.0.1.3680043.8.498.11087053584217547712226323035250837652","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26651323353208729423825552073457972796","series":"1.2.826.0.1.3680043.8.498.38245262347416966240983784013810397305","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.26790702379506190936834447203448882465","series":"1.2.826.0.1.3680043.8.498.10841555934521578415607333181263233138","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.26842258746390750314059176647398369242","series":"1.2.826.0.1.3680043.8.498.10662461909695175516192593157177068816","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.26875920364303755911342338961252130512","series":"1.2.826.0.1.3680043.8.498.99585583768865272249980838040757536596","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.26946980494141305258556994153410856428","series":"1.2.826.0.1.3680043.8.498.57588476976844048739570421456941616314","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27020884753639915245324093232218617967","series":"1.2.826.0.1.3680043.8.498.60655339980250867789624964257952928666","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27034189091164693165076510006436405194","series":"1.2.826.0.1.3680043.8.498.79177293890835683689156698508157656837","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27157074709305277871045901905120405881","series":"1.2.826.0.1.3680043.8.498.10833549821158743626476303944850789691","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27184343589256623289925252380526385858","series":"1.2.826.0.1.3680043.8.498.66625912927137345813421358918140064349","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27192678730060203972082511503480566620","series":"1.2.826.0.1.3680043.8.498.29119707506243242010517943353217828924","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27224327142578332905227939015479767285","series":"1.2.826.0.1.3680043.8.498.73084224539615474896673276879736168555","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27306039618857661522863167157208122018","series":"1.2.826.0.1.3680043.8.498.13393478786442021531940765318962191424","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27324284859290040148888098857880927353","series":"1.2.826.0.1.3680043.8.498.72718510497889166828019612400013462162","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27398034409980350921804398626171135707","series":"1.2.826.0.1.3680043.8.498.12297004800080352419619496249601568362","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27437263843446879932281897446100134924","series":"1.2.826.0.1.3680043.8.498.57952016660464582538662197640603722688","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27474440682441854303485403899867525864","series":"1.2.826.0.1.3680043.8.498.61812012353964895912870155839861203866","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27627190971637735537007595480396264063","series":"1.2.826.0.1.3680043.8.498.83063613621613272879580063581372865077","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27655410895168998898506695863000928400","series":"1.2.826.0.1.3680043.8.498.18651136525052656224858834816773430272","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27671779587445983390840223475173360482","series":"1.2.826.0.1.3680043.8.498.43559075250980184117750543810492368440","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.27723413941379681806317930349613970721","series":"1.2.826.0.1.3680043.8.498.12498153445691921521487655860723357724","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.27737101979486411869208236811885597351","series":"1.2.826.0.1.3680043.8.498.11375401144507611025734098969206135440","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.27796604203156439069742212039368371002","series":"1.2.826.0.1.3680043.8.498.16690436884198907202593933430098533310","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.27890860883826681731748268459485259935","series":"1.2.826.0.1.3680043.8.498.45972846249153383456968422639063567238","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.27934397251575987048967084987994202973","series":"1.2.826.0.1.3680043.8.498.79765867196658305739221024960435029306","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.27990965288663885936880638302158503894","series":"1.2.826.0.1.3680043.8.498.10958902123200715203414123706468016947","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28028890493668487238845659163252231337","series":"1.2.826.0.1.3680043.8.498.13181533798479209162419516648041128728","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28045606887691550130031344985830975325","series":"1.2.826.0.1.3680043.8.498.11793001257722432661039518588666951770","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28153286315438977482271934270979508349","series":"1.2.826.0.1.3680043.8.498.88919293598711895926687885239275621044","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28220626968612492259133318770141901567","series":"1.2.826.0.1.3680043.8.498.13227919874742439146561979562246858756","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28289838818003822677994912735532050038","series":"1.2.826.0.1.3680043.8.498.86786041446060107534219406277435930094","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28309626953500461628112343997868050028","series":"1.2.826.0.1.3680043.8.498.11239762079815561645892732387757510115","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28351884945918062351436698411322252525","series":"1.2.826.0.1.3680043.8.498.61375876835664325480148275674555062840","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.28481442608487320644829189127206806471","series":"1.2.826.0.1.3680043.8.498.25324999801914451583296574371879074821","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28490861391717560759667589681288427737","series":"1.2.826.0.1.3680043.8.498.94518730904457344044531076477394022905","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28595729988867300278425411189173775382","series":"1.2.826.0.1.3680043.8.498.10582446812781706134476928937778802488","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28628472249424790112957421215774036156","series":"1.2.826.0.1.3680043.8.498.10239460498983441748308716464988238612","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28646304373057947277669139986982208473","series":"1.2.826.0.1.3680043.8.498.13902942002683017754225308494693825100","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.28670999234453413675502551916979760344","series":"1.2.826.0.1.3680043.8.498.11681325567600955115695242010008812110","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.28746954364434312909368235768936980607","series":"1.2.826.0.1.3680043.8.498.11598971004955638092908442211531844854","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28779021651975021236045277212517629154","series":"1.2.826.0.1.3680043.8.498.75137642665896593071306438238009605681","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28888941572409703700315590339486528256","series":"1.2.826.0.1.3680043.8.498.17387681867672602389064614976683389689","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.28944156754203202831026290463520445489","series":"1.2.826.0.1.3680043.8.498.70197800335108786635124197375142057988","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.28954361146470748868500417103935640742","series":"1.2.826.0.1.3680043.8.498.74344212998010426323644806229859251189","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29090158364700006304493717187346344597","series":"1.2.826.0.1.3680043.8.498.12994426603617235015812283367841891783","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29280448517432444306374036429824425668","series":"1.2.826.0.1.3680043.8.498.16839209191572280030751487814836252417","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29343735048224321966824641432873034291","series":"1.2.826.0.1.3680043.8.498.12922456837707575798757388937858191890","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.29364640434226980751405780979697632001","series":"1.2.826.0.1.3680043.8.498.10816717024424779126205004461200183570","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29375052467745327893128894062568944496","series":"1.2.826.0.1.3680043.8.498.10461632583436213318621688189541850748","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29443692810539372078922146461173545451","series":"1.2.826.0.1.3680043.8.498.78097161498539620945791108040601576757","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29509443439138257275071354031180789464","series":"1.2.826.0.1.3680043.8.498.12235010397311182973352363212189919171","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.29739171278960083313048266628194182569","series":"1.2.826.0.1.3680043.8.498.18704896103765222130605380906521625125","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.29758331841591711520514486913006094054","series":"1.2.826.0.1.3680043.8.498.11926751237354738536756415811667807748","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.29826129803278661744835136993610646069","series":"1.2.826.0.1.3680043.8.498.29680543845414116685620386457774259127","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.29943089198208468441240036404506927883","series":"1.2.826.0.1.3680043.8.498.90496466568651666857198883378568852652","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30058585660043363937345050478072453854","series":"1.2.826.0.1.3680043.8.498.12189596919922151548090023723010578251","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30131339052241668380936148362472245259","series":"1.2.826.0.1.3680043.8.498.44576895405778054026019391609100391152","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30167179890902431464942556467698238073","series":"1.2.826.0.1.3680043.8.498.11881222448506041171705732727313412907","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30206987686375669521451035370290737934","series":"1.2.826.0.1.3680043.8.498.12925826706182201648260744986949811831","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30246079718471552972130572444383079911","series":"1.2.826.0.1.3680043.8.498.10125030789765676430404935648431987209","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30252501257451878373370333329266264057","series":"1.2.826.0.1.3680043.8.498.12819990572695764639571555444272449369","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30330001893437233719772129404417384472","series":"1.2.826.0.1.3680043.8.498.10929987231765391608979872936278292235","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30420245540489522189154634720778032533","series":"1.2.826.0.1.3680043.8.498.41091269814211590690687725082988668913","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30435446905709054197096833427672604260","series":"1.2.826.0.1.3680043.8.498.13202154906602142048832508927249987859","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30520444048368652235940346583890702451","series":"1.2.826.0.1.3680043.8.498.35122246082562707132968400320779162866","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.30557810085867943803354292590079995622","series":"1.2.826.0.1.3680043.8.498.12115489377119042546887053484406504682","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.30672390215843894382866701432377109444","series":"1.2.826.0.1.3680043.8.498.92119887443244571790609875893169883754","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30690232747840004203140392776057147177","series":"1.2.826.0.1.3680043.8.498.61430453634556909157680505158449539636","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.30756534269149851281518502475458067520","series":"1.2.826.0.1.3680043.8.498.13176775658641037067750707124752412085","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.30832771419882354941430609179589560374","series":"1.2.826.0.1.3680043.8.498.60778237799501481982570018423651837998","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30838586189496669675465778955893367497","series":"1.2.826.0.1.3680043.8.498.84728877396463898626420862246288300451","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.30942874102560162331831960649006411381","series":"1.2.826.0.1.3680043.8.498.85123180907835144244442998608733953340","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.30978547345367837000803077743620171961","series":"1.2.826.0.1.3680043.8.498.10939131305944184109348249100396462974","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.31079821574648958955911441626193848961","series":"1.2.826.0.1.3680043.8.498.54545439708447073091623966784611500149","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31099878224304681441631176766255029271","series":"1.2.826.0.1.3680043.8.498.93823448894983654879575945771055954803","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31171050331393253655760166990008263793","series":"1.2.826.0.1.3680043.8.498.19693203407222524566141345838817921987","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31175617350083221245857813741060042777","series":"1.2.826.0.1.3680043.8.498.79626404042730572306964551109631078956","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.31220970065722563280311764269386457090","series":"1.2.826.0.1.3680043.8.498.12263877684312131876827159700308494660","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31280487369922340170695453174997912050","series":"1.2.826.0.1.3680043.8.498.10670479927252278494415955060458328419","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31284572943302383501947308874558551301","series":"1.2.826.0.1.3680043.8.498.22730240003047429734884863015136291135","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31421989235441217674752676866539112529","series":"1.2.826.0.1.3680043.8.498.12143823277290261912006752636965893415","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.31555333355990829772911515362502000209","series":"1.2.826.0.1.3680043.8.498.80809100919300911284479453514836013270","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31610133883321894510649078615659708431","series":"1.2.826.0.1.3680043.8.498.70232897578699202788218895629154247993","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.31790457319528233269194462675995569260","series":"1.2.826.0.1.3680043.8.498.61339567050409973294129200885715928554","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31827987476038668074615045841102711882","series":"1.2.826.0.1.3680043.8.498.10203436152634579667950337981077670387","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.31851627014639252866084081561126592250","series":"1.2.826.0.1.3680043.8.498.87503225018017311531697288657837808562","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.31899121807187535405063131351086765296","series":"1.2.826.0.1.3680043.8.498.74207662530741940110019193222188952100","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32043012602421532910420313983803503778","series":"1.2.826.0.1.3680043.8.498.55206162279216324789542675587942141255","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32138869097959653224534642965199855627","series":"1.2.826.0.1.3680043.8.498.57329954331864679087923806785375252458","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32226253011052584155674401943276961907","series":"1.2.826.0.1.3680043.8.498.46504172063422730243511622041358876298","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32265622493980668732317541941196894795","series":"1.2.826.0.1.3680043.8.498.10544380559988770524387163425140986968","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32321830776739689645700555055955725945","series":"1.2.826.0.1.3680043.8.498.49657487656121528921826247613085032325","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32413534416513678586709935453506443461","series":"1.2.826.0.1.3680043.8.498.83218321867348166803697788823337126298","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32479930837336533083798155873656743007","series":"1.2.826.0.1.3680043.8.498.56671363613700482994361708825178259848","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32504785212530110880087745488871181111","series":"1.2.826.0.1.3680043.8.498.12287634077080248808200810596779473345","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.32521102053110469817073126425955562934","series":"1.2.826.0.1.3680043.8.498.10452377348451577391643835327018772872","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.32611545463217490575457831874695918229","series":"1.2.826.0.1.3680043.8.498.12126623712046680628976719190026289436","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.32680606387445562892045224227371304764","series":"1.2.826.0.1.3680043.8.498.10475152455923361677875212105011323328","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.32742383730417896527915284404039812924","series":"1.2.826.0.1.3680043.8.498.10541004440575230487762814889881843445","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.32835080036019877751472792740274297358","series":"1.2.826.0.1.3680043.8.498.43907976029565138410241950183282713097","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.32910372391370203605482772577821971133","series":"1.2.826.0.1.3680043.8.498.11317636104982470341075768297472156394","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.33045687849853291066101186962073873042","series":"1.2.826.0.1.3680043.8.498.98935189648345348802981929176091145073","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33114390259375873203134285352951765534","series":"1.2.826.0.1.3680043.8.498.31823481230813266800899333872823382636","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33247377165700300890880607485836825807","series":"1.2.826.0.1.3680043.8.498.11025924593973564590292021403800979272","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33324606756355240783076971858231422510","series":"1.2.826.0.1.3680043.8.498.18569073229648713118444464396962725644","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33339945632086976351786752793064764443","series":"1.2.826.0.1.3680043.8.498.22961805859022270302737540854317934762","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33419086668421177906530808458004386826","series":"1.2.826.0.1.3680043.8.498.73096267657989031356727605297949770184","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33479806419966566464727310013949313521","series":"1.2.826.0.1.3680043.8.498.14824776442053115250884947738944402741","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33614319611050071354692274770073371314","series":"1.2.826.0.1.3680043.8.498.31306530470335238216124343759077777772","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33707148112045883958228656491167647755","series":"1.2.826.0.1.3680043.8.498.84256545820583652655710251964915624141","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.33734536545775442043827840341408174347","series":"1.2.826.0.1.3680043.8.498.94583889234704652941693354133280972013","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33880013318702285028756482543453984598","series":"1.2.826.0.1.3680043.8.498.58598223371714259566817627040661820551","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.33899056854391617633929258302459140830","series":"1.2.826.0.1.3680043.8.498.19800975621334646202486476014921126320","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.33990091057243328830745904178686745167","series":"1.2.826.0.1.3680043.8.498.52306703205794875738655262318723666158","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34040631818813472592501398076075999835","series":"1.2.826.0.1.3680043.8.498.10362452507405962107035884938112738389","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34076627216941578548083498328399240999","series":"1.2.826.0.1.3680043.8.498.28252437031393470549191037048332786822","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34126843286927697018624309801437341727","series":"1.2.826.0.1.3680043.8.498.74872629878723553576155949153970867862","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34262242337778924864334788374037992215","series":"1.2.826.0.1.3680043.8.498.10241913284488506856021886958683640432","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34360298211002490108642003575962873984","series":"1.2.826.0.1.3680043.8.498.41612297579497557519573918923959703532","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34433343440016591348979084714300062954","series":"1.2.826.0.1.3680043.8.498.93893803014035237813223320157310786648","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.34461302710296907289543656738174044465","series":"1.2.826.0.1.3680043.8.498.86355677353229478165641193058044666247","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.34546106114963066554367298449299214677","series":"1.2.826.0.1.3680043.8.498.11431889471052410381932318693825860122","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34574046861696910828478657346725687542","series":"1.2.826.0.1.3680043.8.498.23068953529941258034164184248007275589","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34654275504962399799861007974465776328","series":"1.2.826.0.1.3680043.8.498.25366793924198908014635813558011177871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34717974828302316666446325243570202245","series":"1.2.826.0.1.3680043.8.498.10394947182375544247923690092905167176","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.34753112960188810416088070814158959510","series":"1.2.826.0.1.3680043.8.498.32814777107386143740762712424645328874","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.34795373994453585401035013232069587413","series":"1.2.826.0.1.3680043.8.498.96403389863486380985631365727999961266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.34861721301001052975626513770908147451","series":"1.2.826.0.1.3680043.8.498.21435673481175530020427614468186976199","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35015675197378142698488944856288804597","series":"1.2.826.0.1.3680043.8.498.20709005223603652707666184595718325348","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35026052487843947230089350615730580521","series":"1.2.826.0.1.3680043.8.498.52521355191860194286857109699895371041","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35090276406828326770336478753776261776","series":"1.2.826.0.1.3680043.8.498.11847655895575556597066951778898613720","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35127678462764233938878060108411012978","series":"1.2.826.0.1.3680043.8.498.63848714602553926125180588921688208763","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35230417208602644083092414161817929841","series":"1.2.826.0.1.3680043.8.498.13377942934303287691559584530462125327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35330802038595377723999341992570259909","series":"1.2.826.0.1.3680043.8.498.12494801806538052677947963008136129205","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.35526385429946467546636621603749416429","series":"1.2.826.0.1.3680043.8.498.76489247408227355566201702497901079264","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35572306796137927247415883445612561056","series":"1.2.826.0.1.3680043.8.498.68901813599797376084534531241080289018","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.35639250779062198905356481136455419313","series":"1.2.826.0.1.3680043.8.498.13216180728686960654468809445716940402","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35677515667357567818845364400903668190","series":"1.2.826.0.1.3680043.8.498.60317933987630548991966730473953052168","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35777075794856195065665011443485613681","series":"1.2.826.0.1.3680043.8.498.48553145476506237522507906012612308953","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.35820650938063925991582813178954225720","series":"1.2.826.0.1.3680043.8.498.10365333106224835337473605314499352089","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35836928928058091851993989259747528067","series":"1.2.826.0.1.3680043.8.498.10366721158066228347224354052990634314","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35871201339095431578743898508613853278","series":"1.2.826.0.1.3680043.8.498.86226959293307207526674664321630168376","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35923146679164443588381195543533271939","series":"1.2.826.0.1.3680043.8.498.56108295895501435590740421172820612095","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.35923753382244056317223397206343376734","series":"1.2.826.0.1.3680043.8.498.12469540067997703040137075758183557110","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.35942316720546803304976641081186518384","series":"1.2.826.0.1.3680043.8.498.86455955191431936699966775415654073943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.35971292331621735865925455612813197566","series":"1.2.826.0.1.3680043.8.498.91581050756578351007736422944490133341","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36035264348264378094569428634302238394","series":"1.2.826.0.1.3680043.8.498.13363113710269073089885764264317868673","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36131369236532190338276701919104598475","series":"1.2.826.0.1.3680043.8.498.28501839317373912271028142089327300511","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36180248571044206939894546783755557323","series":"1.2.826.0.1.3680043.8.498.81714587722657347907534386356827854744","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36221680997794815863088530960610069278","series":"1.2.826.0.1.3680043.8.498.67977447186055516031578884731521479404","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36293583160636884934622362743062825202","series":"1.2.826.0.1.3680043.8.498.59484989975387876296577388802720297194","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36315811661139684771871696847516662339","series":"1.2.826.0.1.3680043.8.498.44756393102662879975865759241058550942","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36488342368050295711098478501722217415","series":"1.2.826.0.1.3680043.8.498.10864759012440012514489092260902651252","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36588608184578739092409073014150374185","series":"1.2.826.0.1.3680043.8.498.61079340602562081687664005691997701248","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.36645945703275956149107297016813148264","series":"1.2.826.0.1.3680043.8.498.11713713757559765722122575351746549694","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.36688959138536319644194458211515765203","series":"1.2.826.0.1.3680043.8.498.12976965163258809346353334889431455784","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.36757476214794170118868510630678488185","series":"1.2.826.0.1.3680043.8.498.13186589109577867756411998086984408763","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.36846391753332367835804699576399553980","series":"1.2.826.0.1.3680043.8.498.90839174199960004681211380421774290275","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.36914870739904705183135763739850515617","series":"1.2.826.0.1.3680043.8.498.36478940238973545847153851554352484694","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.36995206854684735704099881847485073991","series":"1.2.826.0.1.3680043.8.498.82160159588748265159554462860370855971","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37007302166102452703965686598524467188","series":"1.2.826.0.1.3680043.8.498.27628924516744520824824088789752341639","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37046961643473249831189497076699393868","series":"1.2.826.0.1.3680043.8.498.13829779071465389637225068264799872501","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37110727485959679337431199930441321268","series":"1.2.826.0.1.3680043.8.498.44740462564219175136074314313464703765","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.37210322466522537523056852160944888029","series":"1.2.826.0.1.3680043.8.498.35864312498110361568963283857810441099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37332768072749723117839591872457536315","series":"1.2.826.0.1.3680043.8.498.37357508489378518541185440880746318859","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37371365833775442014007234936409739360","series":"1.2.826.0.1.3680043.8.498.56785067241941582901753684962414572099","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37413349933694832641902637847629969234","series":"1.2.826.0.1.3680043.8.498.69054129796246182337666805590756648784","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37432328401993499374675847888297416449","series":"1.2.826.0.1.3680043.8.498.12012409078457340543394612063520697405","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37463970179664003092832463732043124908","series":"1.2.826.0.1.3680043.8.498.74424371673301601191381873037919979349","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37508036912473928281276902883575142611","series":"1.2.826.0.1.3680043.8.498.50773273053517407621620396684232760415","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37558426555271635608942179071280432530","series":"1.2.826.0.1.3680043.8.498.20241526401693177468327261354103927153","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.37580209294522316445963449455348371815","series":"1.2.826.0.1.3680043.8.498.70975420155943793442364583176780259866","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37633035024065331400434438202028794597","series":"1.2.826.0.1.3680043.8.498.12013168279040944520665623593828056683","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37695832484726177391252504639155717084","series":"1.2.826.0.1.3680043.8.498.88501661419622535226286977376691532678","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.37782918081276484321055690005029111967","series":"1.2.826.0.1.3680043.8.498.34996583107217734371677195715590540571","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37838990693631869129590283596686992710","series":"1.2.826.0.1.3680043.8.498.74639140234447316010010921661359291327","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.37879265071349098151803248077034280617","series":"1.2.826.0.1.3680043.8.498.20007301914293970572685329969471659446","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.37921127120678939078868474340027123335","series":"1.2.826.0.1.3680043.8.498.61505758001336304154874173621995673531","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38005916106577944091708997377832932938","series":"1.2.826.0.1.3680043.8.498.12422603181508769181681643319054299908","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38039631093067555454700266397073080430","series":"1.2.826.0.1.3680043.8.498.30004284293515148839729339524735330486","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38069035744880456154810472748123499221","series":"1.2.826.0.1.3680043.8.498.16907848865790017273131451563671041609","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38104228771638294693272534500734878990","series":"1.2.826.0.1.3680043.8.498.41450057703909419015014818800052842601","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38224750236138877853888930317337541394","series":"1.2.826.0.1.3680043.8.498.46443397946938178297885855174516081875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38272345519163551901865251730721530169","series":"1.2.826.0.1.3680043.8.498.98857750621083131598679961947406262148","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38303307720827621886420004896097661262","series":"1.2.826.0.1.3680043.8.498.42055077384684642183187057420149338317","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38333934623775971943734069301776225018","series":"1.2.826.0.1.3680043.8.498.34497597871933402431711916811612246292","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38365187857616428023213060377488046375","series":"1.2.826.0.1.3680043.8.498.92486634947095174586300581834513602038","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.38420384692181444061893234153745102059","series":"1.2.826.0.1.3680043.8.498.43634846635683891833430988674530380030","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38485168845116671268554752030254772359","series":"1.2.826.0.1.3680043.8.498.49602514674842412155374926719814805685","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.38528511947818361078957998546327084024","series":"1.2.826.0.1.3680043.8.498.75443400945627442055405692992168039500","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38567342184301320850258918666303107813","series":"1.2.826.0.1.3680043.8.498.10121662642504419892894935579682497587","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38593351637971248239815933638950449587","series":"1.2.826.0.1.3680043.8.498.41117925977552250288164736479705218027","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38628165743951175602803204505216754926","series":"1.2.826.0.1.3680043.8.498.85327003504897515301592281470323768078","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38702677049080452580748506476598606517","series":"1.2.826.0.1.3680043.8.498.43684834072142449550962094490197370496","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38750761816114659458563557493234655604","series":"1.2.826.0.1.3680043.8.498.58464894901150612090666443768094865628","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38796519786809225694393617946244012065","series":"1.2.826.0.1.3680043.8.498.59643786981635405570833564480327149903","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38862475907405766248096419059100131863","series":"1.2.826.0.1.3680043.8.498.83115594503394841815040184287192451460","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38898732985465285166803609252960353405","series":"1.2.826.0.1.3680043.8.498.12375198413450463738124556810370164704","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.38925755872951351471027638563742132144","series":"1.2.826.0.1.3680043.8.498.23828477817697753245431248068900228721","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.38935694477580522005505074220300062204","series":"1.2.826.0.1.3680043.8.498.12691262274232911278210568132671685346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.38981318408220047366318316620365040799","series":"1.2.826.0.1.3680043.8.498.11388328869194382821102584221214456275","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39025433784711757736168236829157583043","series":"1.2.826.0.1.3680043.8.498.44087162717650993092512742298524197669","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39068632185013624181565849012182917752","series":"1.2.826.0.1.3680043.8.498.70311844370892917189399097668304574788","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39190090804401525573012570218955054016","series":"1.2.826.0.1.3680043.8.498.56951807221431865845625895727886096174","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39250844287482885227700969308333354206","series":"1.2.826.0.1.3680043.8.498.11554524531439141707983948605193611910","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39324076659429494696721784912308233388","series":"1.2.826.0.1.3680043.8.498.78812180547611642265380705060848300011","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39399668753650041254826892657562722764","series":"1.2.826.0.1.3680043.8.498.11133750552881038089542930737424842777","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39500553372517290823815606829518305500","series":"1.2.826.0.1.3680043.8.498.48864168281440917690854528188298055620","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.39578126940710463296572383286587298912","series":"1.2.826.0.1.3680043.8.498.31250794046548586869954514227673576136","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.39636073039676814228254715705296535819","series":"1.2.826.0.1.3680043.8.498.32649564267499545621944471422122040403","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.39787654910948635229369478006027304578","series":"1.2.826.0.1.3680043.8.498.47187110664266261760427133025409707423","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.39845300426582562781355154625767485381","series":"1.2.826.0.1.3680043.8.498.33648341154298041087189242627346888015","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.39932424305167611414527482641898907759","series":"1.2.826.0.1.3680043.8.498.10665320886287104594024035177097169850","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.39959632509842689058626880477779808473","series":"1.2.826.0.1.3680043.8.498.73790099724267285099858362938023525773","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40036526071307024601256787605725348112","series":"1.2.826.0.1.3680043.8.498.10945604693641043925246975242603447652","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40131899651518057045444739705301716085","series":"1.2.826.0.1.3680043.8.498.86962996356861619633458576903164237603","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40223239401390462911823193489179244754","series":"1.2.826.0.1.3680043.8.498.22391554814811908695246991616586331261","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.40254123984175499564145728084464192446","series":"1.2.826.0.1.3680043.8.498.52861028463132778312849888226098304151","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40269807401373034066229810558300266830","series":"1.2.826.0.1.3680043.8.498.49800901200137871773894950020909193985","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40372643537313871452054321931253684619","series":"1.2.826.0.1.3680043.8.498.85211254162796946471058150514717217236","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40419759559020255481034237210540871312","series":"1.2.826.0.1.3680043.8.498.10751241444601139437208916191140332435","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.40441879276927383122480339472406105572","series":"1.2.826.0.1.3680043.8.498.12988757606944885696326600159353181506","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40533893516114346078820881178450524201","series":"1.2.826.0.1.3680043.8.498.11619835364720138198531288819091490087","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40576954651310602807432132614823906315","series":"1.2.826.0.1.3680043.8.498.24995472757908429609094792398425977631","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40618283771317160262473004328692613267","series":"1.2.826.0.1.3680043.8.498.10726024243912266896051873805051972872","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40654763252951267464957826289984701872","series":"1.2.826.0.1.3680043.8.498.66613011688474943408309584856419495830","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40718200801895968234499223301900799376","series":"1.2.826.0.1.3680043.8.498.81184949013469079243232113259745454300","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.40823842819404259265489417470859248075","series":"1.2.826.0.1.3680043.8.498.12345797745201233471656310237346524607","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.40837296112545776586904280088902000849","series":"1.2.826.0.1.3680043.8.498.72646914279327534801309947596389854287","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.40930045208346676628855922755791787949","series":"1.2.826.0.1.3680043.8.498.93560223659963234300268288108842553279","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.40987687016832998820664054439678138878","series":"1.2.826.0.1.3680043.8.498.86768610664715369117925054000638424661","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41057428195711190931490429649014776496","series":"1.2.826.0.1.3680043.8.498.11592139375810578901377955303455634085","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41124244151692842048993594448992046260","series":"1.2.826.0.1.3680043.8.498.10580735046872528058385237783612283047","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41240503248178304281405515398211762506","series":"1.2.826.0.1.3680043.8.498.63390814724384722546998757068689017454","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41315359092354940082611381451934245995","series":"1.2.826.0.1.3680043.8.498.12634323663109377602621334390576364346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41366889271483886167284378157366992074","series":"1.2.826.0.1.3680043.8.498.16708994535314616008210116881470540532","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41439889720274233955076963938741061358","series":"1.2.826.0.1.3680043.8.498.17986142581864867502405787909226702725","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41564229176277158115386256329157870722","series":"1.2.826.0.1.3680043.8.498.12135806323410490042132499053609641552","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41600498783384864111179175518614837125","series":"1.2.826.0.1.3680043.8.498.42449379145780158531187185761913058425","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41619164749467663852546208143469380415","series":"1.2.826.0.1.3680043.8.498.78292404626810484188554767365014499372","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41657361598083583047831423768034379414","series":"1.2.826.0.1.3680043.8.498.36881074940626836936465899890655351112","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41672145969790132883562432403529293545","series":"1.2.826.0.1.3680043.8.498.12075593048117682398673136352492544745","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.41698150452794914665945455397334605778","series":"1.2.826.0.1.3680043.8.498.61632024638081900832246913333778939144","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.41769275806689876192405091865340286535","series":"1.2.826.0.1.3680043.8.498.16004496209816051115474542856829300688","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41810826648175757249770990269365542940","series":"1.2.826.0.1.3680043.8.498.22976017895287557571122777852760338932","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.41854610297797849683195403204758363476","series":"1.2.826.0.1.3680043.8.498.80541835872125771619954584436948165755","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.41939824564209194493855981805593285528","series":"1.2.826.0.1.3680043.8.498.74552417513517580047984198580976641880","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42018683052544325209168649309206539721","series":"1.2.826.0.1.3680043.8.498.88557868880959732384589055030157635889","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42051385452979123325023608442860201782","series":"1.2.826.0.1.3680043.8.498.76634083520183770820283634417810703308","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42130173624033946465704266507948122088","series":"1.2.826.0.1.3680043.8.498.51522529939766813582082661217282582157","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42221373360893091454504422964928099406","series":"1.2.826.0.1.3680043.8.498.12893773981280446555468960296036387242","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.42287188181305579531212818123200473947","series":"1.2.826.0.1.3680043.8.498.38381649716183582882450994852297342688","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42414269167296543219685897036829241885","series":"1.2.826.0.1.3680043.8.498.11781206795162664396223736693544432033","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42486421970612099441197418307958561877","series":"1.2.826.0.1.3680043.8.498.82263730299678112813211150495313606654","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42566296562794128096734153388840524012","series":"1.2.826.0.1.3680043.8.498.14575497290780438642976645978272611829","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.42598518026540237824479232797503218019","series":"1.2.826.0.1.3680043.8.498.37439861471292704193535503594776782975","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42634709955067416383713194799683605006","series":"1.2.826.0.1.3680043.8.498.68510231362264667192377881803127151559","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42776652100045349404152232788825670056","series":"1.2.826.0.1.3680043.8.498.81109974607530202047863548254012656487","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42816339547796375466199938922141987775","series":"1.2.826.0.1.3680043.8.498.11960002191971509658919874842025249793","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.42887028634948156598417204377884297339","series":"1.2.826.0.1.3680043.8.498.47918906530237904191565947488943528753","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.42921593713351589642747633505679170597","series":"1.2.826.0.1.3680043.8.498.96023023998929089449072934506292288286","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.42967970121147528831983176261453235744","series":"1.2.826.0.1.3680043.8.498.79546590238893029239812334300805587367","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43014288535685961874615794778401152033","series":"1.2.826.0.1.3680043.8.498.12304132439439269969120874704472934681","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43117007553940146763245418710268028741","series":"1.2.826.0.1.3680043.8.498.66921719609929770060078380067961720786","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43144544376687968624598661376472625623","series":"1.2.826.0.1.3680043.8.498.16198275638577964647983255279319997351","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43300763182450509440662392516941842389","series":"1.2.826.0.1.3680043.8.498.11108525064253273664909054185637327630","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.43339007024446439709455570527734115557","series":"1.2.826.0.1.3680043.8.498.65982530186879525357449085851562363055","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43412154740072739455825163326814980838","series":"1.2.826.0.1.3680043.8.498.55543254845463069711032842672107992199","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43439413368880313042951208701983487343","series":"1.2.826.0.1.3680043.8.498.66730912505808417168257915838368925920","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.43483537125776347707929354413013709736","series":"1.2.826.0.1.3680043.8.498.54153393479955156465825531662361588560","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.43692339773096477132070133629776394739","series":"1.2.826.0.1.3680043.8.498.52799257336802068784685802489910493268","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.43779228281849457650343989788070421822","series":"1.2.826.0.1.3680043.8.498.58697158451568995961645970067076532545","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.43870319732657730742794291443940225011","series":"1.2.826.0.1.3680043.8.498.45695898071525253103901516355358513461","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44024350801191362011463613321559949052","series":"1.2.826.0.1.3680043.8.498.72155128776079009546916411090424123371","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44239214982361627984688924889728201208","series":"1.2.826.0.1.3680043.8.498.37041800573492571824518905876032004722","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44321626715626508427287815810252844889","series":"1.2.826.0.1.3680043.8.498.16001736051529196679474919011166268973","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44350638600163778456163342454087669278","series":"1.2.826.0.1.3680043.8.498.77410344123876936289425833766071514262","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.44405554266141576380633014711246118209","series":"1.2.826.0.1.3680043.8.498.12813609970531935880870974030102323620","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44441813293962885868294317830142850713","series":"1.2.826.0.1.3680043.8.498.12184845586835827119983351499381837116","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44525218699266472141419416162979441275","series":"1.2.826.0.1.3680043.8.498.12530795601482381594400776336843528827","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.44654558140536370040701171884970036040","series":"1.2.826.0.1.3680043.8.498.14148780559834467997176854326167640543","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.44733200650377282012534389271385687852","series":"1.2.826.0.1.3680043.8.498.15899299663188983350517520841365616252","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.44863396093454506630473164990623458425","series":"1.2.826.0.1.3680043.8.498.11299286471395366936375111730382592267","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.44879085997210091159117906674172231025","series":"1.2.826.0.1.3680043.8.498.18307908339064206629810775760385822769","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.44902036121220687805715010480088379768","series":"1.2.826.0.1.3680043.8.498.49789173588764603179665471342852033990","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45033326048521572717771708818514772861","series":"1.2.826.0.1.3680043.8.498.11211349118565839950434943234258854592","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45072896942121803896760563154385918032","series":"1.2.826.0.1.3680043.8.498.14966808022682164231625728942255545574","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45170517567841608067443743961197123093","series":"1.2.826.0.1.3680043.8.498.24291030343955932460771102296456943265","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45251306032215692953561852960585282987","series":"1.2.826.0.1.3680043.8.498.46138920610931837982330466494030371919","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45304570280714567513866386380356988876","series":"1.2.826.0.1.3680043.8.498.45260117272445823764442174271447779951","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45326662718293919909188368237039419949","series":"1.2.826.0.1.3680043.8.498.22265759881506524041031882598516894705","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45367987334597713963053051542179369641","series":"1.2.826.0.1.3680043.8.498.18774352291692162618374228737624304642","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.45392604471538776666611947492530314159","series":"1.2.826.0.1.3680043.8.498.41591780465941277219969469600527946021","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.45462389009834095091353507592651342877","series":"1.2.826.0.1.3680043.8.498.52936722443266899999602332730928069622","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45491168638937605395466251659273753775","series":"1.2.826.0.1.3680043.8.498.34856655088608051193027981971778230396","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45625191292968830335971647388746041117","series":"1.2.826.0.1.3680043.8.498.22766613323484845440255278004757340221","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45678969592219293243366784683191701674","series":"1.2.826.0.1.3680043.8.498.60082250324040882413467873684386936649","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.45768777378108699271714894225826640121","series":"1.2.826.0.1.3680043.8.498.27968774426903191053099180922164698662","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.45865541491386404818767748310365919685","series":"1.2.826.0.1.3680043.8.498.11576833902240738015401177256161381126","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.45894475516096841241467193775629362381","series":"1.2.826.0.1.3680043.8.498.62674946111122354577812166428549739397","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46016776602328665980436110567854612595","series":"1.2.826.0.1.3680043.8.498.45984952056921978268841253852385334621","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46051119885630602872050329412008545893","series":"1.2.826.0.1.3680043.8.498.11878321781274509625920460709000314871","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662","series":"1.2.826.0.1.3680043.8.498.35313574804080516711014517994642675898","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46228189366201925238527911905072482095","series":"1.2.826.0.1.3680043.8.498.21299325720908440950576041203666027077","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46282234455466954264170065462242039568","series":"1.2.826.0.1.3680043.8.498.60796414777164085165944027537396625244","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46305694398420300356121078124030126469","series":"1.2.826.0.1.3680043.8.498.11047372638370056735794466428153179990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.46407639440476738746130651250276012493","series":"1.2.826.0.1.3680043.8.498.12897773444622584235007697835650348126","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46491051379327985199309365945879841064","series":"1.2.826.0.1.3680043.8.498.11721504699496140107344726147373395303","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.46658713832425741743740258795376270265","series":"1.2.826.0.1.3680043.8.498.99565200281316742274797923909330220031","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46715670142711601593792518196789347552","series":"1.2.826.0.1.3680043.8.498.58888108173820433172363501365259597609","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.46770589129181839292542004762882468541","series":"1.2.826.0.1.3680043.8.498.10997772432450169686297210957553151404","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46825768767498441994570613185981335949","series":"1.2.826.0.1.3680043.8.498.82031261657776553210121022141251619208","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.46884541791777826238103426426267571657","series":"1.2.826.0.1.3680043.8.498.12699083748638557392448874671325170028","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.46925959240037679795735705370016021232","series":"1.2.826.0.1.3680043.8.498.10194177062987556934840407609958084206","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47036446751578280498105392189811189787","series":"1.2.826.0.1.3680043.8.498.27670531428479730439451233836802229810","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47063694709201049930726855357000925037","series":"1.2.826.0.1.3680043.8.498.36979995627551181782560535080915965821","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47130872195336642107576758658144949813","series":"1.2.826.0.1.3680043.8.498.12377610240395415081977273649387056760","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47255564202539432049700324486197024855","series":"1.2.826.0.1.3680043.8.498.83331010660039849502380127709886784616","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47336007784831556446570294227732688335","series":"1.2.826.0.1.3680043.8.498.84382647352699283727049663462375310142","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47449893776425045020340604662487159684","series":"1.2.826.0.1.3680043.8.498.53942307723623156138515797823672616120","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.47495385306095610608327984787779011133","series":"1.2.826.0.1.3680043.8.498.53267513935604426819345085791912274744","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47651715105501347255071941051553742767","series":"1.2.826.0.1.3680043.8.498.59680999638825047371946289358267805323","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47678121014268671436574141288012829166","series":"1.2.826.0.1.3680043.8.498.12149213058933319964997605593917389487","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.47753549323738763346112850705924843959","series":"1.2.826.0.1.3680043.8.498.10046105027332935089397099816715465807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.47864423097056282241181943425451597067","series":"1.2.826.0.1.3680043.8.498.35890258285076351688871116158567875934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.47925688677871325890380893763458019552","series":"1.2.826.0.1.3680043.8.498.11189421668918601131691635022674064118","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48033467661219494489168041469309669510","series":"1.2.826.0.1.3680043.8.498.79175843789635418417790893141172076744","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48160569779413970442472500116176876660","series":"1.2.826.0.1.3680043.8.498.96672304157304724550963690671534031379","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48348615349418615469875801439348424274","series":"1.2.826.0.1.3680043.8.498.11341617802848274596509422018595940673","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48417903971145834309474934717837250776","series":"1.2.826.0.1.3680043.8.498.11861500642378962757728267542375770470","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48486018503198264262324670490116120612","series":"1.2.826.0.1.3680043.8.498.28163514420345808735896923717886973131","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.48531746264504523370547117933271161633","series":"1.2.826.0.1.3680043.8.498.82389795930140697945671946383968250876","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.48658459018915173972212724570862424208","series":"1.2.826.0.1.3680043.8.498.38058141563383741292822024104201542479","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48771187298294322173131630438467163244","series":"1.2.826.0.1.3680043.8.498.10927350539337657885835136805129672903","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.48784244479029704181755878324450207126","series":"1.2.826.0.1.3680043.8.498.61398390583574261607460531941620875255","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.48855562507887692822284056171390459619","series":"1.2.826.0.1.3680043.8.498.57927024705912721772953222869307556059","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.48918090379630186340047006216221526186","series":"1.2.826.0.1.3680043.8.498.10723412479261436371720870836312279004","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.48977699132767849812826572970218325717","series":"1.2.826.0.1.3680043.8.498.72151358085631681865188469695059040712","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49047255106144196602523585429683869547","series":"1.2.826.0.1.3680043.8.498.98506674391696308092224182805359424007","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49054454369199540406654153762248931799","series":"1.2.826.0.1.3680043.8.498.12482239242973924496821147551828939336","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49186859342644783675130322288585477727","series":"1.2.826.0.1.3680043.8.498.82792977516051877599283612851392105341","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49287143478474218875661109488367501329","series":"1.2.826.0.1.3680043.8.498.80509723849803836095750074208160243368","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.49379034023506626029648394840207360925","series":"1.2.826.0.1.3680043.8.498.12386633740825580512546560573634992807","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49443962795025989994487968185070137217","series":"1.2.826.0.1.3680043.8.498.10851306295499637036209644836353497860","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49492260217998945587168161555623476238","series":"1.2.826.0.1.3680043.8.498.67411756021840207673352236286942586848","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49575149159425506211571562732929634662","series":"1.2.826.0.1.3680043.8.498.11714204221642893797262713556622239643","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.49603554452147117475589797389254733654","series":"1.2.826.0.1.3680043.8.498.49806174979316906198662965012086613430","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49679916686449433962263937489400526112","series":"1.2.826.0.1.3680043.8.498.47238200259005827026679449390227965808","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.49847104080351361589960954776757484668","series":"1.2.826.0.1.3680043.8.498.19614126249806044249277531486741739952","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.49855677166670499939366963036804180912","series":"1.2.826.0.1.3680043.8.498.88133286957743583898423557094271295483","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.49884243466758407760126403944842240776","series":"1.2.826.0.1.3680043.8.498.12751186650260025914636143274461736353","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.49926524898703962863714803838421958913","series":"1.2.826.0.1.3680043.8.498.43529467600902812161692354863736364551","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50051845967696843074391018845332480653","series":"1.2.826.0.1.3680043.8.498.92612628698853438684785983013247647250","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50073674465735509400964384004695041387","series":"1.2.826.0.1.3680043.8.498.73698444615572254663615855747401138612","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50109255875403812845391217314726500341","series":"1.2.826.0.1.3680043.8.498.11021944862935937765671571459227962515","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50182378358892377789975570268512056553","series":"1.2.826.0.1.3680043.8.498.67587082438954037521961705234579379384","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50220991646207533313828550949660577504","series":"1.2.826.0.1.3680043.8.498.89997102411946684970830795960122696900","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.50261788831539215224639813005933285052","series":"1.2.826.0.1.3680043.8.498.66957136753349862633984449850269391438","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50318760104506317341991314680640521601","series":"1.2.826.0.1.3680043.8.498.12053566315116574221917468139728402480","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50424421287869498746932271082980000521","series":"1.2.826.0.1.3680043.8.498.84822031771315257036420369113622737433","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50487158921933419035809338652990508020","series":"1.2.826.0.1.3680043.8.498.48877369629592430806470867242342534543","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50512021577184367495598050974910055550","series":"1.2.826.0.1.3680043.8.498.51199654503589442400652028358636938628","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50666048236278074641860077012751071758","series":"1.2.826.0.1.3680043.8.498.31570597424367006466707626738610671227","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50680622091377404463269575858175673158","series":"1.2.826.0.1.3680043.8.498.71327909016438723938935336549313144927","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50689256431939553935971505794681061679","series":"1.2.826.0.1.3680043.8.498.12544013351943942030727609430859035719","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.50777252749942894325007417893973309310","series":"1.2.826.0.1.3680043.8.498.11032232415947807771056431679541477925","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50810596378229719761556926464290468652","series":"1.2.826.0.1.3680043.8.498.66035774859586717116212274881033321796","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50823279853002320074080586693604976784","series":"1.2.826.0.1.3680043.8.498.33420513974053271270405468982313832026","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.50848799900564667487859253004538870622","series":"1.2.826.0.1.3680043.8.498.78286642687972842198525782467639444356","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.50935247372419775856440294835587827419","series":"1.2.826.0.1.3680043.8.498.12829345640639606866995847047842392815","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.50980137249034338585131194965993785201","series":"1.2.826.0.1.3680043.8.498.65959640740308057155386513402569801628","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51059575642903646008860869691477305022","series":"1.2.826.0.1.3680043.8.498.68562172258124436840451630211011357198","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51179760508587500025973916664101120069","series":"1.2.826.0.1.3680043.8.498.29975055504114271330926630889611320584","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51297777726540529860433097009654453742","series":"1.2.826.0.1.3680043.8.498.31412817314575281309758997274598861631","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51379389400506715529323532959275254794","series":"1.2.826.0.1.3680043.8.498.12586060336129580978184404652291508740","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51429653138043597611341459799295143598","series":"1.2.826.0.1.3680043.8.498.12814940184191743274736867309519528350","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.51536234256866063100891626028999490545","series":"1.2.826.0.1.3680043.8.498.26286971158421418683422550274718023269","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51676679394093588751371807453793655464","series":"1.2.826.0.1.3680043.8.498.11004080566033948299888469325032089956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51768733026033198941231586462443503827","series":"1.2.826.0.1.3680043.8.498.46379812658536833595184032027184160432","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.51782185934115000642557689300687723099","series":"1.2.826.0.1.3680043.8.498.12295330854999557416965800312449229662","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.51828180600648126726057100018010765613","series":"1.2.826.0.1.3680043.8.498.31109955817121883638970197619869819684","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51837740845904075589194479205218060915","series":"1.2.826.0.1.3680043.8.498.85942506008889131807374431729152646084","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.51910852739185117358682097116614556910","series":"1.2.826.0.1.3680043.8.498.70352187498341071328229980976334546771","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.51970198810758997428998886534742243215","series":"1.2.826.0.1.3680043.8.498.37198816986725739671224707965458300349","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52010165788889734743850946282043771445","series":"1.2.826.0.1.3680043.8.498.36872071399147246467884287459680651942","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52046170319252016846878355722404423329","series":"1.2.826.0.1.3680043.8.498.10217063640203976505090272981816460689","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52098249444968803324423421624756966294","series":"1.2.826.0.1.3680043.8.498.94506311300679619635623132540823567097","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52132258638267172087240021957310219977","series":"1.2.826.0.1.3680043.8.498.52079337482948602738649817105640520777","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52140911874747534807033188343849537207","series":"1.2.826.0.1.3680043.8.498.65384049635301144269159561453150002494","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.52347547859131274836026480407333415929","series":"1.2.826.0.1.3680043.8.498.89756552109846509408063456998322321588","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52409113683924904563676007800283978062","series":"1.2.826.0.1.3680043.8.498.55582499863643603505978572089907396086","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52456272643822591755257802747431239152","series":"1.2.826.0.1.3680043.8.498.13216272980642325224552896797918923068","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52499379042723926379822784411787551701","series":"1.2.826.0.1.3680043.8.498.39746801909137860431876215696558512461","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.52599302991259061057950834864441634066","series":"1.2.826.0.1.3680043.8.498.12102373361710105565752289171202453510","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52639459702574533721011212286428487650","series":"1.2.826.0.1.3680043.8.498.11854336614911285190977202152660841370","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.52844966504845792334740560021996858939","series":"1.2.826.0.1.3680043.8.498.82102558528932730822715706596985613060","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.52849154011665864497632297320961015159","series":"1.2.826.0.1.3680043.8.498.54462373799603350275910573640219360790","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.52985748674486376309978868177895755694","series":"1.2.826.0.1.3680043.8.498.56080746710962909936548060775943401509","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53078461307849490495536549806091931824","series":"1.2.826.0.1.3680043.8.498.44871970790321306731639040843645623470","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53094450151502907325041400000717725711","series":"1.2.826.0.1.3680043.8.498.11959360677387766153956951509499371016","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53137953094305471774552883960361922832","series":"1.2.826.0.1.3680043.8.498.11004622529497181556077977475741701961","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.53173579822475854424126184867040523127","series":"1.2.826.0.1.3680043.8.498.60065996902544749138600770524386451240","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53262958748075737242820629127001253146","series":"1.2.826.0.1.3680043.8.498.46203813180350963903524661369697512262","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53331626867618182966740022261110760133","series":"1.2.826.0.1.3680043.8.498.95582854088759363041140572026863870820","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.53343379040681745197301575417064236501","series":"1.2.826.0.1.3680043.8.498.13343682341871016534458114897824337251","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53370670740736715669115043403494610200","series":"1.2.826.0.1.3680043.8.498.10287760183216902161714739906565413858","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.53490080963614272234072703760363052476","series":"1.2.826.0.1.3680043.8.498.86997783039835350156144124289839106669","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53579416035647672067965892020505016454","series":"1.2.826.0.1.3680043.8.498.11767130706340200574705535810540787764","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53649895712172408476570770924813501118","series":"1.2.826.0.1.3680043.8.498.11884251985500852320645455360516659720","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.53745429232602455103594227782517963027","series":"1.2.826.0.1.3680043.8.498.11711966572541975014501800354878170276","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.53884715696186989798696544661396487883","series":"1.2.826.0.1.3680043.8.498.29171287647383874216000568275059018044","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.53901077733518036755308692304268815965","series":"1.2.826.0.1.3680043.8.498.46568712430483807784182191793620616552","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54000782656400326230944526201670909133","series":"1.2.826.0.1.3680043.8.498.12614062111741603573378169267164532128","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54099199514076260519579096546600505763","series":"1.2.826.0.1.3680043.8.498.28507500829688051679571979728800341705","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54101830029742852876183926998159899424","series":"1.2.826.0.1.3680043.8.498.19847390363475334877162361980839805011","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54142170846176833568961426171739782214","series":"1.2.826.0.1.3680043.8.498.67585263367104486810868711212169243060","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54202117421897784469771192940597638508","series":"1.2.826.0.1.3680043.8.498.15032128807527060399238510110268177919","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54306467780023777062920737767581126383","series":"1.2.826.0.1.3680043.8.498.33192477889995457167180846735443441537","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.54336962625483046366337626878609850219","series":"1.2.826.0.1.3680043.8.498.44019736021047619802589592640262664284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54427039222108917725902677899081931510","series":"1.2.826.0.1.3680043.8.498.45347918733888478630815778154599749749","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54451515695681765924346479727204451807","series":"1.2.826.0.1.3680043.8.498.24897746737059929082176799139016508696","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.54483132338261866032495309977418409299","series":"1.2.826.0.1.3680043.8.498.27564047367701790860685862281846953358","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.54619696938940618755334224550745735815","series":"1.2.826.0.1.3680043.8.498.53972565762648366841815942201878787872","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54669065204808162914386927559244866557","series":"1.2.826.0.1.3680043.8.498.78121154385682630627587753817653731996","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54693572065777563403567480644050627025","series":"1.2.826.0.1.3680043.8.498.12335728891098575620494764680946902374","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.54755249425443469939742671497042113665","series":"1.2.826.0.1.3680043.8.498.70085851146317341897663866719726083043","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.54792838438079861454532183998999249213","series":"1.2.826.0.1.3680043.8.498.18782906573642261919108446095183120743","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.54977581323931277817074741182670080450","series":"1.2.826.0.1.3680043.8.498.18709751104038684569603044904800357655","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55015061060346597564205577461227720715","series":"1.2.826.0.1.3680043.8.498.11523411602471130069641631042839539054","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55061039227197546099231336400032878562","series":"1.2.826.0.1.3680043.8.498.85949270804303246471522267332321650533","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.55105303802677863700162285680213496441","series":"1.2.826.0.1.3680043.8.498.80243485926890145564736482806862433135","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55300053252365037422745072128951402593","series":"1.2.826.0.1.3680043.8.498.94104074199313649352934727187286619182","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55344075316305815718253234447285669886","series":"1.2.826.0.1.3680043.8.498.81165323854147464224318137637098448480","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55397602116058381866334455456729470963","series":"1.2.826.0.1.3680043.8.498.19809878540082404334923329483481758773","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55461938012492393262172182405932212246","series":"1.2.826.0.1.3680043.8.498.75899094307141050371712777754540040922","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55653770127824574123236841437998533498","series":"1.2.826.0.1.3680043.8.498.56177818356944504309832235073594394706","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55700532609077428304194916132759352518","series":"1.2.826.0.1.3680043.8.498.23254301068835435020204704426631451429","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.55729193057949125029301812320609764642","series":"1.2.826.0.1.3680043.8.498.11387722782883389857536141988199271723","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55808772152667135571939417929724526296","series":"1.2.826.0.1.3680043.8.498.15097708898491875531455792658165347995","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.55861544093014499478739864837675161978","series":"1.2.826.0.1.3680043.8.498.89205394706693323466924721437265758801","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.55927741980493661714408715886641034285","series":"1.2.826.0.1.3680043.8.498.11988739023333904965614910854383285076","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.55971241672517044174019654998425819738","series":"1.2.826.0.1.3680043.8.498.56313424721249981818975035183533543452","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56016606949603829995818303105991224499","series":"1.2.826.0.1.3680043.8.498.91401653494109524770138381443624717621","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56061505201159269536816810086342635391","series":"1.2.826.0.1.3680043.8.498.52070509099712172640265830827732559871","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56146582420561901778012523946835033628","series":"1.2.826.0.1.3680043.8.498.56164047941567167632189496685160143941","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56170570792418244380186761574367557535","series":"1.2.826.0.1.3680043.8.498.48776634137764181577807734452397176318","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56266280744044680626884459696038109616","series":"1.2.826.0.1.3680043.8.498.12360992249677515867211926441948004451","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56305445410093481072159375249067050124","series":"1.2.826.0.1.3680043.8.498.59884829160900982229461198989581694505","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56474364950531483257947155855279443851","series":"1.2.826.0.1.3680043.8.498.81696645547900067886508437537966094364","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56512564464301544960446419143135447363","series":"1.2.826.0.1.3680043.8.498.54535610302647039407009910528489667982","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.56564801707939049324472369774251260607","series":"1.2.826.0.1.3680043.8.498.81568141098523388354759710648193451510","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56686055146646210144964207684044815006","series":"1.2.826.0.1.3680043.8.498.12991507820025940827753268535291177057","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56751453017492477106384704540803963879","series":"1.2.826.0.1.3680043.8.498.50479554017979142568333956632004212317","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56796622077427019119540733986070534126","series":"1.2.826.0.1.3680043.8.498.45333709802575893015101313682033290846","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.56814987912934301686833132930648230999","series":"1.2.826.0.1.3680043.8.498.10787000763715901565663027618994946084","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.56882015310689536014061315110969323429","series":"1.2.826.0.1.3680043.8.498.82016149025170644260862257900365289110","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.56894990686683651282564584582335346350","series":"1.2.826.0.1.3680043.8.498.15354535590187023121610734146623792251","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.56939860329243258146102948466697018543","series":"1.2.826.0.1.3680043.8.498.73109770830119434650749028595624835233","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.56991590802231436015843619633497607680","series":"1.2.826.0.1.3680043.8.498.13575446425920610234253607374484698737","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57064618560116981041722203028857007300","series":"1.2.826.0.1.3680043.8.498.10027550161761847752083513853619165044","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57084445429567494093960261980941917499","series":"1.2.826.0.1.3680043.8.498.43197611474638733785610294735044370365","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57160448024145680660335079050450776852","series":"1.2.826.0.1.3680043.8.498.77503027043634465306783657393020568051","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57342523828166309133982222435655662667","series":"1.2.826.0.1.3680043.8.498.15194419281425431119988671103994644545","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57406640473235989459885313797729459197","series":"1.2.826.0.1.3680043.8.498.12874731195255638975565137959161159053","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57615268093937994500995086275007694564","series":"1.2.826.0.1.3680043.8.498.11828351694584991443527035297325741947","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.57768539239475064336346600805236336044","series":"1.2.826.0.1.3680043.8.498.95060502356655803972080421786454211739","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.57802887045514681373294932749517914959","series":"1.2.826.0.1.3680043.8.498.83020877596620707244675676440867057373","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57836524675365700674244122764523161699","series":"1.2.826.0.1.3680043.8.498.84720804447819087035972818501997688827","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.57859543314667538612269642333387175763","series":"1.2.826.0.1.3680043.8.498.12568965992658599728424445923543928764","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.57917634151851433037090858498504057826","series":"1.2.826.0.1.3680043.8.498.12233864073590414338640642155775644374","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.57960501042895683580303516749157258348","series":"1.2.826.0.1.3680043.8.498.68842580432922220973928646025565997716","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58167256197067081436345136244032069163","series":"1.2.826.0.1.3680043.8.498.12210278837967273768533338462429991075","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58280111882249056715877316026191844548","series":"1.2.826.0.1.3680043.8.498.13212795583250747008574471361997146452","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58298767414933858487578597918936525600","series":"1.2.826.0.1.3680043.8.498.31968260050437482134476002171047002697","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.58381033890285289577066078050126251787","series":"1.2.826.0.1.3680043.8.498.12201688516333773942293808937663018101","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.58416798660375193103814365795347286292","series":"1.2.826.0.1.3680043.8.498.48838688731348020369341886146521225637","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58482779112389764165599701941310919660","series":"1.2.826.0.1.3680043.8.498.12760999985468373749389061751000561392","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58550899000434548218916895907938175917","series":"1.2.826.0.1.3680043.8.498.11523148178674144351678846426443849316","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58626635384305289939700801774393536415","series":"1.2.826.0.1.3680043.8.498.93670903073211409095451056562937206986","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58654915518882480245974544469656013058","series":"1.2.826.0.1.3680043.8.498.53852189443443770500219789964623930488","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.58728013757221328590898166513995249334","series":"1.2.826.0.1.3680043.8.498.74845567194813734735255071188169614658","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.58749835013162544157291112101941849646","series":"1.2.826.0.1.3680043.8.498.27364352362269583489383781702544250304","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58776844157027040145262759105410792687","series":"1.2.826.0.1.3680043.8.498.53002782204747239861178191234867874084","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.58835863392949696556026907211309159182","series":"1.2.826.0.1.3680043.8.498.12390454405855478493681843521957033312","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.58888406752590566453001353089284177456","series":"1.2.826.0.1.3680043.8.498.15479079348221458031709585524202508142","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59021913092219789232890327780064145948","series":"1.2.826.0.1.3680043.8.498.31634952452994107242279787229181735687","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.59048646289883992375610835845636263714","series":"1.2.826.0.1.3680043.8.498.10368886690075284753774662893581756204","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59098825208018364061073394433706021769","series":"1.2.826.0.1.3680043.8.498.10332524335240845250267029318957223075","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59254379036462832374262221063106167294","series":"1.2.826.0.1.3680043.8.498.22344028675730518820569867493064717145","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59394856717216352192037635065925607933","series":"1.2.826.0.1.3680043.8.498.89296999236924982817813320239458788032","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59426134288436219215525445839849588348","series":"1.2.826.0.1.3680043.8.498.12128341906874561271843580355215208028","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59476657510212258499047356518058209126","series":"1.2.826.0.1.3680043.8.498.86730818361085947905531168747104910784","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59511171909586924989766685687032462369","series":"1.2.826.0.1.3680043.8.498.63668821436547068997041603451748634417","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59646951619367295963072805933618947027","series":"1.2.826.0.1.3680043.8.498.12809393168763135247041261978871465311","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59708410902079108538978687294461753235","series":"1.2.826.0.1.3680043.8.498.31370101161880814134258684005828582052","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.59828592894392286103154555572376381359","series":"1.2.826.0.1.3680043.8.498.79973958824256016563518309956661810022","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.59923507410103964137819195926587517759","series":"1.2.826.0.1.3680043.8.498.90902376669601227716509687723943289922","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59938475193950583662125713924156951751","series":"1.2.826.0.1.3680043.8.498.37657708352977584981250908327118565884","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.59948559729295284519115137429127740378","series":"1.2.826.0.1.3680043.8.498.10782428858828111243807774268737552720","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.59973351395253648445741322652773810641","series":"1.2.826.0.1.3680043.8.498.12434568205361203672125848894553366771","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60094182309575937331152278560900767484","series":"1.2.826.0.1.3680043.8.498.64138188572117552381220055500414593115","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60189166269847256202786309096136242746","series":"1.2.826.0.1.3680043.8.498.12117055260331355728324225349126553579","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60240929196665639608259748424475337443","series":"1.2.826.0.1.3680043.8.498.24447589533454430351399731611224717337","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60347768295306794807599672836772410567","series":"1.2.826.0.1.3680043.8.498.12141026456314870850742889332602490201","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60435128564448668334758790827765960420","series":"1.2.826.0.1.3680043.8.498.14194501726680239105974840596428876802","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60446085094710623303973373214794179184","series":"1.2.826.0.1.3680043.8.498.94061404171136332435744449765116342765","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60473167624534682552272469123806503959","series":"1.2.826.0.1.3680043.8.498.70362335324085859456988139860894374180","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.60548477793215320533051194155539307724","series":"1.2.826.0.1.3680043.8.498.85268369612037457533071276850697672359","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.60611146141493700985009985723275645474","series":"1.2.826.0.1.3680043.8.498.13045791706696490962303973282166211923","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.60661676782543347731000874708337132276","series":"1.2.826.0.1.3680043.8.498.10868306201684464768224583319537188448","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.60736996513214411991331917490659995815","series":"1.2.826.0.1.3680043.8.498.38902323997851936023867728296016535263","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.60931716414402855657852057585917605786","series":"1.2.826.0.1.3680043.8.498.77404254951982004717236969759286106388","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61024356179284390979230045743542661223","series":"1.2.826.0.1.3680043.8.498.10276684283843493982400092083485044928","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61072566792285434404885416160728089641","series":"1.2.826.0.1.3680043.8.498.74643652070224783921528730997955796014","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61148667330113929729568621580194520169","series":"1.2.826.0.1.3680043.8.498.35851842573349663447203603557269436405","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.61227644940421894951950605627016308160","series":"1.2.826.0.1.3680043.8.498.11413926934502222363146002515473510504","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61313243050348988917987796330691448770","series":"1.2.826.0.1.3680043.8.498.10520537005359695495108149167521414529","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61354742313056287432707914716598361679","series":"1.2.826.0.1.3680043.8.498.81091982680737283075534875473435688837","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61490475141024845836646569993549106628","series":"1.2.826.0.1.3680043.8.498.17316353715951222379344022274244523030","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61529451559250327125552112075119161933","series":"1.2.826.0.1.3680043.8.498.78910709655211327077580133923364908881","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.61570904948128887877542043572765432417","series":"1.2.826.0.1.3680043.8.498.54432169772703132023586014577207636291","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.61715630583919056026446344483871058799","series":"1.2.826.0.1.3680043.8.498.24676576270221403703185485898235327886","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.61812343111344551704536863711676179812","series":"1.2.826.0.1.3680043.8.498.11796023675360144113342156315288266068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.61934170443339188744573234190617483017","series":"1.2.826.0.1.3680043.8.498.11236542147925666027007474913722762659","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.62025036385410820041698409408092924893","series":"1.2.826.0.1.3680043.8.498.63836287303878352330939607862103644897","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62109244272539959226824769201542205950","series":"1.2.826.0.1.3680043.8.498.22286969299489251893553259451400397598","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62192347880785177732202565414591408086","series":"1.2.826.0.1.3680043.8.498.26037824433265788283196386462146627537","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62218684089117864632544493263539598403","series":"1.2.826.0.1.3680043.8.498.12761913587495039535635351715420788202","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62252176347004905787746821113523889523","series":"1.2.826.0.1.3680043.8.498.75300813332192585813281284712709701501","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62400371770962739002393364782172386752","series":"1.2.826.0.1.3680043.8.498.19024660156494780301744134867967447603","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.62420632149863295140175740643640707423","series":"1.2.826.0.1.3680043.8.498.83581575879688919896619377565465952410","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62465595376489211274225312453216559395","series":"1.2.826.0.1.3680043.8.498.12524480704368639096451209748649813465","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62662430565901539554882496013173225758","series":"1.2.826.0.1.3680043.8.498.13661107449936337438489450319591037762","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.62760659205261492447481830774781950875","series":"1.2.826.0.1.3680043.8.498.32069314590728962271901022995013642196","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62830031603985608774418534497520663269","series":"1.2.826.0.1.3680043.8.498.66514411011317862938564584559032993283","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.62924835279557724526138406447993672103","series":"1.2.826.0.1.3680043.8.498.84096420561491254632758318338737631286","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.62983291359021407345388012674614537807","series":"1.2.826.0.1.3680043.8.498.97175925972841280314826847060370827738","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63005973438879239920248535292375481294","series":"1.2.826.0.1.3680043.8.498.46450373842717601373667697640493238616","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63096952391833769395204944538276970114","series":"1.2.826.0.1.3680043.8.498.29627775836457060260288432036469712792","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63126604639589769235241216875934773992","series":"1.2.826.0.1.3680043.8.498.75539617957305984707289273934115894074","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63197503121546251937003511751049515360","series":"1.2.826.0.1.3680043.8.498.71245405041817681173263428838695882232","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63216944317982327305447295456609309952","series":"1.2.826.0.1.3680043.8.498.12810246090502776344601559593207951054","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.63229263722659785142967850542699676113","series":"1.2.826.0.1.3680043.8.498.10666011271293246197060517870905822367","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63280826430574305683941548157934410641","series":"1.2.826.0.1.3680043.8.498.12064050409328776689948129420190691890","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63360204391136938955220146755331421179","series":"1.2.826.0.1.3680043.8.498.12387475574003558924641201281893895517","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63426442500798189991842988966227275166","series":"1.2.826.0.1.3680043.8.498.83430025915541779348841529566201188740","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63510378689078892579575716369639643386","series":"1.2.826.0.1.3680043.8.498.85261893366682958419263339836362342714","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63521726454879203860288713455843702564","series":"1.2.826.0.1.3680043.8.498.10782858802416114383721258129227775553","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63550258873504696667112631869753870779","series":"1.2.826.0.1.3680043.8.498.36584196081453249044886720069477558368","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63564058504808107209753434989270686941","series":"1.2.826.0.1.3680043.8.498.29827150700192983635439011736163798811","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.63673733836218886397484265807806548473","series":"1.2.826.0.1.3680043.8.498.39492816344669414436884690105341904486","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.63726512375511246975989322174096818969","series":"1.2.826.0.1.3680043.8.498.24830026619245989120192410378102606807","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.63884838982826315257233262184190435906","series":"1.2.826.0.1.3680043.8.498.84501813668670275619103370265784798286","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.63986652686300452077410299429386420400","series":"1.2.826.0.1.3680043.8.498.13214029342533938517652352258439262599","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64029768638431636860053484202080135316","series":"1.2.826.0.1.3680043.8.498.41843332463858550290023830687132498365","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64066465191710322298540005681534170036","series":"1.2.826.0.1.3680043.8.498.50355091183758585249254987579720349131","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64172939203926034064618447795131041647","series":"1.2.826.0.1.3680043.8.498.10683482485489089443534047429757920660","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64240637244650029205175793729578897734","series":"1.2.826.0.1.3680043.8.498.24081141244360290417573951901596605956","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64325734401816269429960464734504322970","series":"1.2.826.0.1.3680043.8.498.20079746419011240283924552945469032593","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.64408609256163127278435484217683272910","series":"1.2.826.0.1.3680043.8.498.91711982512971905574842011692644557600","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.64421299492271548595446260681698412384","series":"1.2.826.0.1.3680043.8.498.34595844298237084813921600771136361654","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64487230511522940587342058274736571996","series":"1.2.826.0.1.3680043.8.498.10667791811983677043451028153180018691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64590392444259942263746282444188079724","series":"1.2.826.0.1.3680043.8.498.47679597212492838492056054596602450834","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64640435232894033317418319985727526223","series":"1.2.826.0.1.3680043.8.498.78066312822217701362909244324030093444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.64673069370410118387363952155758553567","series":"1.2.826.0.1.3680043.8.498.54652524118901655454482618533331027399","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.64865300853258390816475799272221318005","series":"1.2.826.0.1.3680043.8.498.12684291626142532307449591885057482523","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.64946212153370539647301308676689632525","series":"1.2.826.0.1.3680043.8.498.91912495223835633211836707678454833382","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65013994414976103275014874284064145461","series":"1.2.826.0.1.3680043.8.498.60157805138449432723373313847701816770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65123055741109319778634264103224173252","series":"1.2.826.0.1.3680043.8.498.11766133166999623154892163885992585884","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65160241567146620406193593170843451582","series":"1.2.826.0.1.3680043.8.498.45397582526841338041527384119357112069","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65222333228514862688331581862087607325","series":"1.2.826.0.1.3680043.8.498.65467765380421830275229442394439918473","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65312506382127486486519184894274326380","series":"1.2.826.0.1.3680043.8.498.84575412086853971739258213886849399474","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65356758967883114612797250487949537460","series":"1.2.826.0.1.3680043.8.498.11997728472089884988641551811079158462","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.65515828911930515002046151253057175178","series":"1.2.826.0.1.3680043.8.498.53255323769229916695493568812601016434","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65530757406333318449497070048719890442","series":"1.2.826.0.1.3680043.8.498.35624995629328099764752648243058770993","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65597375829884314152221851871550349856","series":"1.2.826.0.1.3680043.8.498.12963082839280894549058893566497713172","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.65703265163052901613211590083675186626","series":"1.2.826.0.1.3680043.8.498.10448645164577874634798862106302997820","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.65715827585629489888518528736848969678","series":"1.2.826.0.1.3680043.8.498.11689288357535251359518942626947329517","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65783500506568702195262661666345236666","series":"1.2.826.0.1.3680043.8.498.56182270886759055179417052106351102601","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65792625478083832901309715174678358900","series":"1.2.826.0.1.3680043.8.498.12073442807889744612403337339388879679","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.65855247249608663104822603172047250057","series":"1.2.826.0.1.3680043.8.498.35948238332865869484294532812156815102","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.65909865035783831607773206653044421345","series":"1.2.826.0.1.3680043.8.498.55215746734310754677097851619815704801","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.65977711270942843864266358352913175672","series":"1.2.826.0.1.3680043.8.498.17484399798935815001825677460539937992","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66022464665791424345622894674375661089","series":"1.2.826.0.1.3680043.8.498.11109301460315477500153665675818486327","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66077420344931334696112616504179985591","series":"1.2.826.0.1.3680043.8.498.10898475292334704513499032212604084889","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66156015559325316834458423557757995415","series":"1.2.826.0.1.3680043.8.498.88591760755279570285392804224720257459","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66264392187277583747691643146920431469","series":"1.2.826.0.1.3680043.8.498.70674890081194944083137997241870328315","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.66296018797614563698471347079960740755","series":"1.2.826.0.1.3680043.8.498.13286999593213201771519362983187649200","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.66321903250295261063309947910189882952","series":"1.2.826.0.1.3680043.8.498.12678771913225391981433670996393544627","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66364462115470823900176770413527217503","series":"1.2.826.0.1.3680043.8.498.10484368513108006701893871553854780156","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66404725797392899950170138361835649917","series":"1.2.826.0.1.3680043.8.498.10369333844923019931553437649075689635","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.66596090141798159058634289358454183351","series":"1.2.826.0.1.3680043.8.498.78867223890632840671834928010673379233","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66690100241274246406145618641481787709","series":"1.2.826.0.1.3680043.8.498.13162175282834269775211667628036437708","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66805139646641294853509958688742554525","series":"1.2.826.0.1.3680043.8.498.10926714136237340709313949038883629201","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.66865810098959732285731429066569782524","series":"1.2.826.0.1.3680043.8.498.45595811708185099634714180782928201855","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.66993076561387452114335652021593805727","series":"1.2.826.0.1.3680043.8.498.22755583002250018048122062540757105081","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67153684605171746899496816770567866269","series":"1.2.826.0.1.3680043.8.498.11170540158517120100023339094532389979","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67181088492292316990397177612918195713","series":"1.2.826.0.1.3680043.8.498.96655386006846869692900577526010356100","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.67208608901061914528631080268819069203","series":"1.2.826.0.1.3680043.8.498.10559126290873356564736473472849074929","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67284681623851829916403311739987186159","series":"1.2.826.0.1.3680043.8.498.13368922949780594883519694460854850091","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67386780309737226410965627751074124678","series":"1.2.826.0.1.3680043.8.498.11498839130854253696118228993475992108","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67456402757136973484427155324664966595","series":"1.2.826.0.1.3680043.8.498.11522133315627456261555621349126009788","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67490059263503953166467514121525025332","series":"1.2.826.0.1.3680043.8.498.74805578492858784943254643853120784468","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67550262192397377344258276170992188234","series":"1.2.826.0.1.3680043.8.498.81657554750115440145181647937474224616","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.67613388743126519966672593774754849330","series":"1.2.826.0.1.3680043.8.498.41485319245058030787095252395270550935","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67678576373934022409084801948795250361","series":"1.2.826.0.1.3680043.8.498.58954509536063737176176138061097173634","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67716063997837501476742746033172868748","series":"1.2.826.0.1.3680043.8.498.79656617198796939072277724965354330737","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67740000270419228186274186410889719154","series":"1.2.826.0.1.3680043.8.498.22493537213815330481964488236655387226","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67788047649994083591931245362169694306","series":"1.2.826.0.1.3680043.8.498.10911612479248669963322480413788759972","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67860057199678546706175336855949799228","series":"1.2.826.0.1.3680043.8.498.36075337496988690467722162928005326868","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.67902658411525109462089381503197501728","series":"1.2.826.0.1.3680043.8.498.10433021766432146182333975915714462589","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.67941678374744711267840655419606994796","series":"1.2.826.0.1.3680043.8.498.12764591103990796874522161974340167067","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.67981539513814867568631217960473023750","series":"1.2.826.0.1.3680043.8.498.11704092065978442462179709550282473720","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68079818232244983808025467536454890669","series":"1.2.826.0.1.3680043.8.498.11184204927412649241553756438879226789","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68106879690636836695681159513106553666","series":"1.2.826.0.1.3680043.8.498.89052118222029249616470832432568463561","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68142054543837964766571710567214307830","series":"1.2.826.0.1.3680043.8.498.20157415730448379251931616002292078527","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68166103964518457268906879600985959436","series":"1.2.826.0.1.3680043.8.498.59774876009442056298405492318918226390","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68219942161081583831276769978472014317","series":"1.2.826.0.1.3680043.8.498.12132970385324729980191759600759816530","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.68256141451764547710152696919981893952","series":"1.2.826.0.1.3680043.8.498.94922822833015308733576586216186637058","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68264559730756309944466784364591509088","series":"1.2.826.0.1.3680043.8.498.13700537862993732876991435540356166382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68328948893840135511720724977892905019","series":"1.2.826.0.1.3680043.8.498.50937581331383882497539858452906168005","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68435927323644457286635872824563629385","series":"1.2.826.0.1.3680043.8.498.92076705752581529015330844387337859793","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68448811873610037553554213141537599150","series":"1.2.826.0.1.3680043.8.498.11561785794841837629390838659079485786","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68570907895010847368926943092385685462","series":"1.2.826.0.1.3680043.8.498.10666598395776006571024654219277697679","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68578659816221770558506780445608768600","series":"1.2.826.0.1.3680043.8.498.13665333981517191352574485837569319608","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.68637419175382604890136839259344633105","series":"1.2.826.0.1.3680043.8.498.97709482621437603992008839785909270240","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68740466085414970518655922465908223436","series":"1.2.826.0.1.3680043.8.498.33518812944269434546229569906509899739","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.68777783869253673804970857981124324335","series":"1.2.826.0.1.3680043.8.498.73367355695403426698194381359821148131","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.68806105227826942108149090508198910591","series":"1.2.826.0.1.3680043.8.498.66698686335810612167034405396226803158","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.68919687595587004665658610968739852899","series":"1.2.826.0.1.3680043.8.498.32190567950349623837066950409634787793","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.68971603226593154050514800137550171537","series":"1.2.826.0.1.3680043.8.498.67797271774642704562401290964577702224","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69033651423155214548081618535669641812","series":"1.2.826.0.1.3680043.8.498.83708632885169417334157793180609088181","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69141410772327587821159027820329185006","series":"1.2.826.0.1.3680043.8.498.37555443314201256537167796346787035613","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.69212713525853754921729012449897924087","series":"1.2.826.0.1.3680043.8.498.22649963636314885855841533749973907352","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.69267821440055811776827497076481587198","series":"1.2.826.0.1.3680043.8.498.10815303209134196475452702363656908028","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69397705155385306818511499752200941968","series":"1.2.826.0.1.3680043.8.498.10186258182471641456859330423214982603","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69417319234704516520173921900954337376","series":"1.2.826.0.1.3680043.8.498.85446451364199500058115925804799472258","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.69495014653333523580255444915082056825","series":"1.2.826.0.1.3680043.8.498.57731545071681601512295232223451053036","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.69596668032633976121623939445795750384","series":"1.2.826.0.1.3680043.8.498.36760713216564182212953922660037951952","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69635429172929809539574727887787303282","series":"1.2.826.0.1.3680043.8.498.26714728473558346179119984270345002086","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69673283369917182409597968479300552579","series":"1.2.826.0.1.3680043.8.498.26354114305010111679386328546793761484","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.69730556680449158538173708009262014812","series":"1.2.826.0.1.3680043.8.498.12166718627402067156950182836861868331","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.69828573582708185393942443598783108800","series":"1.2.826.0.1.3680043.8.498.10525010187132024042265028453161460884","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.69930569170269472474738549886709596023","series":"1.2.826.0.1.3680043.8.498.13024150354263398094405427285867672939","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063","series":"1.2.826.0.1.3680043.8.498.95325613602818305445043750946635992403","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70204818275951821519024007466988533363","series":"1.2.826.0.1.3680043.8.498.62891769263160884385901394412120856466","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70291739279544255146458190540233023337","series":"1.2.826.0.1.3680043.8.498.56528159242014061154539351958538930207","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70310491987092072540988486213408328865","series":"1.2.826.0.1.3680043.8.498.12714386553342957069274785364046430266","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70409266114961050236894235887802126808","series":"1.2.826.0.1.3680043.8.498.16911922612510559795406899244736888330","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70464717849928808075676221329034929056","series":"1.2.826.0.1.3680043.8.498.35397962767713394583344716641672115386","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.70478068758081824347762712452252763894","series":"1.2.826.0.1.3680043.8.498.52814861833744781228727602604275623052","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70532653867267627708152426358320452518","series":"1.2.826.0.1.3680043.8.498.12982569745481481497755795457518926712","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.70674155597869824035214737999044110827","series":"1.2.826.0.1.3680043.8.498.37328140268804110345691310289264874270","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.70796360440320543674384511859375295769","series":"1.2.826.0.1.3680043.8.498.10885691273522734551803621048616895860","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70865607713855272648070757909359669526","series":"1.2.826.0.1.3680043.8.498.26253837500845554526218987213986326985","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.70877157427595288430686609728186100169","series":"1.2.826.0.1.3680043.8.498.95723837983148129104113439059801577662","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70909154434106251910698870110315237266","series":"1.2.826.0.1.3680043.8.498.32982335365115421486618609927463742190","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70931245733460964524244528230239226104","series":"1.2.826.0.1.3680043.8.498.11219633757056708186545223498604920934","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.70948162965271404678418299669321564765","series":"1.2.826.0.1.3680043.8.498.48877756795986091604050800666699136570","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.70984418257970716669463829979819981861","series":"1.2.826.0.1.3680043.8.498.58492524718390324421950581070647500283","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71047916312801688293181464016099792199","series":"1.2.826.0.1.3680043.8.498.10728585248599406152561805742662711101","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71157888664602228768823145752952837565","series":"1.2.826.0.1.3680043.8.498.12637032086566882463752473570079927769","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71311031275189592818091380416268522846","series":"1.2.826.0.1.3680043.8.498.16737189948485540924654207237622600481","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71388532549899713881913113571627425347","series":"1.2.826.0.1.3680043.8.498.10742729094365133590379862872192443105","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.71507059468810869004142324822642900237","series":"1.2.826.0.1.3680043.8.498.99753258445415238538545015499031962120","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71636001534145439773297498405674521629","series":"1.2.826.0.1.3680043.8.498.10564260073333319074527028308778863600","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71686717249431292334159336816742076478","series":"1.2.826.0.1.3680043.8.498.73291801040048033423780528212410430956","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71701484984624141610445774249629295610","series":"1.2.826.0.1.3680043.8.498.70770025630371983252935997483325257865","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71736046309441396676679830874326797437","series":"1.2.826.0.1.3680043.8.498.10726370439994129741752280622403264865","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.71749507378205349451480318735471291366","series":"1.2.826.0.1.3680043.8.498.10167003155712742297910157198632228210","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71792468450491064417953341293663396190","series":"1.2.826.0.1.3680043.8.498.27854700018866795365462550698145170069","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.71830371861099586573789550797712053914","series":"1.2.826.0.1.3680043.8.498.36183365423584657140189545424491052445","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.71866327348775269043047475658220220525","series":"1.2.826.0.1.3680043.8.498.10860398989733582906550424233649729621","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.71901039762584748047172479049347609017","series":"1.2.826.0.1.3680043.8.498.70143849195063434288598176752677265444","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72017096768214637927798648334301077145","series":"1.2.826.0.1.3680043.8.498.52187124996077575634489730247223310972","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.72023797276837219152365428999317131110","series":"1.2.826.0.1.3680043.8.498.95625193648692870245605268762927823788","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.72159028826938094045309841541333724597","series":"1.2.826.0.1.3680043.8.498.80919285853950399511510156487005803922","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72185504245484151161044972024469667666","series":"1.2.826.0.1.3680043.8.498.24365925149353507018348555922418217125","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72248728319398595146365363062799556637","series":"1.2.826.0.1.3680043.8.498.10204287106088932285079510282573548756","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72255181548595299889516400134053439401","series":"1.2.826.0.1.3680043.8.498.10184644340942106565704796238140380016","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72313325634455670216423864504103231505","series":"1.2.826.0.1.3680043.8.498.41170900638270758773245275951458974339","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72368645606329000923061796699431437924","series":"1.2.826.0.1.3680043.8.498.35518191119028070229157450432123801649","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72395496925770440630088720587484704971","series":"1.2.826.0.1.3680043.8.498.11512474179977479839876478366099586152","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72495590491484814220060275823761767516","series":"1.2.826.0.1.3680043.8.498.80598039880021619635036189481505239342","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72547312490317214326791189007411860203","series":"1.2.826.0.1.3680043.8.498.83023723216085644465087111163548593436","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72587440202779699193620339987034971895","series":"1.2.826.0.1.3680043.8.498.11843716960349643355959664065098902943","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.72697172162233919540021967137047646179","series":"1.2.826.0.1.3680043.8.498.12378593761169618943083904156459210569","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.72771360268592017354044657882090174639","series":"1.2.826.0.1.3680043.8.498.81780734593677663591519733158052699149","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72800659596099468282135738543971261904","series":"1.2.826.0.1.3680043.8.498.11409960868046396439705534192716177182","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72853333220043794904856138561095171921","series":"1.2.826.0.1.3680043.8.498.61693835729507935918922065312401699768","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.72896586188735703676995894830815493004","series":"1.2.826.0.1.3680043.8.498.41029034783748367593011996865977432761","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.72937134140407212536042212109839895294","series":"1.2.826.0.1.3680043.8.498.12177892585051149903621782715175136021","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73021428171736137311489171988512484948","series":"1.2.826.0.1.3680043.8.498.10943860173950134860822037773146936522","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73037642285055457162691486258363101398","series":"1.2.826.0.1.3680043.8.498.61240789712481348667560706505932582819","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73077380963620788952030569132380549826","series":"1.2.826.0.1.3680043.8.498.11145683616163255714407791539806330392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73170541731840260065335942048010387599","series":"1.2.826.0.1.3680043.8.498.65466809512383443627778616701540229589","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73198335328613992289471885012751338458","series":"1.2.826.0.1.3680043.8.498.19050437970459173803333887827699687691","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73231153161312542696053060380619432126","series":"1.2.826.0.1.3680043.8.498.12835084494447144551243500850563220126","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73254365064404586162477089636000427751","series":"1.2.826.0.1.3680043.8.498.68915521685937574007693869903242683593","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73275019425494043461771574462523171618","series":"1.2.826.0.1.3680043.8.498.17727288274109160667397541929657163893","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73356831368999268488355650751758108912","series":"1.2.826.0.1.3680043.8.498.83161572548798834712007366315118177880","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73448420644251136304786298254801920284","series":"1.2.826.0.1.3680043.8.498.87956131057523780470966625504635794561","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73480422982315825699847838826691577320","series":"1.2.826.0.1.3680043.8.498.43256026565467824817377629734154462014","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.73522298807198784022883320957252530465","series":"1.2.826.0.1.3680043.8.498.26978367953043749138353898958390668040","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73545582819102316606081606511277579853","series":"1.2.826.0.1.3680043.8.498.10592251764601754820240215725156883248","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.73616237718955147476241731466078978343","series":"1.2.826.0.1.3680043.8.498.10487607617705764916387674383403085856","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.73748007182569068263280098473557128226","series":"1.2.826.0.1.3680043.8.498.10858852715568687404473899342847735726","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73825252186234755453650759159564715478","series":"1.2.826.0.1.3680043.8.498.40362524195782934989627157037987649988","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.73926443729786165628848843707532839995","series":"1.2.826.0.1.3680043.8.498.87895862586271485169231472393293964108","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.73988798714023285209290281409329527637","series":"1.2.826.0.1.3680043.8.498.10266608361650587566383206469296338548","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74095402538454760179339842746572802563","series":"1.2.826.0.1.3680043.8.498.33037096289835757246027997133967902065","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74140806279409854828032496578072554031","series":"1.2.826.0.1.3680043.8.498.81327181733077177872090862681979729282","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74206939384612669977629625310277500302","series":"1.2.826.0.1.3680043.8.498.32034963976147075268452147252932328971","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74215611637606164186239511504256137871","series":"1.2.826.0.1.3680043.8.498.76396205057286497274546996442950494241","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74243110959141459268838217468465880820","series":"1.2.826.0.1.3680043.8.498.20728005876130865258019255406933299085","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74247325306307303036016759070403973520","series":"1.2.826.0.1.3680043.8.498.95139405956593855899241907784389244531","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.74282280145383357786409764912398881272","series":"1.2.826.0.1.3680043.8.498.19869322524713756912503810431289771026","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74388661917632804671669952747214084502","series":"1.2.826.0.1.3680043.8.498.48134747812339939481446197851553802803","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74423888576992387013857540016155410962","series":"1.2.826.0.1.3680043.8.498.10881018634597403115821286599850681970","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.74471724874857406578316075456414902954","series":"1.2.826.0.1.3680043.8.498.94720769982902124833463096257256761914","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74688733778107958348746767768064374883","series":"1.2.826.0.1.3680043.8.498.22752891302984859713664182948911048389","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.74725806565437665548056410415178280121","series":"1.2.826.0.1.3680043.8.498.88188153676527552635474218555109902853","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.74864145321761783409935288797327597738","series":"1.2.826.0.1.3680043.8.498.12244080355283735882272548297519137097","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.74914646210033032631540802996162814268","series":"1.2.826.0.1.3680043.8.498.35332489607649964864748245687991344746","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.74940598515597814600837126537274199312","series":"1.2.826.0.1.3680043.8.498.11758737073108145383728272159203227980","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.75126165133796914746114117844697263697","series":"1.2.826.0.1.3680043.8.498.54934390770968298929433463772689736614","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75187434248356774277526985329346125190","series":"1.2.826.0.1.3680043.8.498.43903640528718159973086353576559343273","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75276563505367629340309025030148269450","series":"1.2.826.0.1.3680043.8.498.24083787154979455025820530689092893083","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.75378323910171368568360074903297446681","series":"1.2.826.0.1.3680043.8.498.55582055687821275598906811447462062602","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75419034794059601820999512200601915669","series":"1.2.826.0.1.3680043.8.498.52986070720103401548470959566665982933","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75427432788337288702609576920198810880","series":"1.2.826.0.1.3680043.8.498.10028715832304659785576602700454067426","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75523541256169382504121567002441615798","series":"1.2.826.0.1.3680043.8.498.42901214404284532882230606778396450359","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.75548650589349966221875964542604002971","series":"1.2.826.0.1.3680043.8.498.12899560314912358682844796118281806577","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75627574440860942023321071382766359827","series":"1.2.826.0.1.3680043.8.498.10232401444026155850512297252798811968","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75670996863043899245632198549551296292","series":"1.2.826.0.1.3680043.8.498.98510871476681037510841501140476317102","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.75739561647115166735277767560308263446","series":"1.2.826.0.1.3680043.8.498.37190975950287841292825156225241046568","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75751784078102748563815298069799859542","series":"1.2.826.0.1.3680043.8.498.23979390851836866765961007179193688079","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75773447415943537938555952303103557864","series":"1.2.826.0.1.3680043.8.498.70181778953134739701515585227523559270","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.75924957745868212343758911856809118662","series":"1.2.826.0.1.3680043.8.498.81564267441878757308356478712041923167","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.75980374267002793040832157122311396174","series":"1.2.826.0.1.3680043.8.498.57879319085116417912092912168127855200","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76027799502777251766328820405966979386","series":"1.2.826.0.1.3680043.8.498.11202015328807723098513260517686118726","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76107330932071966465869847071480517541","series":"1.2.826.0.1.3680043.8.498.95064154548004026832759942861985740570","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76125027090596913549224250979076533606","series":"1.2.826.0.1.3680043.8.498.12108250920724353859750404875550155615","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76222770668660274805993655800271653712","series":"1.2.826.0.1.3680043.8.498.66023084568681088364029196292220666322","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76281930374588438789344170829280438697","series":"1.2.826.0.1.3680043.8.498.37972480232946231764103778080533922990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76514087677277144682713362609812138150","series":"1.2.826.0.1.3680043.8.498.17962711343821375913546748165827073462","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76538737802277241976971950690663079376","series":"1.2.826.0.1.3680043.8.498.10906476604700426144597746526067034380","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76620177268652814425535081607587091903","series":"1.2.826.0.1.3680043.8.498.12677594786485777127767844577428830693","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76662146598924492573212477881527658730","series":"1.2.826.0.1.3680043.8.498.76487655894053113886333242080214101651","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.76757251008849492985513280752011229119","series":"1.2.826.0.1.3680043.8.498.10801026754922567858845904461833230653","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.76841077743272729415666709641073096909","series":"1.2.826.0.1.3680043.8.498.10403612365254794042144544411383081399","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.76909970028125318280995002129970165723","series":"1.2.826.0.1.3680043.8.498.51257053284398683569757216796967568408","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.76950154099427419631807259601983648254","series":"1.2.826.0.1.3680043.8.498.13202565015386789094497408996847699261","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77037121474756969001683126693104350509","series":"1.2.826.0.1.3680043.8.498.46660184148408988687257907529225588813","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77060858802024387619459645469871885412","series":"1.2.826.0.1.3680043.8.498.10523580870776422282794726259302759946","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77150650601457686196626648868552167695","series":"1.2.826.0.1.3680043.8.498.78287246621641846634551882259333874137","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77183368292104509335580564463404842046","series":"1.2.826.0.1.3680043.8.498.11616945421855831351616649973341814607","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77244685031928454051409193776790348219","series":"1.2.826.0.1.3680043.8.498.11103752587330343546225167802723728708","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77274115607096351160296319533007121044","series":"1.2.826.0.1.3680043.8.498.84425048733978792865277225689037278443","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77290477043075828139630734968573461136","series":"1.2.826.0.1.3680043.8.498.83051020515024939687777001705202452036","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77306755058630855443644908572157134062","series":"1.2.826.0.1.3680043.8.498.47639961151070940652874471671349010116","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77362718298550276679350855963451855003","series":"1.2.826.0.1.3680043.8.498.25371003339062235719535685903629725583","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77391801789875882109036004511558148658","series":"1.2.826.0.1.3680043.8.498.28110388985460193579097217689758358562","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77421709644087006134426085284321515809","series":"1.2.826.0.1.3680043.8.498.86453057694667192146668253065255517954","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77442647074650043391818472068835112806","series":"1.2.826.0.1.3680043.8.498.29974786594389732097421081566582449878","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77513348456432770689938407624768928320","series":"1.2.826.0.1.3680043.8.498.57554160190238895219481597255158412780","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77541648519018799027301684184453151471","series":"1.2.826.0.1.3680043.8.498.65360808151261369003254707155445419521","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77577135626047243581253857225991426727","series":"1.2.826.0.1.3680043.8.498.34977023039961846225581837052798645206","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77611073647079414772859496272387571059","series":"1.2.826.0.1.3680043.8.498.84280428078816998828590117822096538847","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.77716260980321491689913730053309892120","series":"1.2.826.0.1.3680043.8.498.10358449295672018979670059618254776038","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77740672313593763051900703940745812581","series":"1.2.826.0.1.3680043.8.498.10041731529774502577862769168815916727","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.77760453966349959552328150395676117417","series":"1.2.826.0.1.3680043.8.498.10422157181074909976688773730576999418","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.77801860555102174958520570751272642387","series":"1.2.826.0.1.3680043.8.498.27290762309526000832397237800409068068","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.77889800516853576877948593628420228178","series":"1.2.826.0.1.3680043.8.498.40977168552322094770597693503855064946","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.77911973519830915742655727368286737079","series":"1.2.826.0.1.3680043.8.498.86117594324451738003933367039676390718","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78003548255041391467292087330671945247","series":"1.2.826.0.1.3680043.8.498.29051352178410620123621364021789672366","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78079900750417018394321006978343945239","series":"1.2.826.0.1.3680043.8.498.73221684703446793358328974965134611618","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.78119223228067919465880137763593966456","series":"1.2.826.0.1.3680043.8.498.40846536205791407781910497097996869747","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78198993378521219839327914679211902952","series":"1.2.826.0.1.3680043.8.498.85677132788448259817112453278979959833","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.78238394512811614973689011297363399053","series":"1.2.826.0.1.3680043.8.498.12660240926710638961180435899137941669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78296478954955171036523274995555716268","series":"1.2.826.0.1.3680043.8.498.81698570811096824578030671073721419689","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78465765647012043478539489340002371960","series":"1.2.826.0.1.3680043.8.498.20573563031474143230586806601287352406","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.78512215519177850923279235346968676828","series":"1.2.826.0.1.3680043.8.498.11144137085787180057739852391084947631","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78574437345603052437892513165832297960","series":"1.2.826.0.1.3680043.8.498.44392439710497672985555097449499979620","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78667999747216717251638754803697959701","series":"1.2.826.0.1.3680043.8.498.10659660070461172987616427988490069441","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.78685661415919658264222803849222055515","series":"1.2.826.0.1.3680043.8.498.33037135996590913248676775710644128805","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78765525474890094481249498938657540052","series":"1.2.826.0.1.3680043.8.498.96774432369146343605363225083230594593","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.78812661021494224637586904573732040856","series":"1.2.826.0.1.3680043.8.498.56983092107420893678219194448973624963","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.78972123731732056524948076388825106061","series":"1.2.826.0.1.3680043.8.498.10424445550639318135309111768792245099","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79038153803632889185429988483097161887","series":"1.2.826.0.1.3680043.8.498.87930421056325896753428289713088695886","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79087227780630716132147615846840128474","series":"1.2.826.0.1.3680043.8.498.70373331572946750164243276677372038604","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79122189963582241618519333445055548347","series":"1.2.826.0.1.3680043.8.498.14031948387477950703613326391041950913","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79224785668205420138951103045759609475","series":"1.2.826.0.1.3680043.8.498.10453960821455576170662981061633486281","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79247871688929479411597859576284194165","series":"1.2.826.0.1.3680043.8.498.31744988164860240749541180363401633363","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79267465462008228906382361587050826784","series":"1.2.826.0.1.3680043.8.498.80367198626707445769761705097055513002","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79290157182784080975822485697879660942","series":"1.2.826.0.1.3680043.8.498.52109094824690547661426695460284054254","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79341955397030543255969676347554165142","series":"1.2.826.0.1.3680043.8.498.12885101420344600751502682664234710052","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.79524312100988116256514335064057385347","series":"1.2.826.0.1.3680043.8.498.10999714865482426068240546741520122947","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79565393323323007178523850093527250842","series":"1.2.826.0.1.3680043.8.498.11612897212831067471109732229042982413","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.79609204436595306110253502961172135807","series":"1.2.826.0.1.3680043.8.498.76957849803855300343813069148250841045","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.79672421957132993896448375311641998688","series":"1.2.826.0.1.3680043.8.498.10887729471632073392329353908268441231","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.79741629769872115526056483024289600492","series":"1.2.826.0.1.3680043.8.498.12465549488268277462985299132158869935","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.79977436705480715175480363175316375576","series":"1.2.826.0.1.3680043.8.498.27460216959967740503014117426104498135","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80058869639900582578987760750483296668","series":"1.2.826.0.1.3680043.8.498.31078316459984998768825099908239300571","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80161127293548940216292842707683784128","series":"1.2.826.0.1.3680043.8.498.11317430834699962887120550457799540488","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80192355832283305029770183988436644845","series":"1.2.826.0.1.3680043.8.498.95456956022040304187970281609802153912","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80247974926646352443043954025450139721","series":"1.2.826.0.1.3680043.8.498.85598321689332837285129687162668045187","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80290749690597272632725356393056746299","series":"1.2.826.0.1.3680043.8.498.22930037120877087401534640632387668521","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80313580693550912024103221258669390487","series":"1.2.826.0.1.3680043.8.498.11200652891324666855209351979161674518","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80325205657974094483016625702390834774","series":"1.2.826.0.1.3680043.8.498.96767735518297598123780275858181006284","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80458507774711142534182397228254444268","series":"1.2.826.0.1.3680043.8.498.43982252272912803659823616588741617295","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80514754572353458107286859773300865138","series":"1.2.826.0.1.3680043.8.498.12252223073458120488645012267226912423","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80589796918823112368758999678369292754","series":"1.2.826.0.1.3680043.8.498.28649695150057846235498135122827127480","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80627605463642274852766102874822833365","series":"1.2.826.0.1.3680043.8.498.11973487164651548165010122376831535185","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.80642348849062865125509773837278900750","series":"1.2.826.0.1.3680043.8.498.50294070369230764267516502770601902355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.80661392139668653552110941896475066204","series":"1.2.826.0.1.3680043.8.498.63870405510808349543037046563850418325","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.80726797831040227285328789648680519664","series":"1.2.826.0.1.3680043.8.498.54733046136595583244281483349324076038","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80741358365199721720335324482078866565","series":"1.2.826.0.1.3680043.8.498.10013480214842344762574044010614288147","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80861108320341696163987804408524915999","series":"1.2.826.0.1.3680043.8.498.49534627509008473669007542526483614726","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.80878057166752747059190481131529937715","series":"1.2.826.0.1.3680043.8.498.68551823142288746141057585903754635849","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.80931292574057303692838531323879513417","series":"1.2.826.0.1.3680043.8.498.29941011785493870310032648398185752569","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81024897463656728358703040759074583549","series":"1.2.826.0.1.3680043.8.498.86203853383025627357281881158108701703","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81103809943451790668501089584752866199","series":"1.2.826.0.1.3680043.8.498.16409887849051809166586861103025314777","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81142440700884513521007992957945568924","series":"1.2.826.0.1.3680043.8.498.89775687354256932162492749278475845790","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81269884425000228604280521306949721859","series":"1.2.826.0.1.3680043.8.498.17720807121493490206896861136705780212","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81342416992724702402763591633702802359","series":"1.2.826.0.1.3680043.8.498.12430133600375178237186178028991343669","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81407504276375197122586620633759174421","series":"1.2.826.0.1.3680043.8.498.10626527791581503992184867416968358330","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81464322239381998685473520085495957135","series":"1.2.826.0.1.3680043.8.498.54861646142150193732169466388665817717","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81495303185504163367736663647489556517","series":"1.2.826.0.1.3680043.8.498.11616768950425747846256334171841640747","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81590494282169434250974061642267472166","series":"1.2.826.0.1.3680043.8.498.39689930879565161061265006768122002789","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.81631925197682464058124413934958934086","series":"1.2.826.0.1.3680043.8.498.81309981752880922318249193087002636516","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81651455914141365028015556083080638757","series":"1.2.826.0.1.3680043.8.498.73505520713184654292992801942859148971","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.81698411330214788362326470979160431674","series":"1.2.826.0.1.3680043.8.498.33054077816776989943174336308426289656","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81790990482142045640192729399175106255","series":"1.2.826.0.1.3680043.8.498.33724215686517968017792085321842335018","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81796385087779065457668769443782458844","series":"1.2.826.0.1.3680043.8.498.62908586318660487955104770487688529382","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81822709772232914298414700148454511661","series":"1.2.826.0.1.3680043.8.498.11949212430043287464931014806547089522","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.81892104897856100357169774214718573167","series":"1.2.826.0.1.3680043.8.498.79736417116121052394545923557223536196","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.81964000393640770822199916465038759069","series":"1.2.826.0.1.3680043.8.498.16039262725975308089596298166276191624","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.81981151447274200345866874172324295172","series":"1.2.826.0.1.3680043.8.498.35826552493392860159507134200674833503","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82053853036886620816821481139413594407","series":"1.2.826.0.1.3680043.8.498.63916552647923282920392228430240696871","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82143276307898132314496735515204481508","series":"1.2.826.0.1.3680043.8.498.57706829395762811516866097104575555230","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82157293611630605740641187215639204974","series":"1.2.826.0.1.3680043.8.498.47472875284866725240563787840541871791","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.82272793832748838882752586678639564181","series":"1.2.826.0.1.3680043.8.498.38673165022526558350273772888583806326","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82414828460683289235183375457850096746","series":"1.2.826.0.1.3680043.8.498.57501815741742069746179287802321723865","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82481056562260046738834822905709207128","series":"1.2.826.0.1.3680043.8.498.81460034952984480265225563714545656609","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82523073559792523830827952754346276051","series":"1.2.826.0.1.3680043.8.498.76648307949296969230596102878131749279","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.82579118531566493365068788437906656162","series":"1.2.826.0.1.3680043.8.498.34082265790297123564700633659272938005","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.82591599611540849522609405219902904134","series":"1.2.826.0.1.3680043.8.498.28481982145121634273961361502657113285","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82744514729782879778828512295184306538","series":"1.2.826.0.1.3680043.8.498.63485890196180274044534410811472849009","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.82858981777405241601610241542173548956","series":"1.2.826.0.1.3680043.8.498.12423680567652362823782126199490586462","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.82925474533853765570570726554869886925","series":"1.2.826.0.1.3680043.8.498.58099720534409277687446832181600839840","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.83028939839767027342482365275441078777","series":"1.2.826.0.1.3680043.8.498.33456577589325611943550117107116862641","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83054899303871276184985881638619543362","series":"1.2.826.0.1.3680043.8.498.12821817520894037756385415779348290220","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83061811791652434833797599835671351097","series":"1.2.826.0.1.3680043.8.498.21923234497670458220478909305392464967","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83152040259754415849193046299744699239","series":"1.2.826.0.1.3680043.8.498.10850177224587057532804156154355054173","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83235374118671754689663732057314083332","series":"1.2.826.0.1.3680043.8.498.75672663977365791127295385136090814940","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83271280040777227673434567660749430744","series":"1.2.826.0.1.3680043.8.498.19281169706565207719687217529760529721","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83384018761005700404957421316476952190","series":"1.2.826.0.1.3680043.8.498.14388800218805720232713302598894917918","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.83495448827553002152296206085110883602","series":"1.2.826.0.1.3680043.8.498.44451508654259552189685656411529960784","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.83602205818477323486169938182117122488","series":"1.2.826.0.1.3680043.8.498.32446706170976443076771626235522332777","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.83740112248863750162633901687556705057","series":"1.2.826.0.1.3680043.8.498.71597717113563420289783265243643016460","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.83956129905762530953192730040209719910","series":"1.2.826.0.1.3680043.8.498.38908901163197678766289366842023282931","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84047134932018909998648460546980598926","series":"1.2.826.0.1.3680043.8.498.98567174765610681268808059870159914079","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84062602050757262792437893691591415560","series":"1.2.826.0.1.3680043.8.498.32443228471846694668100035252254850357","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.84069767112794890061709127671908414399","series":"1.2.826.0.1.3680043.8.498.11375931132496139520084866560721844218","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84230432137191939390680841389370206819","series":"1.2.826.0.1.3680043.8.498.12483690641119555080569923159138303263","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84347051904829814300575579670995018991","series":"1.2.826.0.1.3680043.8.498.95174219307006564478018920410429882836","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84477198486144701928775090913243250655","series":"1.2.826.0.1.3680043.8.498.84010797302991033339124737752307977285","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.84612593835203397696029057940345763682","series":"1.2.826.0.1.3680043.8.498.59185945194703088393224649036251166927","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.84646554968523210021448483220679710652","series":"1.2.826.0.1.3680043.8.498.12041597969553853082988349987560832529","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84720476187201404764508451536959273676","series":"1.2.826.0.1.3680043.8.498.98045012386732306359881469324702234489","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.84827027113422423894439539536275453132","series":"1.2.826.0.1.3680043.8.498.49997698396551864651473779091662232766","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.84896756080952941537556818257863912066","series":"1.2.826.0.1.3680043.8.498.11322177627481883791995637897565918588","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.85004373913191467860228868467924401547","series":"1.2.826.0.1.3680043.8.498.17085170045815658895380450583731144476","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85036616529391787087213173897728060798","series":"1.2.826.0.1.3680043.8.498.84763160784210691125836634783404929092","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85108719393967663738580518985514195538","series":"1.2.826.0.1.3680043.8.498.74765958751946999030560874322060570355","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85212353344613461454490252928274988058","series":"1.2.826.0.1.3680043.8.498.42907047542063299325358441793945488904","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85326542895076582985200613540648336047","series":"1.2.826.0.1.3680043.8.498.10117149485828956978712834545182567722","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85347263026430684291021533170619978114","series":"1.2.826.0.1.3680043.8.498.21852273366232440940105264107329998939","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85397267100597758920280635204297906286","series":"1.2.826.0.1.3680043.8.498.70350327198921407407933322037851873412","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85454884397223412543050876117416146750","series":"1.2.826.0.1.3680043.8.498.87359854586235587473411141218665735701","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85475892602877115812059264719695919613","series":"1.2.826.0.1.3680043.8.498.70240815814826128587684861621621303859","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85566900782101872452970957532250214590","series":"1.2.826.0.1.3680043.8.498.12632666097632580554084398324684093769","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85624441072689653093363046226467141762","series":"1.2.826.0.1.3680043.8.498.10810340710002541438296188446380414700","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.85663162480758065951465617999516770647","series":"1.2.826.0.1.3680043.8.498.69113430337175100641540955389812361371","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85715956158218184426400151797364834203","series":"1.2.826.0.1.3680043.8.498.33763664806317505299770453881273956419","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.85728170022176096748897349447671687164","series":"1.2.826.0.1.3680043.8.498.52531624137755952414937933304804251414","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85760360279564672425828346507309104394","series":"1.2.826.0.1.3680043.8.498.61150618359313203258636972954206733931","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.85870577082315172424837183881210532800","series":"1.2.826.0.1.3680043.8.498.22118441735689162564622073053597821342","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85883562680953255833567595509408888410","series":"1.2.826.0.1.3680043.8.498.13132565373342344599479562715050567633","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85932495800565018166769303205378781707","series":"1.2.826.0.1.3680043.8.498.53552553136672438956195281836524906995","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.85976668143927055109184433855426232055","series":"1.2.826.0.1.3680043.8.498.60644907926728242171395880052381173288","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.85992651772359515416050249463715355943","series":"1.2.826.0.1.3680043.8.498.38872809781274245899678645897781648990","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86042942187650835661744645315369706668","series":"1.2.826.0.1.3680043.8.498.83076022004911213337754882778090441097","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86117639434986594266345773277419881144","series":"1.2.826.0.1.3680043.8.498.51854270570724352113168363224176120554","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86216155858324761845740646421500394797","series":"1.2.826.0.1.3680043.8.498.10755889085777225712410050491466421875","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.86263264208309369957750915520115390972","series":"1.2.826.0.1.3680043.8.498.59292282820211755638802048535489379637","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86280033511814602594654628677479528508","series":"1.2.826.0.1.3680043.8.498.67438837495810426720049045552624395387","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86332035495941070455209594003499488889","series":"1.2.826.0.1.3680043.8.498.94349089521928222415026509305325786034","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86351141114604781108401133410934252655","series":"1.2.826.0.1.3680043.8.498.12354366406581143968052286357439756518","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86416984885007871128151024515780322767","series":"1.2.826.0.1.3680043.8.498.99756024259708973384651631882491726753","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86435271359723045834079326733478538148","series":"1.2.826.0.1.3680043.8.498.30330082697081014178501773803712004687","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86532643576908316499388520854820002133","series":"1.2.826.0.1.3680043.8.498.11168346724755334947380235186829342749","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.86583531066383395270546012510184348251","series":"1.2.826.0.1.3680043.8.498.12558747818328091090010975566888438170","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.86715997235054599024919893656486735623","series":"1.2.826.0.1.3680043.8.498.88709822590380596727908964705516825578","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.86876540394528304678398111135600894869","series":"1.2.826.0.1.3680043.8.498.97801593074967272818426686425783684019","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.86974853023888933977090539174537083409","series":"1.2.826.0.1.3680043.8.498.10465505655320846427029228349359777650","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87018440624359538250658207994111333051","series":"1.2.826.0.1.3680043.8.498.10061018232179518089734137452530499136","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87064372531225943854092598173945043861","series":"1.2.826.0.1.3680043.8.498.99001479494277190089259304685734715497","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87185817772180109135714642184874643715","series":"1.2.826.0.1.3680043.8.498.58048760348142840012238552190557667666","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87208714421237491643915928765740747633","series":"1.2.826.0.1.3680043.8.498.13284441792275819356078225966390755392","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87223930502803944549504594723314220105","series":"1.2.826.0.1.3680043.8.498.64415604709403043101203388403446817631","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87253761427012478584873622994909841751","series":"1.2.826.0.1.3680043.8.498.72329455762804127330858522517485249174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87338030934656808032044071206920470285","series":"1.2.826.0.1.3680043.8.498.11633090710349603725369588069694791182","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87415625824966124550466651648244553892","series":"1.2.826.0.1.3680043.8.498.54967722103867934380328562509635844023","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87476514600989622078259281213313867927","series":"1.2.826.0.1.3680043.8.498.11726084125554030356720246241581721718","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.87560916423885591029120178291827033868","series":"1.2.826.0.1.3680043.8.498.37669993143323475000811787128020815217","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.87645071186049036230936726742129446188","series":"1.2.826.0.1.3680043.8.498.12795116696220345977550331533914765393","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87679606849449562627719676100773778015","series":"1.2.826.0.1.3680043.8.498.12123357274662206610238365099419747853","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87761573479019057997334508266888503487","series":"1.2.826.0.1.3680043.8.498.54871986187284607170595413288989867860","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.87797202230328868466634480198668365648","series":"1.2.826.0.1.3680043.8.498.31347934847681914035852198738713758832","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.87845297884497964690686366000533570840","series":"1.2.826.0.1.3680043.8.498.24961159954945737593806138808575645966","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.87998832540756872371878613339628615880","series":"1.2.826.0.1.3680043.8.498.41498863343100561915647475262993369926","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88064086448060280325724227239585797761","series":"1.2.826.0.1.3680043.8.498.88198803157602501711969251735008580913","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88077418639301174409926781329613570435","series":"1.2.826.0.1.3680043.8.498.85157211933861275751721312445130516208","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88189503151202366936330411627291827689","series":"1.2.826.0.1.3680043.8.498.12629073278185303288693271257044865047","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88368930449171804348215887291291274615","series":"1.2.826.0.1.3680043.8.498.73709356688034162897760146187686608535","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88466268868654468582661984391239692466","series":"1.2.826.0.1.3680043.8.498.12547822749235868679518789081179916406","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88505044370271842163477475980178015566","series":"1.2.826.0.1.3680043.8.498.10237734388271885925719306140705650241","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88570871026022705696415758052703394453","series":"1.2.826.0.1.3680043.8.498.46875908868670365082362268761049333983","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88574487918084130981335189023288704909","series":"1.2.826.0.1.3680043.8.498.62286553916211428353506141704567925357","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88653085234639096911379842849193874077","series":"1.2.826.0.1.3680043.8.498.52150321061023240695496562542843228274","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88679328523895187500714090963655363787","series":"1.2.826.0.1.3680043.8.498.51070716672988366376622479595569699309","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88711356400820562340756013729440417015","series":"1.2.826.0.1.3680043.8.498.17272176370866126877552415316040370127","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.88752039058074948027322975455607258941","series":"1.2.826.0.1.3680043.8.498.11277478597280339444935439034832530284","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.88801151487634957068752641200383474212","series":"1.2.826.0.1.3680043.8.498.54786454784329308975353713632612087837","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.88872155446935518174304990308380906112","series":"1.2.826.0.1.3680043.8.498.96352343184328941448469028045186966567","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.88896246957673838955012057210098034177","series":"1.2.826.0.1.3680043.8.498.11537926880976933410994516920880793130","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.88964371740619947193957393682095448256","series":"1.2.826.0.1.3680043.8.498.40080285253180151340092548627066002643","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.89045629523485194086986785222960864883","series":"1.2.826.0.1.3680043.8.498.52129219726968247354811955141473846636","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89074359486713340585710357399403393294","series":"1.2.826.0.1.3680043.8.498.24226318679697603582368597783008195548","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89123035099342878362576960682354791116","series":"1.2.826.0.1.3680043.8.498.10753929022446935678976745291111177503","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89240184178186399632216878563608101260","series":"1.2.826.0.1.3680043.8.498.12039070491359905130947778086669762849","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.89302191571937190529967168726521003428","series":"1.2.826.0.1.3680043.8.498.51118140251581704111284469054454531194","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89332108074590559691047812196991368746","series":"1.2.826.0.1.3680043.8.498.62447332521346501443855693457237081058","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89350648596674998203116057439419132072","series":"1.2.826.0.1.3680043.8.498.92763071696479537983605675869403856134","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89423600201830685256510321877396547497","series":"1.2.826.0.1.3680043.8.498.44585219108984688315594303186872384905","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89464179959474561696319051639748844244","series":"1.2.826.0.1.3680043.8.498.37371162368578026770768748237545355459","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.89576499383950909889665633610791093334","series":"1.2.826.0.1.3680043.8.498.10968891303104749055226988798403237586","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89720428711655200442262503508409163392","series":"1.2.826.0.1.3680043.8.498.91895698497105225928246382486269808703","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89746004948304570186940283501211828475","series":"1.2.826.0.1.3680043.8.498.40211637723016291350787021567373652094","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.89857801102910604766959864464865910585","series":"1.2.826.0.1.3680043.8.498.13437184027839750122198191832357293057","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.89943815722422273305517145510644432987","series":"1.2.826.0.1.3680043.8.498.39743792199516939309493093781851509157","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.89987866402625167201267936974120824712","series":"1.2.826.0.1.3680043.8.498.59143944335322342641717281363126060490","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90037507233074511524602897439257102032","series":"1.2.826.0.1.3680043.8.498.12627423084288795265368907132309997128","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90159195278632080234412316065469499332","series":"1.2.826.0.1.3680043.8.498.39714722153628559133137786086256575453","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90172333170788218623219231170360850947","series":"1.2.826.0.1.3680043.8.498.43753689220874776512944599062568478172","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90223492515368389331845681759355261030","series":"1.2.826.0.1.3680043.8.498.12921166391128367791701100406796757793","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90251533154603802045666102095577144628","series":"1.2.826.0.1.3680043.8.498.12350649463900546412746618008596652073","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90274711315692820326973277297753398634","series":"1.2.826.0.1.3680043.8.498.10424430860040158579336427029862115408","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90288598546566291188788375261370919222","series":"1.2.826.0.1.3680043.8.498.96240106589845149701822424514528096061","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90331082101665002495739626395247831768","series":"1.2.826.0.1.3680043.8.498.30464654634310403073026464882168769693","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90433488790413321102007716267340900056","series":"1.2.826.0.1.3680043.8.498.53000796224237878490764251187549690544","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90466423642678253933409966898120881010","series":"1.2.826.0.1.3680043.8.498.49008578332451649004894550298681178766","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.90508461713737190527021531642732695353","series":"1.2.826.0.1.3680043.8.498.37865021518415771182050214559573377257","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90551187439998209641370107095424494085","series":"1.2.826.0.1.3680043.8.498.13363326897203144439018492941918763557","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.90629760164319310375909187696899268080","series":"1.2.826.0.1.3680043.8.498.89271542740740140178465940260136110908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.90751504670572401790063555695835439317","series":"1.2.826.0.1.3680043.8.498.10771529694146997565769954522872765570","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90764713690393618577990540005126771712","series":"1.2.826.0.1.3680043.8.498.10648190695068069896952659554444136362","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.90893882071188409362296099956462540506","series":"1.2.826.0.1.3680043.8.498.50184520053320323165480705690742573089","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.90985037792656682958281230441654772668","series":"1.2.826.0.1.3680043.8.498.99567580196968118121119057821170796029","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91106275065657355304239097295699332135","series":"1.2.826.0.1.3680043.8.498.41231395497644807507283562430778232353","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91124979914013108948829704350601987380","series":"1.2.826.0.1.3680043.8.498.13393014261082142218569655651758251815","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91168270990167204047014161851340054645","series":"1.2.826.0.1.3680043.8.498.40285523343483077011333318871639042089","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91220255464004258635083940797356822178","series":"1.2.826.0.1.3680043.8.498.40226447511462762737588662749248996024","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91262503278090288446189575519187653125","series":"1.2.826.0.1.3680043.8.498.19454915901051638657603139267984855056","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91328985601265188961626433080766431089","series":"1.2.826.0.1.3680043.8.498.11678807590482706687076534202474957140","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91361379904133499492465873800011777365","series":"1.2.826.0.1.3680043.8.498.11435161521929192543149412670580143466","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.91391446167553025113367459399831345295","series":"1.2.826.0.1.3680043.8.498.56834841259731821379452160306145868008","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91468214041781427810665883790357021637","series":"1.2.826.0.1.3680043.8.498.14016766690825227020335239619646027878","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91581717063042044543371480238854954692","series":"1.2.826.0.1.3680043.8.498.87052111292921797873577709598594135080","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91660278231317256353374962767408532443","series":"1.2.826.0.1.3680043.8.498.63191566363446075429803458299656367407","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91727248305411866145131034915485728037","series":"1.2.826.0.1.3680043.8.498.51868490574552863272776112159378117397","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.91839696756303525362262251587171486273","series":"1.2.826.0.1.3680043.8.498.19391141414176353291128611964684220317","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.91897419008563027309752122760214275182","series":"1.2.826.0.1.3680043.8.498.11705821864354679277637946068163698486","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.91913894091933783677568170210064807665","series":"1.2.826.0.1.3680043.8.498.10331337904364362223730809416231577770","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.91979363712708769014277114061159398016","series":"1.2.826.0.1.3680043.8.498.17618679740113802741266818645008823120","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92063841381005962285592592672311044745","series":"1.2.826.0.1.3680043.8.498.21761965181505591413483190011170317901","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92127426417720107708158713612888411025","series":"1.2.826.0.1.3680043.8.498.10348156661377191120362127148340942984","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92249560662718438332916289759028859881","series":"1.2.826.0.1.3680043.8.498.27303206044807053743821762845888513946","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.92301915122994899889226415477164044164","series":"1.2.826.0.1.3680043.8.498.93642620813709868443795528044212541465","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92407834172018110142589158281575319154","series":"1.2.826.0.1.3680043.8.498.61290733100344986898734124549618654288","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92445779773398371452908052305534240654","series":"1.2.826.0.1.3680043.8.498.10624167519253570658064386745886379456","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92581137608969847756039955144478716506","series":"1.2.826.0.1.3680043.8.498.53836875693827167343208371719139661089","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.92621720172216795135762712303124932801","series":"1.2.826.0.1.3680043.8.498.13256853238406904178314443839818352322","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.92650426727723557280117118587545413917","series":"1.2.826.0.1.3680043.8.498.64138448377387627279578230009575267092","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92682287574388138675492893420192905780","series":"1.2.826.0.1.3680043.8.498.78413390359460676641323045800946869095","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.92758635328468215409701857842011454608","series":"1.2.826.0.1.3680043.8.498.10262159277771764098654661323118689739","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.92854445271746966212168817610305595078","series":"1.2.826.0.1.3680043.8.498.45651183303604840337108931334472419174","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93026010065162534828966791702007067882","series":"1.2.826.0.1.3680043.8.498.12739257963010029044871038409946670336","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93062749017302143105207135454940053142","series":"1.2.826.0.1.3680043.8.498.41701181508443960878501721200563391277","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93141871723058796160969424330424882014","series":"1.2.826.0.1.3680043.8.498.12655269587680545610271021841473899302","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93188773180403601524488403580611534053","series":"1.2.826.0.1.3680043.8.498.12381387897984540622781230890859603711","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93230313557251171119475173996942942855","series":"1.2.826.0.1.3680043.8.498.35945016904292281053255412699586919068","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.93318293884671598055223778443354732793","series":"1.2.826.0.1.3680043.8.498.10447516940980817595209781155830640480","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93359387652821116106937882388838959209","series":"1.2.826.0.1.3680043.8.498.46066982789403183951797581146886365682","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93555726419626687118723083838655757151","series":"1.2.826.0.1.3680043.8.498.48336846076682721579398272466824762812","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.93653421810951484288813767943453706233","series":"1.2.826.0.1.3680043.8.498.65833189236206859970047858193804699848","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93719148934772792523193842156217224487","series":"1.2.826.0.1.3680043.8.498.10909415160385104524399857124944945957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.93747304211180517979657679136059719120","series":"1.2.826.0.1.3680043.8.498.19827244834297437246470153444076951118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.93902628119199416703899320994283313812","series":"1.2.826.0.1.3680043.8.498.55664229626588204639063665017244563473","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.93976819306514730278115540117878688308","series":"1.2.826.0.1.3680043.8.498.13101021265730830411276321209559451351","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94027547274004055924993074754507802303","series":"1.2.826.0.1.3680043.8.498.48171170347970202735676878222173177484","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94069670011071682445520414370532541612","series":"1.2.826.0.1.3680043.8.498.67524381948129254117605770433004257369","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94252884278314596961426629318911909075","series":"1.2.826.0.1.3680043.8.498.38766062948963546423189220846940401090","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94274898137310893945218731717226819541","series":"1.2.826.0.1.3680043.8.498.97021433364468673150298035320637751406","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94335780236370744781443631231792792083","series":"1.2.826.0.1.3680043.8.498.12604262684207469645948316337280057698","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94393348753936295687030819683853459606","series":"1.2.826.0.1.3680043.8.498.67029675860913660398466256068985219212","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94448720962054029575571654250994769402","series":"1.2.826.0.1.3680043.8.498.12229855168400329501395017884593087134","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94594009911808653224418094594012527944","series":"1.2.826.0.1.3680043.8.498.58268554925311610219998613051222056869","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.94607524665407029611795868588964734480","series":"1.2.826.0.1.3680043.8.498.92112170349265628205009645284913342608","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94658317716039594639343437780115140531","series":"1.2.826.0.1.3680043.8.498.57587987608278201375427791593562788967","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94722363227977785887442194094888460046","series":"1.2.826.0.1.3680043.8.498.16105182556390120040215440240007374348","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.94753144127920558221268610279343359449","series":"1.2.826.0.1.3680043.8.498.23990519024090052393781307290880994468","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94774472240035721003264951868030913589","series":"1.2.826.0.1.3680043.8.498.11401283718419698410748301132552933481","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.94921929374845793599006362190119159537","series":"1.2.826.0.1.3680043.8.498.10909817820501600918600514029059753832","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.94945670991495145462605178471390637325","series":"1.2.826.0.1.3680043.8.498.11635554921927377484763601708357199641","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95026524886584356226576936309513335214","series":"1.2.826.0.1.3680043.8.498.70790027583029637074495100434592755484","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95060380238431209646600378585422128745","series":"1.2.826.0.1.3680043.8.498.19569832938337244378630053279967620046","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95113114010658169441709314948308683982","series":"1.2.826.0.1.3680043.8.498.64740346877053542646124965054696621339","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95193282490279394602564266574551123732","series":"1.2.826.0.1.3680043.8.498.72233253649229595267815411395930090983","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95223549428749744711279985897241419512","series":"1.2.826.0.1.3680043.8.498.30415673915416959687544702139793304346","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95321750682026667237717880394536750835","series":"1.2.826.0.1.3680043.8.498.67885736434421530167603900123023770190","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95351925765761617117057799846763403963","series":"1.2.826.0.1.3680043.8.498.10048713072268890750198666318760216140","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95433650759419050864053270236035947981","series":"1.2.826.0.1.3680043.8.498.10852729735700669441140275035812125640","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.95497572581157858069692837655072209020","series":"1.2.826.0.1.3680043.8.498.80783863745735302147896486898010540576","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95555755747044090477918262640906805885","series":"1.2.826.0.1.3680043.8.498.11216583101535872247673097383757844168","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95570207430778332247200880037915693375","series":"1.2.826.0.1.3680043.8.498.31042418971246103590894755125103603759","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95601371640669355936785637162110963091","series":"1.2.826.0.1.3680043.8.498.50397971542191273421430819896380694901","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95638601079527306640829424921455011698","series":"1.2.826.0.1.3680043.8.498.65720820291238437340864275949951867504","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.95705755527332814775935856896567782522","series":"1.2.826.0.1.3680043.8.498.62610738826530176020932306207670579745","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95727100148189624148200652225439854460","series":"1.2.826.0.1.3680043.8.498.83967136765203909953427489271526445908","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.95796009319232592137564145023096016872","series":"1.2.826.0.1.3680043.8.498.63128063707681066484797939897837640641","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.95864678405973312042815189771902289608","series":"1.2.826.0.1.3680043.8.498.17757354504217216427640169705429963043","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.95918254432056542701730614049432316728","series":"1.2.826.0.1.3680043.8.498.61381323051771310815544395749030522596","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96054287426681123062488435620352206601","series":"1.2.826.0.1.3680043.8.498.74699531134835288647894175015445837293","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96117880374445894729658092255088269716","series":"1.2.826.0.1.3680043.8.498.12145075984219180131050290682281038706","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.96152894373838410779390571000523933753","series":"1.2.826.0.1.3680043.8.498.13478067585007673893758978123166013023","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96293531288862652713666056362328893428","series":"1.2.826.0.1.3680043.8.498.12929805483077584302974243428304523336","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96323485839061204012967074568679195142","series":"1.2.826.0.1.3680043.8.498.47632636948280424897305663813782501662","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96483513352751590579274577125356217169","series":"1.2.826.0.1.3680043.8.498.12640303405637349953442496442783249805","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96559819562317669526803257835398470484","series":"1.2.826.0.1.3680043.8.498.87074378250755913984616317844865860663","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96562865861483086790615358719391771217","series":"1.2.826.0.1.3680043.8.498.84680945299599438679895350240053682799","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.96614828054790815108190478795539254570","series":"1.2.826.0.1.3680043.8.498.34023540025978409704736269165519587550","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.96791010280967772394523567484937888186","series":"1.2.826.0.1.3680043.8.498.85792287982282727680650310646177180793","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.96846796615932368166905347054245013501","series":"1.2.826.0.1.3680043.8.498.42119828585289179040223023339804496479","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.96944926470854551064948200936284887742","series":"1.2.826.0.1.3680043.8.498.96916049578120287025288423314240574302","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.96950011369238410949579095568317214261","series":"1.2.826.0.1.3680043.8.498.49856074684723163314188932278352601110","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97020520267554584684160936968569132254","series":"1.2.826.0.1.3680043.8.498.28498849462830399414480471067946328759","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97055845147740306780908051236353219644","series":"1.2.826.0.1.3680043.8.498.12361209371067847850690323582671426428","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97135552871905937467775630206570358085","series":"1.2.826.0.1.3680043.8.498.93567063140615601149743554503233211731","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97220604868276521004168045790587997687","series":"1.2.826.0.1.3680043.8.498.56948538257205557841238955735151966957","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97259766233250760343109933426776941754","series":"1.2.826.0.1.3680043.8.498.54792428102517649575747323337400797384","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97269744864161337523188705262974234442","series":"1.2.826.0.1.3680043.8.498.12871648191974633893651549257412844086","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97303893645051596044169857585815133097","series":"1.2.826.0.1.3680043.8.498.15737460332740505598699018331126180684","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.97353980946516912216109392482235435606","series":"1.2.826.0.1.3680043.8.498.28558413399290447454378870118165835912","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97485476951995325021464683747847126869","series":"1.2.826.0.1.3680043.8.498.12847743113855742782109137769831697327","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97496319463700021339637597270948309369","series":"1.2.826.0.1.3680043.8.498.10213293600123722037956129949380958194","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97520184067825971918316082134210579391","series":"1.2.826.0.1.3680043.8.498.10047149081643727582361533331099856959","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97579738268290461886850261405719501384","series":"1.2.826.0.1.3680043.8.498.60340718691995184401258101706361673344","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97658595748192892503053963872409769829","series":"1.2.826.0.1.3680043.8.498.46376039048482306312882374427669185509","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.97696436660690920809826140686268769509","series":"1.2.826.0.1.3680043.8.498.10265126837592905636369238529672028708","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.97731968872354496470096174559789586056","series":"1.2.826.0.1.3680043.8.498.86503283580686559278139784635286543920","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.97830923192341890652952842494018952920","series":"1.2.826.0.1.3680043.8.498.13852545462235926302893304618969156773","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.97909881548577311929907613359449243075","series":"1.2.826.0.1.3680043.8.498.12065664986603104687920709985066923230","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98022738778461179495257509215244053198","series":"1.2.826.0.1.3680043.8.498.12690919560632806042393561974166566781","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98028018520101798863795620289151810642","series":"1.2.826.0.1.3680043.8.498.10204616566061193502766688519962652177","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98064941971367073978404461105852658265","series":"1.2.826.0.1.3680043.8.498.82563680182842672804030495784014077921","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98107112397588450345153336193341117589","series":"1.2.826.0.1.3680043.8.498.10596987945279037236050305367413438908","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98143909986671737130865120569673002676","series":"1.2.826.0.1.3680043.8.498.40449769170768949325318643683678731262","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.98181040478767526925214101502401448074","series":"1.2.826.0.1.3680043.8.498.12597897113113096548226336355469888197","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98255454187594895604610283922744869720","series":"1.2.826.0.1.3680043.8.498.26274478529928590950856115479174725836","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98343185597651455245996667389303915779","series":"1.2.826.0.1.3680043.8.498.61265269007083980028622659461588250782","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98400072082366198211931728015692551635","series":"1.2.826.0.1.3680043.8.498.12190638991320645789643296228594031046","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.98541661035340329000785741171799559104","series":"1.2.826.0.1.3680043.8.498.73797141009521769231437354878512574685","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98676079331571304021387833410440136393","series":"1.2.826.0.1.3680043.8.498.83079768733136778288768927790681587832","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.98781117432577714521231232440503778261","series":"1.2.826.0.1.3680043.8.498.10894208817016556640736504786442504160","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98824066826946884510314135448574304233","series":"1.2.826.0.1.3680043.8.498.11064239783877942433784484527483674874","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.98883718766501717651651876978607853803","series":"1.2.826.0.1.3680043.8.498.53493584648864826005002576555762970916","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.98914528123216365374152816424181510781","series":"1.2.826.0.1.3680043.8.498.43971348164232806014075630071047923980","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99077241529686301278201179461896593659","series":"1.2.826.0.1.3680043.8.498.12158191064087307900534978493670330765","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99108147913292332418681050058312852494","series":"1.2.826.0.1.3680043.8.498.33306791204492974988927084749008845118","plane":"Coronal","fluid":0,"slot":"COR_T1"},{"study":"1.2.826.0.1.3680043.8.498.99237666579371266866432031468578336182","series":"1.2.826.0.1.3680043.8.498.77352332391950985772165791089429212524","plane":"Axial","fluid":0,"slot":"AX_T1"},{"study":"1.2.826.0.1.3680043.8.498.99257151613846084492342083136797222759","series":"1.2.826.0.1.3680043.8.498.10482024555490761490529417956932095632","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99413398303724030328059886749330413303","series":"1.2.826.0.1.3680043.8.498.80069060445125156385233497804903941908","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99438805806853675793560604625783460513","series":"1.2.826.0.1.3680043.8.498.35512252819558707290544208946022043606","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99448612525108835020450967944392129536","series":"1.2.826.0.1.3680043.8.498.50871021209713754782831958841796889846","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99493257395588095270208592176887619190","series":"1.2.826.0.1.3680043.8.498.76428902599504547762540925475513447108","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99577045720075471058329330051900767206","series":"1.2.826.0.1.3680043.8.498.38662302965772935816508932844745198212","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99651725851820083142440965238449655973","series":"1.2.826.0.1.3680043.8.498.55035669079461747592208793249480086778","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99710592666731203419714499168506345263","series":"1.2.826.0.1.3680043.8.498.13335220307995571884694301831854539909","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99768154649604643975127490783989114506","series":"1.2.826.0.1.3680043.8.498.18247559701490950705051685558619508063","plane":"Coronal","fluid":1,"slot":"COR_FS"},{"study":"1.2.826.0.1.3680043.8.498.99826037216341310731701527407057049820","series":"1.2.826.0.1.3680043.8.498.12723944002573784499653596840821017997","plane":"Sagittal","fluid":0,"slot":"SAG_T1"},{"study":"1.2.826.0.1.3680043.8.498.99864558747514747686151806909794288262","series":"1.2.826.0.1.3680043.8.498.51298636925191348309051704222020365453","plane":"Sagittal","fluid":1,"slot":"SAG_FS"},{"study":"1.2.826.0.1.3680043.8.498.99926624968240681772735102303531613209","series":"1.2.826.0.1.3680043.8.498.72368507579017597862126524135905800463","plane":"Axial","fluid":1,"slot":"AX_FS"},{"study":"1.2.826.0.1.3680043.8.498.99957611207313098696146908027229590966","series":"1.2.826.0.1.3680043.8.498.75317509519712148912025658897439447721","plane":"Sagittal","fluid":1,"slot":"SAG_FS"}]')
print('series 320 rgb, UID figes depuis crop130_rgb25d :', len(jobs))
assert len(jobs) == 1778
assert jobs and {'study', 'series', 'plane', 'fluid', 'slot'} <= set(jobs[0])


In [ ]:

def _sort_key(ds) -> float:
    inst = getattr(ds, "InstanceNumber", None)
    if inst is not None:
        try:
            return float(inst)
        except (TypeError, ValueError):
            pass
    ipp = getattr(ds, "ImagePositionPatient", None)
    iop = getattr(ds, "ImageOrientationPatient", None)
    if ipp is not None and iop is not None:
        row = np.array(iop[:3], dtype=np.float64)
        col = np.array(iop[3:], dtype=np.float64)
        normal = np.cross(row, col)
        return float(np.dot(np.array(ipp, dtype=np.float64), normal))
    if ipp is not None:
        return float(ipp[2])
    return 0.0


def _spacing_mm(ds):
    ps = getattr(ds, "PixelSpacing", None) or getattr(ds, "ImagerPixelSpacing", None)
    if ps is None:
        return None
    try:
        return float(ps[0]), float(ps[1])
    except (TypeError, ValueError, IndexError):
        return None


def _center_crop(img: np.ndarray, row_mm, col_mm) -> tuple[np.ndarray, str]:
    h, w = img.shape[:2]
    if row_mm is not None and col_mm is not None and row_mm > 0 and col_mm > 0:
        ch = int(round(CROP_MM / row_mm))
        cw = int(round(CROP_MM / col_mm))
        ch = min(h, max(8, ch))
        cw = min(w, max(8, cw))
        mode = "mm"
    else:
        ch = max(8, int(round(h * CROP_REL)))
        cw = max(8, int(round(w * CROP_REL)))
        ch, cw = min(h, ch), min(w, cw)
        mode = "rel"
    y0 = (h - ch) // 2
    x0 = (w - cw) // 2
    return img[y0:y0 + ch, x0:x0 + cw], mode


def _rescale(ds, arr: np.ndarray) -> np.ndarray:
    arr = arr.astype(np.float32, copy=False)
    slope = float(getattr(ds, "RescaleSlope", 1.0) or 1.0)
    intercept = float(getattr(ds, "RescaleIntercept", 0.0) or 0.0)
    arr = arr * slope + intercept
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def _to_u8(sl: np.ndarray) -> np.ndarray:
    sl = cv2.resize(sl, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return np.clip(sl * 255.0, 0, 255).round().astype(np.uint8)


def convert_series(series_dir: Path) -> tuple[np.ndarray, dict]:
    files = sorted(series_dir.glob("*.dcm"))
    if not files:
        raise FileNotFoundError(f"pas de .dcm dans {series_dir}")
    datasets = []
    for f in files:
        try:
            datasets.append(pydicom.dcmread(str(f), stop_before_pixels=False))
        except Exception:
            continue
    if not datasets:
        raise RuntimeError("aucun DICOM lisible")
    datasets.sort(key=_sort_key)

    frames = []
    sp = None
    for ds in datasets:
        arr = ds.pixel_array
        if sp is None:
            sp = _spacing_mm(ds)
        arr = _rescale(ds, arr)
        if arr.ndim == 2:
            frames.append(arr)
        elif arr.ndim == 3:
            frames.extend([arr[i] for i in range(arr.shape[0])])
        else:
            raise RuntimeError(f"ndim inattendu {arr.ndim}")
    if not frames:
        raise RuntimeError("aucune coupe")

    row_mm, col_mm = (sp if sp is not None else (None, None))
    cropped, crop_mode = [], None
    for im in frames:
        c, crop_mode = _center_crop(im, row_mm, col_mm)
        cropped.append(c)
    h0, w0 = cropped[0].shape[:2]
    stacked = []
    for c in cropped:
        if c.shape[:2] != (h0, w0):
            c = cv2.resize(c, (w0, h0), interpolation=cv2.INTER_AREA)
        stacked.append(c)
    vol = np.stack(stacked, axis=0)  # (n, h, w) volume ORIGINAL trie

    lo, hi = np.percentile(vol, 1.0), np.percentile(vol, 99.0)
    if hi <= lo:
        hi = lo + 1.0
    vol = np.clip((vol - lo) / (hi - lo), 0, 1)

    n = vol.shape[0]
    a = int(np.floor(BAND[0] * (n - 1)))
    b = int(np.ceil(BAND[1] * (n - 1)))
    if b - a + 1 < 8:
        a, b = 0, n - 1
    # memes 24 centres que crop130 (linspace dans la bande),
    # mais voisins ±1 dans le volume original, pas dans les 24
    idx_band = np.linspace(0, b - a, N_SLICES).round().astype(int)
    idx_orig = np.clip(a + idx_band, 0, n - 1)

    out = np.empty((N_SLICES, 3, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    last = n - 1
    n_clamp = 0
    for i, k in enumerate(idx_orig):
        k = int(k)
        km = 0 if k <= 0 else k - 1
        kp = last if k >= last else k + 1
        if km == k or kp == k:
            n_clamp += 1
        out[i, 0] = _to_u8(vol[km])
        out[i, 1] = _to_u8(vol[k])
        out[i, 2] = _to_u8(vol[kp])

    meta = {
        "n_raw": int(n),
        "band": [a, b],
        "crop_mode": crop_mode,
        "spacing_mm": None if sp is None else [float(sp[0]), float(sp[1])],
        "src_hw": [int(h0), int(w0)],
        "n_files": len(datasets),
        "n_clamp": int(n_clamp),
        "rgb": "dicom_neighbors",
    }
    return out, meta


def series_path(study_uid: str, series_uid: str) -> Path:
    p = TRAIN_SERIES_DIR / study_uid / series_uid
    if p.exists():
        return p
    p2 = TRAIN_SERIES_DIR / series_uid
    return p2


In [ ]:

import zipfile

zip_path = OUT_DIR / f"volumes_crop130_rgb25d_320_part{SHARD_ID}.zip"
manifest_path = OUT_DIR / f"manifest_rgb25d_320_part{SHARD_ID}.csv"
recipe_path = OUT_DIR / "RECIPE.json"

recipe = {
    "crop_mm": CROP_MM,
    "crop_rel": CROP_REL,
    "band": list(BAND),
    "n_slices": N_SLICES,
    "img_size": IMG_SIZE,
    "planes": PLANES,
    "shard_id": SHARD_ID,
    "n_shards": N_SHARDS,
    "n_jobs": len(jobs),
    "shape": [N_SLICES, 3, IMG_SIZE, IMG_SIZE],
    "rgb": "dicom_neighbors",
    "note": (
        "1 serie fluid-prioritaire par plan. Crop 130 mm (PixelSpacing) sinon "
        "centre 70 %. Bande 20-80 %, linspace 24 centres. Pour chaque centre k "
        "dans le volume ORIGINAL : R=k-1, G=k, B=k+1 (1 coupe DICOM d'ecart, "
        "clamp aux bords). Resize 320, uint8. PAS les voisins des 24 linspace."
    ),
}
recipe_path.write_text(json.dumps(recipe, indent=2), encoding="utf-8")
print(json.dumps(recipe, indent=2))

def _one(job):
    sid = job["series"]
    t0 = time.time()
    try:
        p = series_path(job["study"], sid)
        vol, meta = convert_series(p)
        buf = io.BytesIO()
        np.savez_compressed(buf, data=vol)
        payload = buf.getvalue()
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 1,
            "error": "",
            "nbytes": len(payload),
            "dt": round(time.time() - t0, 3),
            **meta,
        }
        return sid, payload, rec
    except Exception as e:
        rec = {
            "StudyInstanceUID": job["study"],
            "SeriesInstanceUID": sid,
            "plane": job["plane"],
            "fluid": job["fluid"],
            "slot": job.get("slot", ""),
            "ok": 0,
            "error": f"{type(e).__name__}: {e}"[:300],
            "nbytes": 0,
            "dt": round(time.time() - t0, 3),
            "n_raw": None, "band": None, "crop_mode": None,
            "spacing_mm": None, "src_hw": None, "n_files": None,
            "n_clamp": None, "rgb": None,
        }
        return sid, None, rec

t_all = time.time()
rows = []
n_ok = n_fail = 0
bytes_sum = 0

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as zf:
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        futs = [ex.submit(_one, job) for job in jobs]
        for i, fut in enumerate(as_completed(futs), 1):
            sid, payload, rec = fut.result()
            rows.append(rec)
            if payload is not None:
                zf.writestr(f"{sid}.npz", payload)
                n_ok += 1
                bytes_sum += rec["nbytes"]
            else:
                n_fail += 1
            if i % 25 == 0 or i == len(jobs):
                elapsed = time.time() - t_all
                rate = i / max(elapsed, 1e-6)
                eta = (len(jobs) - i) / max(rate, 1e-6)
                print(
                    f"[{i}/{len(jobs)}] ok={n_ok} fail={n_fail} "
                    f"zip={bytes_sum/1e9:.2f}GB  {rate:.2f} s/serie  ETA {eta/60:.0f} min",
                    flush=True,
                )
                pd.DataFrame(rows).to_csv(manifest_path, index=False)

pd.DataFrame(rows).to_csv(manifest_path, index=False)
print("termine", zip_path, "size", zip_path.stat().st_size / 1e9, "GB")
print("ok", n_ok, "fail", n_fail, "en", (time.time() - t_all) / 60, "min")
print("manifest", manifest_path)
if n_fail:
    fails = [r for r in rows if not r["ok"]]
    print("exemples d'erreurs :")
    for r in fails[:8]:
        print(" ", r["SeriesInstanceUID"], r["error"])


In [ ]:

import zipfile as zfmod
with zfmod.ZipFile(zip_path) as zf:
    names = [n for n in zf.namelist() if n.endswith(".npz")]
    print("npz dans le zip :", len(names))
    for name in names[:3]:
        with zf.open(name) as f:
            vol = np.load(io.BytesIO(f.read()))["data"]
        assert vol.shape == (N_SLICES, 3, IMG_SIZE, IMG_SIZE), vol.shape
        assert vol.dtype == np.uint8, vol.dtype
        rg = np.abs(vol[:, 0].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        bg = np.abs(vol[:, 2].astype(np.float32) - vol[:, 1].astype(np.float32)).mean()
        print(
            name, vol.shape, vol.dtype,
            "min", int(vol.min()), "max", int(vol.max()),
            "mean", float(vol.mean()),
            "|R-G|", round(float(rg), 2), "|B-G|", round(float(bg), 2),
        )
print("attendu : shape (24, 3, 320, 320), |R-G| et |B-G| > 0 (voisins DICOM reels)")
print("fichiers working :", sorted(p.name for p in OUT_DIR.iterdir() if p.is_file()))
